## Create a simple UI in Streamlit without using built-in layout functions, manually structuring components using HTML and CSS, and ensuring responsiveness across different screen sizes.

In [1]:
%pip install streamlit pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit
import pandas
import sklearn
import joblib

print("All libraries installed successfully")

All libraries installed successfully


In [3]:
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("heart.csv")
df = df.drop_duplicates().dropna()

X = df.drop(columns=["target"])
y = df["target"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

joblib.dump(model, "heart_model_v20261013_160903.joblib")

print("Model saved successfully!")

Model saved successfully!


In [4]:
%cd "30oct folder ka complete path"
!streamlit run app.py

[WinError 2] The system cannot find the file specified: '30oct folder ka complete path'
C:\Users\Admin\heart_disease\4th_Nov


Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [5]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


## Task 1: Implement user input handling for prediction by validating inputs dynamically, handling edge cases such as missing or incorrect values, and optimizing input processing for real-time predictions.

In [6]:
%%writefile input.py
"""
HeartCare - user input handling for prediction.

1. Dynamic validation : rules (allowed categories, number ranges) heart.csv se khud banti hain.
2. Edge cases         : missing, text, out-of-range, unknown category, inf/NaN, extra keys, wrong length.
3. Real-time speed    : ek hi predict_proba pass, repeated input ka cache, batch me vectorized validation.
"""
import numpy as np
import pandas as pd
from functools import lru_cache

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
MISSING_WORDS = {"", "na", "n/a", "nan", "null", "none", "?", "-", "unknown", "not known"}
MAX_MISSING = 3           

def build_rules(df):
    """Training data se rules: <= 5 alag values wala column = category, baaki = number."""
    rules = {}
    for f in FEATURES:
        col = df[f].astype(float).dropna()
        vals = np.sort(col.unique())
        r = {"min": float(col.min()), "max": float(col.max())}
        if len(vals) <= 5:
            r.update(kind="category", allowed=vals.tolist(), fill=float(col.mode()[0]))
        else:
            span = col.max() - col.min()
            lo, hi = col.min() - 0.25 * span, col.max() + 0.25 * span    
            if col.min() >= 0:
                lo = max(lo, 0.0)                                       
            r.update(kind="number", hard_lo=float(lo), hard_hi=float(hi), fill=float(col.median()))
        rules[f] = r
    return rules


def load_rules(csv_path="heart.csv"):
    return build_rules(pd.read_csv(csv_path).drop_duplicates().dropna())


def is_missing(v):
    if v is None or v is pd.NA:
        return True
    if isinstance(v, str):
        return v.strip().lower() in MISSING_WORDS
    return isinstance(v, (float, np.floating)) and np.isnan(v)


def to_number(v):
    if isinstance(v, (bool, int, float, np.integer, np.floating)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower().replace(",", ".")           
        return {"true": 1.0, "false": 0.0}.get(t) if t in ("true", "false") else float(t)
    raise TypeError("number nahi hai")


def validate_input(record, rules):
    """record = dict ya 13 values ki list. Return: {ok, x, errors, warnings, filled}"""
    out = {"ok": False, "x": None, "errors": [], "warnings": [], "filled": []}
    if isinstance(record, (list, tuple, np.ndarray)):
        if len(record) != len(FEATURES):
            out["errors"].append(f"{len(FEATURES)} values chahiye, {len(record)} mili")
            return out
        record = dict(zip(FEATURES, record))
    if not isinstance(record, dict):
        out["errors"].append("input dict ya list hona chahiye")
        return out

    rec = {str(k).strip().lower().replace(" ", "_"): v for k, v in record.items()}   
    extra = [k for k in rec if k not in rules]
    if extra:
        out["warnings"].append(f"extra fields ignore kiye: {extra}")

    x = np.zeros(len(FEATURES))
    for i, f in enumerate(FEATURES):
        v, r = rec.get(f), rules[f]
        if is_missing(v):
            x[i] = r["fill"]; out["filled"].append(f); continue
        try:
            val = to_number(v)
        except (ValueError, TypeError):
            out["errors"].append(f"{f}: '{v}' number nahi hai"); continue
        if not np.isfinite(val):
            out["errors"].append(f"{f}: inf/NaN allowed nahi"); continue
        if r["kind"] == "category":
            if val not in r["allowed"]:
                out["errors"].append(f"{f}: {val:g} allowed nahi (allowed: {[int(a) for a in r['allowed']]})")
        elif not r["hard_lo"] <= val <= r["hard_hi"]:
            out["errors"].append(f"{f}: {val:g} impossible value (limit {r['hard_lo']:.1f} se {r['hard_hi']:.1f})")
        elif not r["min"] <= val <= r["max"]:
            out["warnings"].append(f"{f}: {val:g} training range [{r['min']:g}, {r['max']:g}] ke bahar hai, prediction kam bharosemand")
        x[i] = val

    if len(out["filled"]) > MAX_MISSING:
        out["errors"].append(f"{len(out['filled'])} fields missing hain (max {MAX_MISSING} allowed)")
    elif out["filled"]:
        out["warnings"].append(f"missing fields median/mode se bhare gaye: {out['filled']}")
    if not out["errors"]:
        out["ok"], out["x"] = True, x
    return out


def predict_result(model, x):
    """x = 13 numbers. Pehle 2 pass (predict + predict_proba) the, ab sirf 1 pass."""
    probabilities = model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0]
    prediction = int(model.classes_[np.argmax(probabilities)])
    return prediction, probabilities


def make_predictor(model, size=1024):
    """Same input dobara aaye to model dobara nahi chalta (cache)."""
    @lru_cache(maxsize=size)
    def cached(key):
        return predict_result(model, list(key))
    return lambda x: cached(tuple(float(v) for v in x))


def _to_numbers(s):
    """Series -> (numbers, missing_mask). Text wali value NaN banti hai aur missing nahi maani jati."""
    if pd.api.types.is_numeric_dtype(s):
        num = s.astype(float).to_numpy()
        return num, np.isnan(num)
    txt = s.astype(str).str.strip().str.lower()
    miss = (s.isna() | txt.isin(MISSING_WORDS)).to_numpy()
    num = pd.to_numeric(txt.str.replace(",", ".", regex=False).replace({"true": "1", "false": "0"}), errors="coerce").to_numpy(dtype=float)
    return num, miss


def validate_batch(raw, rules):
    """DataFrame ki saari rows ek saath validate. Return: X, ok(bool array), reason(Series)"""
    raw = raw.rename(columns=lambda c: str(c).strip().lower().replace(" ", "_"))
    n = len(raw)
    X, bad, n_miss = np.zeros((n, len(FEATURES))), np.zeros(n, bool), np.zeros(n, int)
    reason = pd.Series([""] * n, index=raw.index, dtype=object)
    for i, f in enumerate(FEATURES):
        r = rules[f]
        if f in raw:
            num, miss = _to_numbers(raw[f])
        else:
            num, miss = np.full(n, np.nan), np.ones(n, bool)
        bad_f = ~miss & ~np.isfinite(num)                                
        ok_num = ~miss & np.isfinite(num)
        if r["kind"] == "category":
            bad_f |= ok_num & ~np.isin(num, r["allowed"])
        else:
            bad_f |= ok_num & ((num < r["hard_lo"]) | (num > r["hard_hi"]))
        reason = reason + np.where(bad_f, f + " galat; ", "")
        bad |= bad_f; n_miss += miss
        X[:, i] = np.where(miss | bad_f, r["fill"], num)
    too_many = n_miss > MAX_MISSING
    reason = reason + np.where(too_many, "bahut fields missing; ", "")
    return X, ~(bad | too_many), reason


def predict_batch(model, rules, raw):
    """CSV/DataFrame ke liye: ek hi predict_proba call poori valid rows par."""
    X, ok, reason = validate_batch(raw, rules)
    out = pd.DataFrame({"status": np.where(ok, "ok", "rejected"), "reason": reason.str.strip("; ").to_numpy(),
                        "prediction": np.nan, "probability_%": np.nan}, index=raw.index)
    if ok.any():
        p = model.predict_proba(pd.DataFrame(X[ok], columns=FEATURES))
        out.loc[ok, "prediction"] = model.classes_[p.argmax(axis=1)]
        out.loc[ok, "probability_%"] = (p.max(axis=1) * 100).round(2)
    return out

Overwriting input.py


In [7]:
import warnings; warnings.filterwarnings('ignore')
import joblib
import pandas as pd
from input import *

model = joblib.load('heart_model_v20261013_160903.joblib')
rules = load_rules('heart.csv')

pd.DataFrame(rules).T[['kind', 'min', 'max', 'hard_lo', 'hard_hi', 'allowed', 'fill']]

,kind,min,max,hard_lo,hard_hi,allowed,fill
age,number,29.0,77.0,17.0,89.0,NaN,55.5
sex,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",1.0
cp,category,0.0,3.0,NaN,NaN,"[0.0, 1.0, 2.0, 3.0]",0.0
trestbps,number,94.0,200.0,67.5,226.5,NaN,130.0
chol,number,126.0,564.0,16.5,673.5,NaN,240.5
fbs,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",0.0
restecg,category,0.0,2.0,NaN,NaN,"[0.0, 1.0, 2.0]",1.0
thalach,number,71.0,202.0,38.25,234.75,NaN,152.5
exang,category,0.0,1.0,NaN,NaN,"[0.0, 1.0]",0.0
oldpeak,number,0.0,6.2,0.0,7.75,NaN,0.8


In [8]:
good = dict(pd.read_csv('heart.csv').iloc[0][FEATURES])    
def with_(**kw):
    d = dict(good); d.update(kw); return d

cases = {
    'valid':                (good, True),
    'numbers as text':      ({k: str(v) for k, v in good.items()}, True),
    'comma decimal 2,3':    (with_(oldpeak='2,3'), True),
    'keys alag case/space': ({' Age ': 63, 'SEX': 1, **{k: v for k, v in good.items() if k not in ('age', 'sex')}}, True),
    '13 values ki list':    (list(good.values()), True),
    '1 missing (None)':     (with_(chol=None), True),
    '3 missing (mixed)':    (with_(chol='', trestbps='NA', thal=float('nan')), True),
    '5 missing':            (with_(chol=None, trestbps=None, thal=None, ca=None, slope=None), False),
    'empty dict':           ({}, False),
    'list too short':       ([1, 2], False),
    'not dict/list':        ('abc', False),
    'text in number':       (with_(age='sixty'), False),
    'age = 250':            (with_(age=250), False),
    'negative chol':        (with_(chol=-5), False),
    'inf value':            (with_(thalach=float('inf')), False),
    'unknown category':     (with_(cp=9), False),
    'category 1.5':         (with_(cp=1.5), False),
    'chol 600 (range ke bahar)': (with_(chol=600), True),
    'extra field':          ({**good, 'name': 'Riya'}, True),
}
rows = []
for name, (rec, expected) in cases.items():
    r = validate_input(rec, rules)
    rows.append((name, expected, r['ok'], expected == r['ok'], (r['errors'] + r['warnings'] or [''])[0][:60]))
res = pd.DataFrame(rows, columns=['case', 'expected_ok', 'got_ok', 'match', 'message'])
print('Matched:', res['match'].sum(), 'of', len(res))
res

Matched: 19 of 19


,case,expected_ok,got_ok,match,message
0,valid,True,True,True,
1,numbers as text,True,True,True,
2,"comma decimal 2,3",True,True,True,
3,keys alag case/space,True,True,True,
4,13 values ki list,True,True,True,
5,1 missing (None),True,True,True,missing fields median/mode se bhare gaye: ['ch...
6,3 missing (mixed),True,True,True,missing fields median/mode se bhare gaye: ['tr...
7,5 missing,False,False,True,5 fields missing hain (max 3 allowed)
8,empty dict,False,False,True,13 fields missing hain (max 3 allowed)
9,list too short,False,False,True,"13 values chahiye, 2 mili"


In [9]:
messy = pd.DataFrame([good, with_(age='abc'), with_(chol=None), with_(cp=7), with_(trestbps=900),
                      with_(thal=None, ca=None, slope=None, oldpeak=None), good])
pd.concat([messy[['age', 'cp', 'chol', 'trestbps']], predict_batch(model, rules, messy)], axis=1)

,age,cp,chol,trestbps,status,reason,prediction,probability_%
0,63.0,3.0,233.0,145.0,ok,,1.0,78.0
1,abc,3.0,233.0,145.0,rejected,age galat,NaN,NaN
2,63.0,3.0,NaN,145.0,ok,,1.0,78.0
3,63.0,7.0,233.0,145.0,rejected,cp galat,NaN,NaN
4,63.0,3.0,233.0,900.0,rejected,trestbps galat,NaN,NaN
5,63.0,3.0,233.0,145.0,rejected,bahut fields missing,NaN,NaN
6,63.0,3.0,233.0,145.0,ok,,1.0,78.0


In [10]:
import time
records = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].to_dict('records')[:100]

def old_way(rec):                                   
    d = pd.DataFrame([list(rec.values())], columns=FEATURES)
    return model.predict(d)[0], model.predict_proba(d)[0]

def new_way(rec):                                 
    return predict_result(model, validate_input(rec, rules)['x'])

predictor = make_predictor(model)
def new_cached(rec):                               
    return predictor(validate_input(rec, rules)['x'])

def ms_per_call(fn):
    t = time.perf_counter(); [fn(r) for r in records]; return (time.perf_counter() - t) / len(records) * 1000

ms_per_call(old_way)                              
speed = {'purana (2 pass, bina validation)': ms_per_call(old_way), 'naya: validation + 1 pass': ms_per_call(new_way)}
ms_per_call(new_cached); speed['naya + cache (same input dobara)'] = ms_per_call(new_cached)

big = pd.concat([pd.read_csv('heart.csv')[FEATURES]] * 100)      
t = time.perf_counter(); out = predict_batch(model, rules, big); speed['batch (30,300 rows, per row)'] = (time.perf_counter() - t) / len(big) * 1000

same = all(old_way(r)[0] == new_way(r)[0] for r in records)
print('Prediction purane tareeke jaisi hi hai:', same)
pd.Series(speed, name='ms per prediction').round(3)

Prediction purane tareeke jaisi hi hai: True


purana (2 pass, bina validation)    15.759
naya: validation + 1 pass            7.853
naya + cache (same input dobara)     0.074
batch (30,300 rows, per row)         0.007
Name: ms per prediction, dtype: float64

## Task 2: Display model predictions dynamically by implementing custom caching mechanisms to improve performance, ensuring smooth updates without unnecessary reloading, and handling concurrent user requests efficiently.

In [11]:
%%writefile cache.py
"""
HeartCare - custom caching for predictions (Streamlit ka built-in cache nahi).

- LRU     : size limit; sabse purani entry hatti hai
- TTL     : entry kuch der baad expire (model/data purana na ho)
- Thread-safe : Lock se
- Single-flight : same input par 10 users ek saath aaye to model sirf 1 baar chalta hai,
                  baaki 9 wahi result ka wait karte hain (concurrent requests efficient)
"""
import time
import threading
from collections import OrderedDict


class _Call:
    """Jo computation chal rahi hai, uska result yahan rakhte hain."""
    def __init__(self):
        self.done = threading.Event()
        self.value = None
        self.error = None


class PredictionCache:

    def __init__(self, max_size=512, ttl=600):
        self.max_size = max_size
        self.ttl = ttl                    
        self._data = OrderedDict()          
        self._inflight = {}                
        self._lock = threading.Lock()
        self.hits = self.misses = self.waits = self.evictions = self.expired = 0

    def get_or_compute(self, key, compute):
        """Cache me ho to wahi do, warna compute() chalao (same key par ek hi baar)."""
        with self._lock:
            item = self._data.get(key)
            if item is not None:
                value, created = item
                if self.ttl is None or time.monotonic() - created < self.ttl:
                    self._data.move_to_end(key)        
                    self.hits += 1
                    return value
                del self._data[key]                    
                self.expired += 1

            call = self._inflight.get(key)
            owner = call is None
            if owner:
                call = self._inflight[key] = _Call()
                self.misses += 1
            else:
                self.waits += 1

        if owner:
            try:
                call.value = compute()                 
            except BaseException as error:
                call.error = error                       
            with self._lock:
                if call.error is None:
                    self._data[key] = (call.value, time.monotonic())
                    self._data.move_to_end(key)
                    while len(self._data) > self.max_size:
                        self._data.popitem(last=False)
                        self.evictions += 1
                del self._inflight[key]
            call.done.set()
        else:
            call.done.wait()

        if call.error is not None:
            raise call.error
        return call.value

    def clear(self):
        with self._lock:
            self._data.clear()

    def stats(self):
        with self._lock:
            total = self.hits + self.misses + self.waits
            return {"size": len(self._data), "hits": self.hits, "misses": self.misses,
                    "waits": self.waits, "evictions": self.evictions, "expired": self.expired,
                    "hit_rate_%": round(100 * (self.hits + self.waits) / total, 1) if total else 0.0}

Overwriting cache.py


In [12]:
import time
from cache import PredictionCache

small = PredictionCache(max_size=2, ttl=0.5)
runs = []
def work(v):
    runs.append(v); return v * 10

small.get_or_compute('a', lambda: work(1)); small.get_or_compute('a', lambda: work(1))   
small.get_or_compute('b', lambda: work(2)); small.get_or_compute('c', lambda: work(3))   
print('model/compute kitni baar chala:', len(runs), '(3 alag keys, 1 repeat)')
small.get_or_compute('a', lambda: work(1))                                                  
print('evicted key dobara compute hui:', runs.count(1) == 2)
time.sleep(0.6)
small.get_or_compute('c', lambda: work(3))                                                
print(small.stats())

model/compute kitni baar chala: 3 (3 alag keys, 1 repeat)
evicted key dobara compute hui: True
{'size': 2, 'hits': 1, 'misses': 5, 'waits': 0, 'evictions': 2, 'expired': 1, 'hit_rate_%': 16.7}


In [13]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

patients = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].values[:10]
requests = [patients[i % 10] for i in range(60)]

def predict_p1(x):
    return model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0, 1]

t = time.perf_counter()
with ThreadPoolExecutor(12) as pool: plain = list(pool.map(predict_p1, requests))
t_plain = time.perf_counter() - t

cache = PredictionCache(); runs = {'n': 0}
def cached_predict(x):
    def compute():
        runs['n'] += 1
        return predict_p1(x)
    return cache.get_or_compute(tuple(x), compute)

t = time.perf_counter()
with ThreadPoolExecutor(12) as pool: cached = list(pool.map(cached_predict, requests))
t_cache = time.perf_counter() - t

print(f'Bina cache : 60 model runs, {t_plain * 1000:.0f} ms')
print(f'Custom cache: {runs["n"]} model runs, {t_cache * 1000:.0f} ms')
print('Dono ke results same:', np.allclose(plain, cached))
print(cache.stats())

Bina cache : 60 model runs, 806 ms
Custom cache: 10 model runs, 137 ms
Dono ke results same: True
{'size': 10, 'hits': 47, 'misses': 10, 'waits': 3, 'evictions': 0, 'expired': 0, 'hit_rate_%': 83.3}


## Task 3: Visualize prediction confidence using bar charts by manually computing probability distributions, implementing real-time updates with animations, and allowing users to interactively explore model confidence levels.

In [14]:
%%writefile confidence.py
"""
HeartCare - prediction confidence ko bar charts me dikhana.

1. Probability distribution HAATH se: Random Forest ki har tree ka probability nikal ke khud average karte hain
   (model.predict_proba ka use sirf check karne ke liye).
2. Charts HTML + CSS se bante hain. Har bar purani value se nayi value tak animate hota hai (CSS keyframes).
3. Interactive: threshold slider, what-if (ek feature badalke confidence kaise badalti hai).
"""
import numpy as np
import pandas as pd

FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
CLASS_NAMES = {0: "Lower Risk", 1: "Higher Risk"}       
COLORS = {0: "#177245", 1: "#b42323"}          
N_BINS = 10


def analyse(model, x):
    """Ek patient ke liye poori confidence picture. Cache me yahi dict jati hai."""
    x32 = np.ascontiguousarray(x, dtype=np.float32).reshape(1, -1)             
    per_tree = np.array([t.predict_proba(x32, check_input=False)[0] for t in model.estimators_])
    probs = per_tree.mean(axis=0)                                                  
    votes = np.bincount(per_tree.argmax(axis=1), minlength=len(probs))           
    hist = np.histogram(per_tree[:, 1], bins=N_BINS, range=(0, 1))[0]              
    return {"probs": probs, "votes": votes, "hist": hist, "n_trees": len(per_tree),
            "std": float(per_tree[:, 1].std()), "x": np.asarray(x, dtype=float)}


def decide(probs, threshold=0.5):
    """Threshold ke hisaab se class (default 0.5 = model.predict jaisa)."""
    return int(probs[1] > threshold)


def sweep(model, x, feature, values):
    """What-if: ek feature ki value badalke P(class 1) (ek hi predict_proba call me)."""
    rows = np.tile(np.asarray(x, dtype=float), (len(values), 1))
    rows[:, FEATURES.index(feature)] = values
    return model.predict_proba(pd.DataFrame(rows, columns=FEATURES))[:, 1]


def sweep_values(rule, n=10):
    """Rules (input.py) se feature ki values: category = allowed, number = n barabar points."""
    if rule["kind"] == "category":
        return [float(v) for v in rule["allowed"]]
    return [float(v) for v in np.round(np.linspace(rule["min"], rule["max"], n), 1)]


CSS = """
<style>
.hc-chart { background: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 16px; margin-bottom: 14px; }
.hc-chart h4 { margin: 0 0 12px 0; color: #142b53; font-size: 16px; }
.hc-row { display: flex; align-items: center; margin: 8px 0; }
.hc-label { width: 120px; font-size: 14px; color: #334155; }
.hc-track { position: relative; flex: 1; height: 22px; background: #eef2f7; border-radius: 6px; overflow: hidden; }
.hc-fill { height: 100%; width: var(--to); animation: hc-grow 0.9s ease-out; }
.hc-value { width: 64px; text-align: right; font-size: 14px; font-weight: 600; color: #142b53; }
.hc-thr { position: absolute; top: 0; bottom: 0; width: 2px; background: #142b53; }
.hc-cols { display: flex; align-items: flex-end; height: 130px; gap: 6px; border-bottom: 1px solid #cbd5e1; }
.hc-colwrap { flex: 1; height: 100%; display: flex; flex-direction: column; justify-content: flex-end; align-items: center; }
.hc-col { width: 100%; height: var(--to); border-radius: 4px 4px 0 0; animation: hc-rise 0.9s ease-out; }
.hc-colnum { font-size: 11px; color: #475569; height: 14px; }
.hc-xlabels { display: flex; gap: 6px; margin-top: 4px; }
.hc-xlabels span { flex: 1; text-align: center; font-size: 11px; color: #475569; }
.hc-note { font-size: 13px; color: #64748b; margin-top: 8px; }
@keyframes hc-grow { from { width: var(--from); } to { width: var(--to); } }
@keyframes hc-rise { from { height: var(--from); } to { height: var(--to); } }
@media (max-width: 600px) { .hc-label { width: 84px; font-size: 12px; } .hc-value { width: 52px; } }
@media (prefers-reduced-motion: reduce) { .hc-fill, .hc-col { animation: none; } }
</style>
"""


def _prev(prev, i, scale):
    """Purani value (animation ka start). Pehli baar 0 se grow hota hai."""
    return 0.0 if prev is None or i >= len(prev) else float(prev[i]) * scale


def bar_chart(title, labels, values, prev=None, colors=None, threshold=None, note=""):
    """Horizontal bars. values 0-1 (probability). threshold = vertical line."""
    rows = ""
    for i, (lab, v) in enumerate(zip(labels, values)):
        color = colors[i] if colors else "#244577"
        thr = f'<div class="hc-thr" style="left:{threshold * 100:.1f}%"></div>' if threshold is not None else ""
        rows += (f'<div class="hc-row"><div class="hc-label">{lab}</div>'
                 f'<div class="hc-track"><div class="hc-fill" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div>{thr}</div>'
                 f'<div class="hc-value">{v * 100:.1f}%</div></div>')
    note_html = f'<div class="hc-note">{note}</div>' if note else ""
    return f'<div class="hc-chart"><h4>{title}</h4>{rows}{note_html}</div>'


def column_chart(title, x_labels, values, prev=None, color="#244577", counts=None, note=""):
    """Vertical bars. values 0-1 (height ka fraction); counts = bar ke upar likha number."""
    cols, labs = "", ""
    for i, (lab, v) in enumerate(zip(x_labels, values)):
        shown = counts[i] if counts is not None else f"{v * 100:.0f}%"
        cols += (f'<div class="hc-colwrap"><div class="hc-colnum">{shown}</div>'
                 f'<div class="hc-col" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div></div>')
        labs += f"<span>{lab}</span>"
    note_html = f'<div class="hc-note">{note}</div>' if note else ""
    return f'<div class="hc-chart"><h4>{title}</h4><div class="hc-cols">{cols}</div><div class="hc-xlabels">{labs}</div>{note_html}</div>'

Overwriting confidence.py


In [15]:
import numpy as np
import importlib, confidence
importlib.reload(confidence)
from confidence import *

X_all = pd.read_csv('heart.csv').drop_duplicates()[FEATURES].values
manual = np.array([analyse(model, x)['probs'] for x in X_all])
sk = model.predict_proba(pd.DataFrame(X_all, columns=FEATURES))
print('Patients:', len(X_all), '| max |manual - sklearn| =', np.abs(manual - sk).max())
print('Prediction same (threshold 50%):', all(decide(p) == model.classes_[k] for p, k in zip(manual, sk.argmax(1))))

Patients: 302 | max |manual - sklearn| = 0.0
Prediction same (threshold 50%): True


In [16]:
from IPython.display import HTML

x = X_all[0]
info = analyse(model, x)
labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
print('Probabilities:', info['probs'], '| votes:', info['votes'], '| tree spread (std):', round(info['std'], 3))

HTML(CSS
     + bar_chart('Class probabilities (trees ka average)', labels, info['probs'], colors=[COLORS[0], COLORS[1]], threshold=0.5)
     + bar_chart('Tree votes', labels, info['votes'] / info['n_trees'], colors=[COLORS[0], COLORS[1]])
     + column_chart('Har tree ka Higher Risk probability (%)', [f'{i*10}-{i*10+10}' for i in range(10)],
                    info['hist'] / info['hist'].max(), counts=[int(h) for h in info['hist']]))

Probabilities: [0.22 0.78] | votes: [22 78] | tree spread (std): 0.414


In [17]:
for thr in (0.3, 0.5, 0.7, 0.9):
    print(f'threshold {thr:.0%}: prediction = {CLASS_NAMES[decide(info["probs"], thr)]}')

rules = load_rules('heart.csv')
feature = 'cp'
values = sweep_values(rules[feature])
p_values = sweep(model, x, feature, values)
HTML(CSS + column_chart(f'What-if: {feature} badalne par {CLASS_NAMES[1]} probability', [f'{v:g}' for v in values], p_values, color='#d64545',
                        note='Baaki saari values wahi rakhi gayi hain; sirf ' + feature + ' badla.'))

threshold 30%: prediction = Higher Risk
threshold 50%: prediction = Higher Risk
threshold 70%: prediction = Higher Risk
threshold 90%: prediction = Lower Risk


In [18]:
%%writefile app_v1.py

import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

from input import FEATURES, load_rules, validate_input, predict_batch    
from cache import PredictionCache                                         
from confidence import (CSS, CLASS_NAMES, COLORS, analyse, decide, sweep, 
                        sweep_values, bar_chart, column_chart)

st.set_page_config(
    page_title="HeartCare",
    layout="centered"
)

MODEL_FILE = "heart_model_v20261013_160903.joblib"
DATA_FILE = "heart.csv"
MAX_BATCH_ROWS = 100000


@st.cache_resource
def load_everything():
    model = joblib.load(MODEL_FILE)
    rules = load_rules(DATA_FILE)
    cache = PredictionCache(max_size=512, ttl=600)
    return model, rules, cache, MODEL_FILE


try:
    model, rules, cache, model_version = load_everything()
except FileNotFoundError as e:
    st.error(f"File nahi mili: {e.filename}. app.py ke same folder me rakho.")
    st.stop()


def hint(f):
    r = rules[f]
    if r["kind"] == "category":
        return "/".join(str(int(a)) for a in r["allowed"])
    return f"{r['min']:g} to {r['max']:g}"


def choice(label, f):
    return st.selectbox(f"{label} ({hint(f)})", [None] + rules[f]["allowed"],
                        index=1, format_func=lambda v: "Not known" if v is None else str(int(v)))


def number(label, f, default):
    return st.text_input(f"{label} ({hint(f)})", value=default)


def previous(key, current):
    """Chart ki pichli values (animation start). Session me yaad rakhte hain."""
    old = st.session_state.get("prev", {}).get(key)
    st.session_state.setdefault("prev", {})[key] = list(current)
    return old


st.markdown("""
<style>
.stApp {
    background: #f3f6fb;
}

header, footer {
    visibility: hidden;
}

.main .block-container {
    max-width: 850px;
    padding: 25px 20px;
}

.header {
    background: #142b53;
    color: white;
    padding: 25px;
    border-radius: 14px;
    text-align: center;
    margin-bottom: 25px;
}

.section {
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 18px;
    border: 1px solid #e2e8f0;
}

.section h3 {
    color: #142b53;
}

.stButton button {
    background: #142b53;
    color: white;
    width: 100%;
    height: 45px;
    border-radius: 8px;
}

.stButton button:hover {
    background: #244577;
    color: white;
}

@media(max-width: 600px) {
    .main .block-container {
        padding: 12px 10px;
    }

    .header {
        padding: 18px;
    }

    .header h1 {
        font-size: 24px;
    }

    .section {
        padding: 14px;
    }
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>HeartCare</h1>
    <p>Heart Disease Risk Prediction System</p>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="section">
    <h3>Patient Information</h3>
    <p>Enter patient medical details below.</p>
</div>
""", unsafe_allow_html=True)

st.caption("Koi field khali chhodo ya 'Not known' chuno to median/mode se bhar diya jayega (max 3 fields).")

age = number("Age", "age", "45")
sex = choice("Sex", "sex")
cp = choice("Chest Pain Type", "cp")
trestbps = number("Resting Blood Pressure", "trestbps", "120")
chol = number("Cholesterol", "chol", "200")
fbs = choice("Fasting Blood Sugar", "fbs")
restecg = choice("Resting ECG", "restecg")
thalach = number("Maximum Heart Rate", "thalach", "150")
exang = choice("Exercise Induced Angina", "exang")
oldpeak = number("Oldpeak", "oldpeak", "1.0")
slope = choice("Slope", "slope")
ca = choice("Major Vessels", "ca")
thal = choice("Thal", "thal")

live = st.checkbox("Live update (input badalte hi prediction update ho)")
clicked = st.button("Predict Heart Disease Risk")

if clicked or live:

    record = dict(zip(FEATURES, [
        age, sex, cp, trestbps, chol,
        fbs, restecg, thalach, exang,
        oldpeak, slope, ca, thal
    ]))

    check = validate_input(record, rules)

    for message in check["errors"]:
        st.error(message)

    if check["ok"]:
        st.session_state["check"] = check    
    else:
        st.session_state.pop("check", None)

check = st.session_state.get("check")

if check:

    st.markdown(CSS, unsafe_allow_html=True)

    st.markdown("""
<div class="section">
    <h3>Prediction & Confidence</h3>
</div>
""", unsafe_allow_html=True)

    threshold = st.slider("Decision threshold (%): Higher Risk tab maana jaye jab probability isse zyada ho", 1, 99, 50) / 100
    explore = st.selectbox("What-if: ek feature badalke confidence dekho", ["(none)"] + FEATURES)

    start = time.perf_counter()
    key = (model_version, tuple(float(v) for v in check["x"]))
    info = cache.get_or_compute(key, lambda: analyse(model, check["x"]))
    elapsed = (time.perf_counter() - start) * 1000

    for message in check["warnings"]:
        st.warning(message)

    probs = info["probs"]
    prediction = decide(probs, threshold)

    if prediction == 1:
        st.error(f"{CLASS_NAMES[1]} Prediction")
    else:
        st.success(f"{CLASS_NAMES[0]} Prediction")

    st.write("Model Confidence:", round(max(probs) * 100, 2), "%")

    if threshold != 0.5:
        st.caption(f"Custom threshold {threshold * 100:.0f}%: {CLASS_NAMES[1]} probability {probs[1] * 100:.1f}% hai, "
                   f"isliye prediction {CLASS_NAMES[prediction]} dikh raha hai (model ka default threshold 50% hai).")
    st.caption(f"{info['votes'][1]} of {info['n_trees']} trees ne {CLASS_NAMES[1]} chuna | trees ke beech spread (std): {info['std']:.2f}")

    labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
    st.markdown(bar_chart("Class probabilities (trees ka average)", labels, probs,
                          prev=previous("probs", probs), colors=[COLORS[0], COLORS[1]], threshold=threshold,
                          note=f"Kali line = threshold ({threshold * 100:.0f}%)"), unsafe_allow_html=True)

    votes = info["votes"] / info["n_trees"]
    st.markdown(bar_chart("Tree votes (kitni trees ne kaunsi class chuni)", labels, votes,
                          prev=previous("votes", votes), colors=[COLORS[0], COLORS[1]]), unsafe_allow_html=True)

    hist = info["hist"]
    bins = [f"{i * 10}-{i * 10 + 10}" for i in range(len(hist))]
    st.markdown(column_chart(f"Har tree ka {CLASS_NAMES[1]} probability (%) - kitni trees kis range me",
                             bins, hist / hist.max(), prev=previous("hist", hist / hist.max()),
                             color="#244577", counts=[int(h) for h in hist],
                             note="Bar ke upar tree ginti. Sab trees ek jaisi ho to bars sirf kinaron par aate hain."),
                unsafe_allow_html=True)

    if explore != "(none)":
        values = sweep_values(rules[explore])
        p_values = sweep(model, check["x"], explore, values)
        st.markdown(column_chart(f"What-if: {explore} badalne par {CLASS_NAMES[1]} probability",
                                 [f"{v:g}" for v in values], p_values, prev=previous("sweep_" + explore, p_values),
                                 color="#d64545", note=f"Baaki saari values wahi rakhi gayi hain; sirf {explore} badla."),
                    unsafe_allow_html=True)

    if not live:
        st.caption("Ye result pichhle Predict click ke inputs ka hai. Naye inputs ke liye dobara Predict dabao ya Live update on karo.")

    stats = cache.stats()
    st.caption(f"Is request me {elapsed:.2f} ms | Cache: {stats['hits']} hits, {stats['misses']} misses, "
               f"{stats['waits']} coalesced, hit rate {stats['hit_rate_%']}% (size {stats['size']})")

st.markdown("""
<div class="section">
    <h3>Batch Prediction (CSV)</h3>
    <p>Bahut patients ek saath: CSV me columns age, sex, cp, trestbps, chol, fbs,
    restecg, thalach, exang, oldpeak, slope, ca, thal hone chahiye.</p>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Upload CSV", type=["csv"])

data = None

if uploaded is not None:

    try:
        data = pd.read_csv(uploaded)
    except Exception:
        st.error("CSV padh nahi paye. File sahi CSV format me honi chahiye.")

if data is not None:

    clean_names = [str(c).strip().lower().replace(" ", "_") for c in data.columns]
    missing_cols = [f for f in FEATURES if f not in clean_names]

    if len(data) == 0:
        st.error("CSV khali hai.")
    elif len(data) > MAX_BATCH_ROWS:
        st.error(f"Max {MAX_BATCH_ROWS} rows allowed hain.")
    else:
        if missing_cols:
            st.warning(f"Ye columns CSV me nahi mile (missing maane gaye): {missing_cols}")

        start = time.perf_counter()
        results = predict_batch(model, rules, data)
        elapsed = (time.perf_counter() - start) * 1000

        rejected = int((results["status"] == "rejected").sum())
        st.write(f"Rows: {len(data)} | Predicted: {len(data) - rejected} | Rejected: {rejected}")
        st.caption(f"{elapsed:.0f} ms ({elapsed / len(data):.3f} ms per row)")

        output = pd.concat([data, results], axis=1)
        st.dataframe(output.head(100))

        st.download_button("Download results", output.to_csv(index=False), "heart_predictions.csv", "text/csv")

st.markdown("""
<div style="text-align:center;color:#64748b;padding:20px;">
    HeartCare | Educational Purpose Only
    <br>
    Not a substitute for professional medical diagnosis.
</div>
""", unsafe_allow_html=True)

Overwriting app_v1.py


## Task 4: Test UI for usability by conducting user experience research, gathering qualitative feedback, identifying pain points in interaction flow, and iteratively refining the interface based on usability heuristics.

In [19]:
import pandas as pd
v1_issues = pd.DataFrame([
    ('P1', 'Age/BP/Chol/Thalach/Oldpeak text box me type karne padte hain; galat type karo to Predict ke baad hi error milta hai', 'H5 Error prevention', 3, 'Number fields ab slider (range data se)'),
    ('P2', 'Predict dabane ke baad result 13 fields ke NEECHE aata hai; scroll karna padta hai', 'H1 Visibility of system status', 3, 'Result card form ke UPAR (st.empty) - slider hilate hi dikhta hai'),
    ('P3', 'Live update default OFF; result purane input ka hota hai (stale)', 'H1 Visibility of system status', 2, 'Live update default ON + risk card par pichhle update se farak'),
    ('P4', 'Sab inputs ek saath dobara default par laane ka koi button nahi', 'H3 User control and freedom', 2, 'Reset to typical values button'),
    ('P5', '0/1/2/3 codes bina hint ke (sex, cp, thal ...)', 'H2 Match with real world / H10 Help', 3, 'Har field par hover/tap help (?). Poora meaning dataset me documented nahi - open issue'),
    ('P6', '"Not known" kabhi selectbox me, kabhi khali text box - alag-alag tareeke', 'H4 Consistency', 2, 'Ek jagah: Values I don\'t know (max 3) + estimated marker'),
    ('P7', '13 fields ek lambi flat list', 'H8 Aesthetic and minimalist design', 2, '4 chhote groups (Basic / Chest / Tests / Scan)'),
    ('P8', 'Prediction kyun aaya - koi explanation nahi', 'H1 / H10', 2, 'SHAP section: har feature ka contribution'),
    ('P9', 'Value kitni "normal" hai ye pata nahi chalta', 'H6 Recognition rather than recall', 1, 'Har slider ke neeche percentile line'),
    ('P10', 'App par feedback dene ki jagah nahi', 'H9 / research', 1, 'Feedback form -> feedback.csv'),
], columns=['id', 'pain point', 'heuristic', 'severity', 'fix in v2']).sort_values('severity', ascending=False)
print('Total issues:', len(v1_issues), '| severity >= 3:', (v1_issues.severity >= 3).sum())
v1_issues

Total issues: 10 | severity >= 3: 3


,id,pain point,heuristic,severity,fix in v2
0,P1,Age/BP/Chol/Thalach/Oldpeak text box me type k...,H5 Error prevention,3,Number fields ab slider (range data se)
1,P2,Predict dabane ke baad result 13 fields ke NEE...,H1 Visibility of system status,3,Result card form ke UPAR (st.empty) - slider h...
4,P5,"0/1/2/3 codes bina hint ke (sex, cp, thal ...)",H2 Match with real world / H10 Help,3,Har field par hover/tap help (?). Poora meanin...
2,P3,Live update default OFF; result purane input k...,H1 Visibility of system status,2,Live update default ON + risk card par pichhle...
3,P4,Sab inputs ek saath dobara default par laane k...,H3 User control and freedom,2,Reset to typical values button
5,P6,"""Not known"" kabhi selectbox me, kabhi khali te...",H4 Consistency,2,Ek jagah: Values I don't know (max 3) + estima...
6,P7,13 fields ek lambi flat list,H8 Aesthetic and minimalist design,2,4 chhote groups (Basic / Chest / Tests / Scan)
7,P8,Prediction kyun aaya - koi explanation nahi,H1 / H10,2,SHAP section: har feature ka contribution
8,P9,"Value kitni ""normal"" hai ye pata nahi chalta",H6 Recognition rather than recall,1,Har slider ke neeche percentile line
9,P10,App par feedback dene ki jagah nahi,H9 / research,1,Feedback form -> feedback.csv


In [20]:
v2_open = pd.DataFrame([
    ('Q1', 'Categories ke codes (sex, cp, thal...) ka asli meaning dataset me nahi diya; sirf "code 0-3" bata paate hain', 'H2 Match with real world', 3, 'Dataset documentation chahiye; tab tak label nahi badal sakte (galat label ka risk)'),
    ('Q2', 'Phone par Cholesterol slider (126-564) me ek-ek unit ka fine control mushkil ho sakta hai', 'H7 Flexibility and efficiency', 2, 'Real users ke saath phone par test chahiye; zaroorat ho to number box add karo'),
    ('Q3', '"pts" (percentage points) shabd naye user ko samajh na aaye', 'H2 Match with real world', 2, 'Feedback form me poocho; "points" ka example line add kar sakte hain'),
    ('Q4', 'Slider sirf dataset range (min-max) tak jaata hai; CSV batch wider range accept karta hai', 'H4 Consistency', 1, 'Intentional: single patient UI ko safe rakha'),
    ('Q5', 'Koi real user test nahi hua - upar ke saare ratings mera apna evaluation hain', 'Method limit', 4, 'Feedback form + study plan (neeche) se real users se data lo'),
], columns=['id', 'open issue', 'heuristic', 'severity', 'next step'])
print('v1 issues fixed (P1-P4, P6-P10): 9 of 10 | P5 partly fixed (tooltip) -> Q1')
v2_open

v1 issues fixed (P1-P4, P6-P10): 9 of 10 | P5 partly fixed (tooltip) -> Q1


,id,open issue,heuristic,severity,next step
0,Q1,"Categories ke codes (sex, cp, thal...) ka asli...",H2 Match with real world,3,Dataset documentation chahiye; tab tak label n...
1,Q2,Phone par Cholesterol slider (126-564) me ek-e...,H7 Flexibility and efficiency,2,Real users ke saath phone par test chahiye; za...
2,Q3,"""pts"" (percentage points) shabd naye user ko s...",H2 Match with real world,2,"Feedback form me poocho; ""points"" ka example l..."
3,Q4,Slider sirf dataset range (min-max) tak jaata ...,H4 Consistency,1,Intentional: single patient UI ko safe rakha
4,Q5,Koi real user test nahi hua - upar ke saare ra...,Method limit,4,Feedback form + study plan (neeche) se real us...


In [21]:
%%writefile feedback.py
"""
HeartCare - usability feedback ikatha karna aur analyze karna.

- add_feedback : app ke andar se feedback CSV me (thread-safe, health inputs store nahi hote)
- summarize    : success rate, ease rating, device-wise, aur comments ke themes (manual keyword coding)
- sus_score    : System Usability Scale (10 sawaal) ka standard scoring
"""
import csv
import os
import threading
import time
from collections import Counter

import pandas as pd

FIELDS = ["time", "version", "device", "task_success", "ease_1to5", "comment"]
_LOCK = threading.Lock()

THEMES = {
    "labels/jargon":  ["confus", "unclear", "jargon", "term", "medical", "understand", "samajh", "meaning", "label"],
    "length/scroll":  ["long", "scroll", "many", "too much", "lamba", "lambi", "zyada"],
    "mobile/touch":   ["phone", "mobile", "small", "touch", "tap"],
    "speed":          ["slow", "lag", "wait", "loading", "dhima"],
    "charts/results": ["chart", "graph", "bar", "shap", "result", "explanation"],
    "inputs":         ["slider", "type", "typing", "input", "field", "default"],
}


def _safe(text, limit=500):
    """CSV injection se bachao: =,+,-,@ se shuru text ke aage ' laga do. Length limit."""
    text = str(text).replace("\r", " ").replace("\n", " ").strip()[:limit]
    return "'" + text if text[:1] in ("=", "+", "-", "@") else text


def add_feedback(path, device, task_success, ease, comment="", version="v2"):
    """Ek feedback row save karo. Galat values par ValueError."""
    if task_success not in ("Yes", "Partly", "No"):
        raise ValueError("task_success: Yes / Partly / No")
    if not 1 <= int(ease) <= 5:
        raise ValueError("ease 1 se 5 ke beech hona chahiye")
    row = [time.strftime("%Y-%m-%d %H:%M:%S"), version, _safe(device, 30), task_success, int(ease), _safe(comment)]
    with _LOCK:                                                   
        new = not os.path.exists(path)
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new:
                w.writerow(FIELDS)
            w.writerow(row)


def load_feedback(path):
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=FIELDS)


def summarize(df):
    """Feedback ka summary. Kam responses (n<5) par conclusions pakke nahi hote, isliye n bhi return hota hai."""
    if len(df) == 0:
        return {"n": 0}
    themes = Counter()
    for c in df["comment"].fillna("").str.lower():
        for name, words in THEMES.items():
            if any(w in c for w in words):
                themes[name] += 1
    return {"n": len(df),
            "success_rate_%": round((df["task_success"] == "Yes").mean() * 100, 1),
            "mean_ease": round(df["ease_1to5"].mean(), 2),
            "by_device": df.groupby("device")["ease_1to5"].agg(["count", "mean"]).round(2).to_dict("index"),
            "themes": dict(themes.most_common())}


def sus_score(answers):
    """SUS: 10 jawab (1-5). Odd sawaal: jawab-1, even sawaal: 5-jawab, total x 2.5 -> 0-100."""
    if len(answers) != 10 or any(not 1 <= a <= 5 for a in answers):
        raise ValueError("SUS ke liye 10 jawab chahiye, har ek 1 se 5")
    return sum((a - 1) if i % 2 == 0 else (5 - a) for i, a in enumerate(answers)) * 2.5

Overwriting feedback.py


In [22]:
import os, tempfile, importlib
import feedback; importlib.reload(feedback)
from feedback import add_feedback, load_feedback, summarize, sus_score

tmp = os.path.join(tempfile.mkdtemp(), 'test_feedback.csv')
tests = [('Phone', 'Yes', 5, 'bahut aasaan'), ('Phone', 'Partly', 2, 'slider chhota hai, value set karna mushkil'),
         ('Laptop', 'Yes', 4, ''), ('Tablet', 'No', 1, 'result samajh nahi aaya, confusing'),
         ('Phone', 'Yes', 3, '=HYPERLINK("http://x")  <- CSV injection test')]
for d, s, e, c in tests: add_feedback(tmp, d, s, e, c)
print('--- bad input reject hota hai?')
for args in [('Phone', 'Maybe', 3), ('Phone', 'Yes', 9)]:
    try: add_feedback(tmp, *args)
    except ValueError as err: print('  rejected:', args, '->', err)
print('--- summary (sirf TEST data)')
df_fb = load_feedback(tmp); s = summarize(df_fb); print(s)
print('--- injection saaf hua?', df_fb.comment.iloc[-1])
print('--- SUS check: all best =', sus_score([5, 1] * 5), '| all neutral =', sus_score([3] * 10), '| all worst =', sus_score([1, 5] * 5))

--- bad input reject hota hai?
  rejected: ('Phone', 'Maybe', 3) -> task_success: Yes / Partly / No
  rejected: ('Phone', 'Yes', 9) -> ease 1 se 5 ke beech hona chahiye
--- summary (sirf TEST data)
{'n': 5, 'success_rate_%': np.float64(60.0), 'mean_ease': np.float64(3.0), 'by_device': {'Laptop': {'count': 1, 'mean': 4.0}, 'Phone': {'count': 3, 'mean': 3.33}, 'Tablet': {'count': 1, 'mean': 1.0}}, 'themes': {'inputs': 1, 'labels/jargon': 1, 'charts/results': 1}}
--- injection saaf hua? '=HYPERLINK("http://x")  <- CSV injection test
--- SUS check: all best = 100.0 | all neutral = 50.0 | all worst = 0.0


## Task 5: Add interactive graphs and sliders for inputs by implementing custom visualization components, ensuring compatibility with multiple device types, and dynamically linking input changes to real-time model updates.

In [23]:
%%writefile components.py
"""
HeartCare - custom visualization components (HTML + inline SVG + CSS, koi chart library nahi).

- risk_card   : live result card (sliders hilte hi update; purani value se nayi tak animate)
- curve_chart : interactive risk curve (SVG): population histogram + model ka risk curve + "aap yahan hain" marker
- percentile_text : slider ke neeche "aap dataset me kahan hain"

Multiple devices: SVG viewBox se chart apne aap scale hota hai, cards flex-wrap karte hain,
fonts mobile par chhote nahi hote, animations 'reduced motion' setting ko maante hain.
"""
import numpy as np

from confidence import CLASS_NAMES, COLORS

CSS = """
<style>
.ui-card { background: white; border: 1px solid #e2e8f0; border-radius: 14px; padding: 16px 18px; margin-bottom: 14px; }
.ui-top { display: flex; flex-wrap: wrap; align-items: center; gap: 12px 20px; }
.ui-big { font-size: 40px; font-weight: 700; line-height: 1.1; color: #142b53; }
.ui-label { font-size: 16px; font-weight: 600; }
.ui-sub { font-size: 13px; color: #64748b; }
.ui-meter { position: relative; height: 14px; border-radius: 7px; margin-top: 14px; background: linear-gradient(90deg, #2e9e6b, #f2c94c, #d64545); }
.ui-pin { position: absolute; top: -5px; width: 6px; height: 24px; border-radius: 3px; background: #142b53; left: var(--to); margin-left: -3px; animation: ui-move 0.8s ease-out; }
.ui-thr { position: absolute; top: -2px; bottom: -2px; width: 2px; background: #475569; opacity: .7; }
.ui-scale { display: flex; justify-content: space-between; font-size: 12px; color: #64748b; margin-top: 4px; }
.ui-svg { width: 100%; height: auto; max-width: 640px; display: block; margin: 0 auto; }
.ui-svg text { font-family: inherit; fill: #475569; }
.ui-line { fill: none; stroke: #d64545; stroke-width: 2.5; stroke-linejoin: round; stroke-dasharray: 1; stroke-dashoffset: 0; animation: ui-draw 1s ease-out; }
.ui-pt:hover { r: 6; }
@keyframes ui-move { from { left: var(--from); } to { left: var(--to); } }
@keyframes ui-draw { from { stroke-dashoffset: 1; } to { stroke-dashoffset: 0; } }
@media (max-width: 600px) { .ui-big { font-size: 32px; } .ui-card { padding: 12px; } }
@media (prefers-reduced-motion: reduce) { .ui-pin, .ui-line { animation: none; } }
</style>
"""


def risk_card(prob, threshold=0.5, prev_prob=None, estimated=()):
    """Top par live card: P(Higher Risk), decision, pichle update se farak, meter."""
    decision = 1 if prob > threshold else 0
    start = prob if prev_prob is None else prev_prob
    delta = "" if prev_prob is None or abs(prob - prev_prob) < 1e-9 else \
        f'<span class="ui-sub">{"&#9650;" if prob > prev_prob else "&#9660;"} {abs(prob - prev_prob) * 100:.1f} pts pichhle update se</span>'
    est = f'<div class="ui-sub">Estimated (aapne nahi diya): {", ".join(estimated)}</div>' if estimated else ""
    return (f'<div class="ui-card"><div class="ui-top"><div class="ui-big">{prob * 100:.1f}%</div>'
            f'<div><div class="ui-label" style="color:{COLORS[decision]}">{CLASS_NAMES[decision]}</div>'
            f'<div class="ui-sub">{CLASS_NAMES[1]} probability (threshold {threshold * 100:.0f}%)</div>{delta}</div></div>'
            f'<div class="ui-meter"><div class="ui-thr" style="left:{threshold * 100:.1f}%"></div>'
            f'<div class="ui-pin" style="--from:{start * 100:.1f}%;--to:{prob * 100:.1f}%"></div></div>'
            f'<div class="ui-scale"><span>0%</span><span>50%</span><span>100%</span></div>{est}</div>')


def curve_chart(title, feature, xs, ps, current_x, current_p, population, is_category, threshold=0.5):
    """SVG risk curve. xs/ps = feature ki values aur model ka P(Higher Risk); population = dataset ki us feature ki values."""
    W, H, L, R, T, B = 380, 240, 44, 10, 14, 46
    pw, ph = W - L - R, H - T - B
    xs = np.asarray(xs, float); ps = np.asarray(ps, float)
    lo, hi = xs.min(), xs.max()
    sx = (lambda v: L + (np.where(np.asarray(xs) == v)[0][0] + 0.5) / len(xs) * pw) if is_category else \
         (lambda v: L + (float(v) - lo) / (hi - lo + 1e-9) * pw)
    sy = lambda p: T + (1 - float(p)) * ph
    out = [f'<svg class="ui-svg" viewBox="0 0 {W} {H}" role="img" aria-label="{title}">']
    for g in (0, .25, .5, .75, 1):                                                  
        out.append(f'<line x1="{L}" x2="{W - R}" y1="{sy(g):.1f}" y2="{sy(g):.1f}" stroke="#e2e8f0"/><text x="{L - 6}" y="{sy(g) + 4:.1f}" font-size="13" text-anchor="end">{g * 100:.0f}%</text>')
    pop = np.asarray(population, float)                                              
    if is_category:
        counts = np.array([(pop == v).sum() for v in xs]); edges = None
    else:
        counts, edges = np.histogram(pop, bins=12, range=(lo, hi))
    top = counts.max() or 1
    for i, c in enumerate(counts):
        if is_category:
            x0, bw = L + i / len(xs) * pw + 4, pw / len(xs) - 8
        else:
            x0, bw = L + (edges[i] - lo) / (hi - lo + 1e-9) * pw + 1, (edges[i + 1] - edges[i]) / (hi - lo + 1e-9) * pw - 2
        bh = c / top * ph * 0.35
        out.append(f'<rect x="{x0:.1f}" y="{T + ph - bh:.1f}" width="{max(bw, 1):.1f}" height="{bh:.1f}" fill="#cbd5e1" opacity=".6"><title>{int(c)} patients</title></rect>')
    ty = sy(threshold)                                                              
    out.append(f'<line x1="{L}" x2="{W - R}" y1="{ty:.1f}" y2="{ty:.1f}" stroke="#475569" stroke-dasharray="4 4"/><text x="{W - R}" y="{ty - 4:.1f}" font-size="12" text-anchor="end">threshold</text>')
    pts = " ".join(f"{sx(x):.1f},{sy(p):.1f}" for x, p in zip(xs, ps))
    out.append(f'<polyline class="ui-line" pathLength="1" points="{pts}"/>')
    for x, p in zip(xs, ps):                                                        
        out.append(f'<circle class="ui-pt" cx="{sx(x):.1f}" cy="{sy(p):.1f}" r="4" fill="#d64545"><title>{feature} = {x:g}: {CLASS_NAMES[1]} {p * 100:.1f}%</title></circle>')
    cx = sx(xs[np.abs(xs - current_x).argmin()]) if is_category else sx(min(max(current_x, lo), hi))  
    out.append(f'<line x1="{cx:.1f}" x2="{cx:.1f}" y1="{T}" y2="{T + ph}" stroke="#142b53" stroke-width="2"/>'
               f'<circle cx="{cx:.1f}" cy="{sy(current_p):.1f}" r="7" fill="#142b53" stroke="white" stroke-width="2"><title>Current: {feature} = {current_x:g}, {current_p * 100:.1f}%</title></circle>')
    step = max(1, len(xs) // 6)                                                     
    for i in range(0, len(xs), step):
        out.append(f'<text x="{sx(xs[i]):.1f}" y="{H - 26}" font-size="13" text-anchor="middle">{xs[i]:g}</text>')
    out.append(f'<text x="{L + pw / 2}" y="{H - 6}" font-size="13" text-anchor="middle">{feature} (grey = dataset ke patients)</text></svg>')
    return f'<div class="hc-chart"><h4>{title}</h4>{"".join(out)}</div>'


def percentile_text(value, population):
    """Slider value ka dataset me position."""
    pop = np.asarray(population, float)
    pct = (pop < value).mean() * 100
    return f"Dataset me {pct:.0f}% patients se zyada (median {np.median(pop):g})"

Overwriting components.py


In [24]:
import numpy as np, importlib, time
import components as ui; importlib.reload(ui)
from confidence import analyse, sweep, sweep_values, FEATURES
from cache import PredictionCache

df_all = pd.read_csv('heart.csv').drop_duplicates()
x = df_all[FEATURES].values[0].astype(float)
cache_t5 = PredictionCache(max_size=512, ttl=600)
age_i = FEATURES.index('age')

print('age  P(Higher Risk)  time')
last = None
for age in range(40, 71, 5):
    xx = x.copy(); xx[age_i] = age
    t = time.perf_counter()
    p = cache_t5.get_or_compute(('t5', tuple(xx)), lambda: analyse(model, xx))['probs'][1]
    print(f'{age}   {p:6.1%}        {(time.perf_counter() - t) * 1000:.1f} ms')

# Curve ka har point model se match karta hai?
vals = sweep_values(rules['age'], 25); curve = sweep(model, x, 'age', vals)
direct = [model.predict_proba(pd.DataFrame([np.where(np.arange(13) == age_i, v, x)], columns=FEATURES))[0, 1] for v in vals]
print('curve vs model.predict_proba, max difference =', float(np.abs(np.array(curve) - np.array(direct)).max()))

age  P(Higher Risk)  time
40    74.0%        5.8 ms
45    73.0%        2.7 ms
50    74.0%        3.1 ms
55    78.0%        3.1 ms
60    73.0%        3.2 ms
65    78.0%        3.0 ms
70    76.0%        3.0 ms
curve vs model.predict_proba, max difference = 0.0


In [25]:
from IPython.display import HTML
from confidence import CSS
p = analyse(model, x)['probs'][1]
p_prev = 0.35                       
vals_a = sweep_values(rules['age'], 25); ps_a = sweep(model, x, 'age', vals_a)
vals_c = sweep_values(rules['cp']);       ps_c = sweep(model, x, 'cp', vals_c)
demo_html = (CSS + ui.CSS + ui.risk_card(p, 0.5, p_prev, ['chol'])
             + ui.curve_chart('Risk curve: age badalne par risk', 'age', vals_a, ps_a, float(x[0]), p, df_all['age'].values, False)
             + ui.curve_chart('Risk curve: cp (category)', 'cp', vals_c, ps_c, float(x[2]), p, df_all['cp'].values, True))
print(ui.percentile_text(63, df_all['age'].values))
HTML(demo_html)

Dataset me 80% patients se zyada (median 55.5)


### Step 3: Alag screen widths par

In [26]:
import html as _h
from IPython.display import HTML
page = '<html><head><meta name="viewport" content="width=device-width,initial-scale=1"></head><body style="margin:0;background:#f3f6fb;font-family:sans-serif"><div style="padding:12px 10px">' + demo_html + '</div></body></html>'
frames = ''.join(f'<div style="margin:8px;display:inline-block;vertical-align:top"><b>{n} ({w}px)</b><br>'
                 f'<iframe srcdoc="{_h.escape(page)}" style="width:{w}px;height:{hh}px;border:1px solid #aaa;background:white"></iframe></div>'
                 for n, w, hh in [('Phone', 360, 700), ('Tablet', 768, 700)])
HTML(frames)

In [27]:
import os
open('responsive_test.html', 'w').write(page.replace('<div style="padding:12px 10px">', '<div style="max-width:850px;margin:auto;padding:12px 10px">'))
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as pw:
        b = pw.chromium.launch()
        for name, w in [('phone', 360), ('tablet', 768), ('laptop', 1280)]:
            pg = b.new_context(viewport={'width': w, 'height': 800}).new_page()
            pg.goto('file://' + os.path.abspath('responsive_test.html')); pg.wait_for_timeout(1200)
            r = pg.evaluate("() => { const d = document.documentElement; const t = [...document.querySelectorAll('svg text')].map(e => e.getBoundingClientRect().height);"
                            " return {scrollW: d.scrollWidth, clientW: d.clientWidth, minSvgTextPx: Math.round(Math.min(...t))} }")
            print(f'{name:6s} {w:5d}px  sideways scroll: {r["scrollW"] > r["clientW"]}  | smallest chart text: {r["minSvgTextPx"]}px')
        b.close()
except Exception as e:
    print('Playwright is machine par nahi hai (optional):', type(e).__name__, '- upar iframe preview se visually dekho.')

Playwright is machine par nahi hai (optional): ModuleNotFoundError - upar iframe preview se visually dekho.


## Task 6: Display SHAP values in the UI for interpretability by manually computing feature attributions, developing an interactive visualization that allows users to explore individual contributions, and ensuring transparency in model decisions.

In [28]:
%%writefile shap_values.py
"""
HeartCare - SHAP values (haath se) aur unka interactive display.

Shapley value: ek feature ka "fair share" jo prediction ko average patient se door le jata hai.
  phi_i = sum over coalitions S (i ke bina)  |S|! (M-|S|-1)! / M!  *  [ v(S + i) - v(S) ]
  v(S)  = S ke features patient se, baaki features background (reference) patients se; model ka average output.
Yahan koi `shap` library nahi. 13 features = 2^13 = 8192 coalitions, saare exactly evaluate hote hain.
Contributions "probability points" (percentage points) me hain: P(class 1) = P(Higher Risk).
"""
from math import factorial
import numpy as np
import pandas as pd

from confidence import FEATURES, CLASS_NAMES, COLORS

M = len(FEATURES)
MASKS = ((np.arange(2 ** M)[:, None] >> np.arange(M)) & 1).astype(bool)     
SIZES = MASKS.sum(axis=1)
WEIGHTS = np.array([factorial(s) * factorial(M - s - 1) / factorial(M) for s in range(M)])


def build_background(df, k=20, seed=0):
    """Reference patients: training data se k random rows."""
    return df[FEATURES].sample(k, random_state=seed).to_numpy(dtype=float)


def coalition_values(model, x, background):
    """v(S) har coalition ke liye (8192 values): model ka average P(class 1)."""
    k = len(background)
    X = np.where(MASKS[None, :, :], np.asarray(x, float)[None, None, :], background[:, None, :])   
    p = model.predict_proba(pd.DataFrame(X.reshape(-1, M), columns=FEATURES))[:, 1]
    return p.reshape(k, -1).mean(axis=0)


def shapley_exact(model, x, background):
    """Exact Shapley values. Return: phi (13,), base value, prediction f(x)."""
    v = coalition_values(model, x, background)
    phi = np.zeros(M)
    for i in range(M):
        without = np.where(~MASKS[:, i])[0]                     
        phi[i] = (WEIGHTS[SIZES[without]] * (v[without | (1 << i)] - v[without])).sum()
    return phi, float(v[0]), float(v[-1])                        


def shapley_sampling(model, x, background, n_perm=200, seed=0):
    """Doosra independent tareeka (verification ke liye): random orderings me feature ek-ek karke patient wali value par laate hain."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    phi = np.zeros(M)
    for _ in range(n_perm):
        order = rng.permutation(M)
        b = background[rng.integers(len(background))].copy()
        rows = [b.copy()]
        for j in order:
            b[j] = x[j]; rows.append(b.copy())
        p = model.predict_proba(pd.DataFrame(np.array(rows), columns=FEATURES))[:, 1]
        phi[order] += np.diff(p)
    return phi / n_perm


def explain(model, x, background):
    phi, base, fx = shapley_exact(model, x, background)
    return {"phi": phi, "base": base, "pred": fx, "x": np.asarray(x, float), "k": len(background)}


def fmt(v):
    return f"{v:g}"


def plain_summary(info, filled=()):
    """Top drivers ek line me, aam bhasha me."""
    order = np.argsort(-np.abs(info["phi"]))[:3]
    parts = []
    for i in order:
        f = FEATURES[i]; pts = info["phi"][i] * 100
        direction = CLASS_NAMES[1] if pts > 0 else CLASS_NAMES[0]
        est = " (estimated)" if f in filled else ""
        parts.append(f"{f} = {fmt(info['x'][i])}{est} {direction} ki taraf {abs(pts):.1f} pts")
    return "Sabse bade kaaran: " + "; ".join(parts) + "."


def feature_detail(info, feature, bg_mean, filled=()):
    """Ek feature ka detail: value, average patient se compare, contribution."""
    i = FEATURES.index(feature); pts = info["phi"][i] * 100
    side = f"{CLASS_NAMES[1]} ki taraf" if pts > 0 else f"{CLASS_NAMES[0]} ki taraf"
    est = " Ye value aapne nahi di, isliye typical value se bhari gayi hai." if feature in filled else ""
    return (f"<b>{feature} = {fmt(info['x'][i])}</b> (reference patients ka average {bg_mean[i]:.1f}). "
            f"Is feature ne prediction ko <b>{abs(pts):.1f} percentage points {side}</b> dhakela.{est}")


SHAP_CSS = """
<style>
.sv-row { display: flex; align-items: center; margin: 6px 0; padding: 2px 4px; border-radius: 6px; }
.sv-row.sel { background: #eef4ff; outline: 2px solid #244577; }
.sv-label { width: 128px; font-size: 14px; color: #334155; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.sv-track { position: relative; flex: 1; height: 22px; background: #f1f5f9; border-radius: 6px; }
.sv-mid { position: absolute; left: 50%; top: 0; bottom: 0; width: 2px; background: #64748b; }
.sv-bar { position: absolute; top: 3px; bottom: 3px; width: var(--to); animation: sv-grow 0.9s ease-out; }
.sv-bar.pos { left: 50%; border-radius: 0 5px 5px 0; }
.sv-bar.neg { right: 50%; border-radius: 5px 0 0 5px; }
.sv-val { width: 78px; text-align: right; font-size: 13px; font-weight: 600; color: #142b53; }
.sv-total { display: flex; justify-content: space-between; font-size: 14px; padding: 8px 4px 0 4px; color: #142b53; border-top: 1px solid #e2e8f0; margin-top: 8px; flex-wrap: wrap; gap: 6px; }
.sv-legend { font-size: 12px; color: #475569; margin-top: 6px; }
@keyframes sv-grow { from { width: var(--from); } to { width: var(--to); } }
@media (max-width: 600px) { .sv-label { width: 92px; font-size: 12px; } .sv-val { width: 62px; font-size: 12px; } }
@media (prefers-reduced-motion: reduce) { .sv-bar { animation: none; } }
</style>
"""


def shap_chart(info, top_n=8, selected=None, prev=None, filled=()):
    """Diverging bars: kaun sa feature prediction ko kis taraf aur kitna dhakel raha hai.
    prev = {feature: pichli phi} (animation ka start)."""
    phi = info["phi"]; order = np.argsort(-np.abs(phi)); top = order[:top_n]; rest = order[top_n:]
    scale = max(np.abs(phi[order[0]]), 1e-9)                                  
    rows = ""
    for i in top:
        f = FEATURES[i]; v = phi[i]; cls = "pos" if v > 0 else "neg"
        old = (prev or {}).get(f)
        start = abs(old) / scale * 50 if old is not None and np.sign(old) == np.sign(v) else 0.0
        color = COLORS[1] if v > 0 else COLORS[0]
        est = "*" if f in filled else ""
        sel = " sel" if f == selected else ""
        rows += (f'<div class="sv-row{sel}" title="{f}: {v * 100:+.2f} percentage points"><div class="sv-label">{f} = {fmt(info["x"][i])}{est}</div>'
                 f'<div class="sv-track"><div class="sv-mid"></div><div class="sv-bar {cls}" style="--from:{start:.1f}%;--to:{abs(v) / scale * 50:.1f}%;background:{color}"></div></div>'
                 f'<div class="sv-val">{v * 100:+.1f} pts</div></div>')
    if len(rest):
        other = phi[rest].sum() * 100
        rows += (f'<div class="sv-row"><div class="sv-label">baaki {len(rest)} features</div><div class="sv-track"><div class="sv-mid"></div></div>'
                 f'<div class="sv-val">{other:+.1f} pts</div></div>')
    total = (f'<div class="sv-total"><span>Average patient: <b>{info["base"] * 100:.1f}%</b></span>'
             f'<span>+ contributions: <b>{phi.sum() * 100:+.1f} pts</b></span><span>= Is patient: <b>{info["pred"] * 100:.1f}%</b></span></div>')
    legend = (f'<div class="sv-legend"><span style="color:{COLORS[1]}">&#9632;</span> {CLASS_NAMES[1]} ki taraf &nbsp; '
              f'<span style="color:{COLORS[0]}">&#9632;</span> {CLASS_NAMES[0]} ki taraf &nbsp; (* = estimated value)</div>')
    return f'<div class="hc-chart"><h4>Kis feature ne prediction ko kitna dhakela (SHAP values)</h4>{rows}{total}{legend}</div>'

Overwriting shap_values.py


In [29]:
import importlib, time, numpy as np
import shap_values as sv; importlib.reload(sv)

bg = sv.build_background(df_all, k=20)                      
errs, secs = [], []
for x_i in df_all[FEATURES].values[:30].astype(float):
    t = time.perf_counter(); info = sv.explain(model, x_i, bg); secs.append(time.perf_counter() - t)
    errs.append(abs(info['base'] + info['phi'].sum() - info['pred']))
print('30 patients | base + sum(SHAP) = prediction, max error:', max(errs))
print(f'time per patient: {np.median(secs) * 1000:.0f} ms (8192 coalitions)')
info = sv.explain(model, x, bg)
print(f"average patient {info['base']:.1%} + contributions {info['phi'].sum():+.1%} = {info['pred']:.1%}")

30 patients | base + sum(SHAP) = prediction, max error: 2.220446049250313e-16
time per patient: 652 ms (8192 coalitions)
average patient 47.1% + contributions +30.8% = 78.0%


In [30]:
rows = []
for j in range(5):
    xj = df_all[FEATURES].values[j * 40].astype(float)
    ex = sv.explain(model, xj, bg)['phi']
    sm = sv.shapley_sampling(model, xj, bg, n_perm=200)
    rows.append({'patient': j * 40, 'correlation': round(float(np.corrcoef(ex, sm)[0, 1]), 3),
                 'max diff (pts)': round(float(np.abs(ex - sm).max() * 100), 2)})
pd.DataFrame(rows)

,patient,correlation,max diff (pts)
0,0,0.997,1.05
1,40,0.996,1.28
2,80,0.997,0.87
3,120,0.995,1.77
4,160,0.995,1.21


In [31]:
xj = df_all[FEATURES].values[0].astype(float)
phis = [sv.explain(model, xj, sv.build_background(df_all, k=20, seed=s))['phi'] for s in range(5)]
corr = np.corrcoef(phis)[np.triu_indices(5, 1)]
top3 = [set(np.argsort(-np.abs(p))[:3]) for p in phis]
overlap = [len(top3[a] & top3[b]) for a in range(5) for b in range(a + 1, 5)]
print(f'seed-to-seed correlation: min {corr.min():.2f}, mean {corr.mean():.2f}')
print(f'top-3 features common: min {min(overlap)} of 3, mean {np.mean(overlap):.1f} of 3')
print('top-3 (seed 0):', [FEATURES[i] for i in np.argsort(-np.abs(phis[0]))[:3]])

seed-to-seed correlation: min 0.93, mean 0.96
top-3 features common: min 2 of 3, mean 2.1 of 3
top-3 (seed 0): ['ca', 'cp', 'thal']


In [32]:
sample = df_all[FEATURES].astype(float).sample(60, random_state=1)
S = np.array([sv.explain(model, r.values, bg)['phi'] for _, r in sample.iterrows()]) * 100
tab = []
for f, cut in [('ca', 0.5), ('exang', 0.5), ('oldpeak', 1.0), ('thalach', 150), ('age', 55), ('sex', 0.5)]:
    i = FEATURES.index(f); hi = sample[f].values > cut
    tab.append({'feature': f, 'cut': cut, 'avg SHAP when value <= cut': round(float(S[~hi, i].mean()), 1),
                'avg SHAP when value > cut': round(float(S[hi, i].mean()), 1)})
print('Positive = Higher Risk (class 1) ki taraf')
pd.DataFrame(tab).set_index('feature')

Positive = Higher Risk (class 1) ki taraf


,cut,avg SHAP when value <= cut,avg SHAP when value > cut
feature,,,
ca,0.5,9.8,-6.8
exang,0.5,2.4,-5.2
oldpeak,1.0,4.9,-5.7
thalach,150.0,-5.4,3.9
age,55.0,1.9,-2.4
sex,0.5,7.3,-1.1


In [33]:
from IPython.display import HTML
from confidence import CSS
import shap_values as sv
info = sv.explain(model, x, bg)
print(sv.plain_summary(info, filled=['chol']))
# 'ca' bar chuna gaya (inspect): app me ye selectbox se hota hai
HTML(CSS + sv.SHAP_CSS + sv.shap_chart(info, top_n=8, selected='ca', filled=['chol'])
     + '<p style="font-family:sans-serif;max-width:600px">' + sv.feature_detail(info, 'ca', bg.mean(axis=0), ['chol']) + '</p>')

Sabse bade kaaran: ca = 0 Higher Risk ki taraf 12.8 pts; cp = 3 Higher Risk ki taraf 11.1 pts; thal = 1 Higher Risk ki taraf 7.4 pts.


In [34]:
%%writefile app_v2_single.py
import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

from input import FEATURES, load_rules, validate_input, predict_batch      
from cache import PredictionCache                                        
from confidence import (CSS, CLASS_NAMES, COLORS, analyse, decide, sweep, 
                        sweep_values, bar_chart, column_chart)
import components as ui                                                    
import shap_values as sv                                                  
from feedback import add_feedback                                         

st.set_page_config(
    page_title="HeartCare",
    layout="centered"
)

MODEL_FILE = "heart_model_v20261013_160903.joblib"
DATA_FILE = "heart.csv"
FEEDBACK_FILE = "feedback.csv"
MAX_BATCH_ROWS = 100000
MAX_UNKNOWN = 3


@st.cache_resource
def load_everything():
    model = joblib.load(MODEL_FILE)
    rules = load_rules(DATA_FILE)
    cache = PredictionCache(max_size=512, ttl=600)
    df = pd.read_csv(DATA_FILE).drop_duplicates()
    background = sv.build_background(df, k=20)
    return model, rules, cache, MODEL_FILE, df, background


try:
    model, rules, cache, model_version, data_df, background = load_everything()
except FileNotFoundError as e:
    st.error(f"File nahi mili: {e.filename}. app.py ke same folder me rakho.")
    st.stop()

SECTIONS = [
    ("Basic", ["age", "sex"]),
    ("Chest & heart rate", ["cp", "exang", "thalach", "oldpeak", "slope"]),
    ("Tests (blood / ECG)", ["trestbps", "chol", "fbs", "restecg"]),
    ("Scan results", ["ca", "thal"]),
]
LABELS = {"age": "Age", "sex": "Sex", "cp": "Chest Pain Type", "trestbps": "Resting Blood Pressure",
          "chol": "Cholesterol", "fbs": "Fasting Blood Sugar", "restecg": "Resting ECG",
          "thalach": "Maximum Heart Rate", "exang": "Exercise Induced Angina", "oldpeak": "Oldpeak",
          "slope": "Slope", "ca": "Major Vessels", "thal": "Thal"}
HELP = {"age": "Years me.", "sex": "Dataset ka code: 0 ya 1.", "cp": "Chest pain type ka code (0 se 3).",
        "trestbps": "Resting blood pressure, mm Hg.", "chol": "Serum cholesterol, mg/dl.",
        "fbs": "Fasting blood sugar ka code (0 ya 1).", "restecg": "Resting ECG result ka code (0 se 2).",
        "thalach": "Exercise me maximum heart rate, beats per minute.", "exang": "Exercise se chest pain hua? 0 ya 1.",
        "oldpeak": "Exercise ke baad ST depression (number).", "slope": "ST segment slope ka code (0 se 2).",
        "ca": "Major vessels ki ginti (dataset code 0 se 4).", "thal": "Thal test ka code (0 se 3)."}
WHOLE = {"age", "trestbps", "chol", "thalach"}    


def previous(key, current):
    """Chart ki pichli values (animation start). Session me yaad rakhte hain."""
    old = st.session_state.get("prev", {}).get(key)
    st.session_state.setdefault("prev", {})[key] = list(current)
    return old


def default_for(f):
    r = rules[f]
    if r["kind"] == "number":
        return int(round(r["fill"])) if f in WHOLE else float(r["fill"])
    return int(r["fill"])


def reset_inputs():
    for f in FEATURES:
        st.session_state["in_" + f] = default_for(f)
    st.session_state["unknown"] = []
    st.session_state.pop("check", None)


def widget(f):
    """Number -> slider, 2 se 4 options -> radio, zyada options -> selectbox. Default median/mode."""
    r = rules[f]; key = "in_" + f
    if r["kind"] == "number":
        if f in WHOLE:
            st.session_state.setdefault(key, default_for(f))
            v = st.slider(LABELS[f], int(r["min"]), int(r["max"]), key=key, help=HELP[f])
        else:
            st.session_state.setdefault(key, default_for(f))
            v = st.slider(LABELS[f], float(r["min"]), float(r["max"]), step=0.1, key=key, help=HELP[f])
        st.caption(ui.percentile_text(v, data_df[f].values))
        return v
    options = [int(a) for a in r["allowed"]]
    st.session_state.setdefault(key, default_for(f))
    if len(options) <= 4:
        return st.radio(LABELS[f], options, key=key, horizontal=True, help=HELP[f])
    return st.selectbox(LABELS[f], options, key=key, help=HELP[f])


st.markdown("""
<style>
.stApp {
    background: #f3f6fb;
}

header, footer {
    visibility: hidden;
}

.main .block-container {
    max-width: 850px;
    padding: 25px 20px;
}

.header {
    background: #142b53;
    color: white;
    padding: 25px;
    border-radius: 14px;
    text-align: center;
    margin-bottom: 25px;
}

.section {
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 18px;
    border: 1px solid #e2e8f0;
}

.section h3 {
    color: #142b53;
}

.stButton button {
    background: #142b53;
    color: white;
    width: 100%;
    height: 45px;
    border-radius: 8px;
}

.stButton button:hover {
    background: #244577;
    color: white;
}

@media(max-width: 600px) {
    .main .block-container {
        padding: 12px 10px;
    }

    .header {
        padding: 18px;
    }

    .header h1 {
        font-size: 24px;
    }

    .section {
        padding: 14px;
    }
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="header">
    <h1>HeartCare</h1>
    <p>Heart Disease Risk Prediction System</p>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="section">
    <h3>Patient Information</h3>
    <p>Enter patient medical details below.</p>
</div>
""", unsafe_allow_html=True)


st.caption("Sliders/options se values chuno. Koi value pata na ho to neeche 'Values I don't know' me chuno (max 3), wo median/mode se bhar di jayegi.")

top = st.empty()
st.markdown(CSS + ui.CSS + sv.SHAP_CSS, unsafe_allow_html=True)

values = {}
for title, feats in SECTIONS:
    st.markdown(f'<div class="section"><h3>{title}</h3></div>', unsafe_allow_html=True)
    for f in feats:
        values[f] = widget(f)

unknown = st.multiselect("Values I don't know (max 3)", [LABELS[f] for f in FEATURES], key="unknown")
too_many = len(unknown) > MAX_UNKNOWN
if too_many:
    st.error(f"Maximum {MAX_UNKNOWN} values 'unknown' ho sakti hain. Ek-do hata do.")

live = st.checkbox("Live update (slider hilate hi prediction update ho)", value=True)
clicked = st.button("Predict Heart Disease Risk")
st.button("Reset to typical values", on_click=reset_inputs)

if (clicked or live) and not too_many:

    record = {f: (None if LABELS[f] in unknown else values[f]) for f in FEATURES}
    check = validate_input(record, rules)

    for message in check["errors"]:
        st.error(message)

    if check["ok"]:
        st.session_state["check"] = check     
    else:
        st.session_state.pop("check", None)

if too_many:
    st.session_state.pop("check", None)     

check = st.session_state.get("check")

if check:

    st.markdown("""
<div class="section">
    <h3>Prediction & Confidence</h3>
</div>
""", unsafe_allow_html=True)

    threshold = st.slider("Decision threshold (%): Higher Risk tab maana jaye jab probability isse zyada ho", 1, 99, 50, key="threshold") / 100
    explore = st.selectbox("Risk curve: ek feature badalke dekho", FEATURES, index=0, key="explore")
    top_n = st.slider("SHAP: kitne top features dikhayein", 3, 13, 8, key="top_n")

    start = time.perf_counter()
    key = (model_version, tuple(float(v) for v in check["x"]))
    info = cache.get_or_compute(key, lambda: analyse(model, check["x"]))
    elapsed = (time.perf_counter() - start) * 1000

    for message in check["warnings"]:
        st.warning(message)

    filled = list(check["filled"])
    probs = info["probs"]
    prediction = decide(probs, threshold)

    p1 = float(probs[1])
    prev_card = st.session_state.get("last_p1")
    st.session_state["last_p1"] = p1
    top.markdown(ui.risk_card(p1, threshold, prev_card, filled), unsafe_allow_html=True)

    st.write("Model Confidence:", round(max(probs) * 100, 2), "%")
    if threshold != 0.5:
        st.caption(f"Custom threshold {threshold * 100:.0f}%: {CLASS_NAMES[1]} probability {probs[1] * 100:.1f}% hai, "
                   f"isliye prediction {CLASS_NAMES[prediction]} dikh raha hai (model ka default threshold 50% hai).")
    st.caption(f"{info['votes'][1]} of {info['n_trees']} trees ne {CLASS_NAMES[1]} chuna | trees ke beech spread (std): {info['std']:.2f}")

    labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
    st.markdown(bar_chart("Class probabilities (trees ka average)", labels, probs,
                          prev=previous("probs", probs), colors=[COLORS[0], COLORS[1]], threshold=threshold,
                          note=f"Kali line = threshold ({threshold * 100:.0f}%)"), unsafe_allow_html=True)

    votes = info["votes"] / info["n_trees"]
    st.markdown(bar_chart("Tree votes (kitni trees ne kaunsi class chuni)", labels, votes,
                          prev=previous("votes", votes), colors=[COLORS[0], COLORS[1]]), unsafe_allow_html=True)

    hist = info["hist"]
    bins = [f"{i * 10}-{i * 10 + 10}" for i in range(len(hist))]
    st.markdown(column_chart(f"Har tree ka {CLASS_NAMES[1]} probability (%) - kitni trees kis range me",
                             bins, hist / hist.max(), prev=previous("hist", hist / hist.max()),
                             color="#244577", counts=[int(h) for h in hist],
                             note="Bar ke upar tree ginti."), unsafe_allow_html=True)

    x = check["x"]; r = rules[explore]; fi = FEATURES.index(explore)
    xs = sweep_values(r, 25) if r["kind"] == "number" else sweep_values(r)
    ps = sweep(model, x, explore, xs)
    st.markdown(ui.curve_chart(f"Risk curve: {explore} badalne par risk", explore, xs, ps, float(x[fi]), p1,
                               data_df[explore].values, r["kind"] == "category", threshold), unsafe_allow_html=True)

    st.markdown("""
<div class="section">
    <h3>Why this prediction? (SHAP values)</h3>
    <p>Har feature ne risk ko average patient se kitna upar/neeche dhakela. Ye Shapley values
    haath se nikali gayi hain (13 features ke saare combinations).</p>
</div>
""", unsafe_allow_html=True)

    skey = (model_version, "shap", tuple(float(v) for v in x), len(background))
    sinfo = cache.get_or_compute(skey, lambda: sv.explain(model, x, background))
    inspect = st.selectbox("Ek feature ko detail me dekho", ["(none)"] + FEATURES, key="inspect")
    sel = None if inspect == "(none)" else inspect

    old_phi = previous("shap", sinfo["phi"])
    shap_prev = dict(zip(FEATURES, old_phi)) if old_phi else None

    st.caption(sv.plain_summary(sinfo, filled))
    st.markdown(sv.shap_chart(sinfo, top_n, sel, prev=shap_prev, filled=filled), unsafe_allow_html=True)
    if sel:
        st.markdown(sv.feature_detail(sinfo, sel, background.mean(axis=0), filled), unsafe_allow_html=True)
    st.caption("Transparency: ye model ke patterns hain, medical karan (cause) nahi. Reference = dataset ke 20 patients ka average. "
               "Base + saare contributions = final probability (exactly).")

    if not live:
        st.caption("Ye result pichhle Predict click ke inputs ka hai. Naye inputs ke liye dobara Predict dabao ya Live update on karo.")

    stats = cache.stats()
    st.caption(f"Is request me {elapsed:.2f} ms | Cache: {stats['hits']} hits, {stats['misses']} misses, "
               f"{stats['waits']} coalesced, hit rate {stats['hit_rate_%']}% (size {stats['size']})")

st.markdown("""
<div class="section">
    <h3>Batch Prediction (CSV)</h3>
    <p>Bahut patients ek saath: CSV me columns age, sex, cp, trestbps, chol, fbs,
    restecg, thalach, exang, oldpeak, slope, ca, thal hone chahiye.</p>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Upload CSV", type=["csv"])

data = None

if uploaded is not None:

    try:
        data = pd.read_csv(uploaded)
    except Exception:
        st.error("CSV padh nahi paye. File sahi CSV format me honi chahiye.")

if data is not None:

    clean_names = [str(c).strip().lower().replace(" ", "_") for c in data.columns]
    missing_cols = [f for f in FEATURES if f not in clean_names]

    if len(data) == 0:
        st.error("CSV khali hai.")
    elif len(data) > MAX_BATCH_ROWS:
        st.error(f"Max {MAX_BATCH_ROWS} rows allowed hain.")
    else:
        if missing_cols:
            st.warning(f"Ye columns CSV me nahi mile (missing maane gaye): {missing_cols}")

        start = time.perf_counter()
        results = predict_batch(model, rules, data)
        elapsed = (time.perf_counter() - start) * 1000

        rejected = int((results["status"] == "rejected").sum())
        st.write(f"Rows: {len(data)} | Predicted: {len(data) - rejected} | Rejected: {rejected}")
        st.caption(f"{elapsed:.0f} ms ({elapsed / len(data):.3f} ms per row)")

        output = pd.concat([data, results], axis=1)
        st.dataframe(output.head(100))

        st.download_button("Download results", output.to_csv(index=False), "heart_predictions.csv", "text/csv")


st.markdown("""
<div class="section">
    <h3>Feedback</h3>
    <p>Ye app kaisa laga? Isse hum interface sudharte hain.</p>
</div>
""", unsafe_allow_html=True)

fb_device = st.radio("Aap kis device par ho?", ["Phone", "Tablet", "Laptop"], horizontal=True, key="fb_device")
fb_task = st.radio("Kya aap apna kaam (risk dekhna) bina madad ke kar paye?", ["Yes", "Partly", "No"], horizontal=True, key="fb_task")
fb_ease = st.slider("Use karna kitna aasaan laga? (1 = mushkil, 5 = bahut aasaan)", 1, 5, 4, key="fb_ease")
fb_text = st.text_area("Kuch atka ya confusing laga? (optional)", key="fb_text", max_chars=500)
if st.button("Send feedback"):
    try:
        add_feedback(FEEDBACK_FILE, fb_device, fb_task, fb_ease, fb_text)
        st.success("Shukriya! Feedback save ho gaya.")
    except Exception as e:
        st.error(f"Feedback save nahi hua: {e}")

st.markdown("""
<div style="text-align:center;color:#64748b;padding:20px;">
    HeartCare | Educational Purpose Only
    <br>
    Not a substitute for professional medical diagnosis.
</div>
""", unsafe_allow_html=True)

Overwriting app_v2_single.py


In [35]:
import sys, types, time, warnings; warnings.filterwarnings('ignore')

def ui_probe(app_file, session=None, typed=None, clicks=(), resources=None):
    """Return: widgets ki ginti + kya result dikha. session = v2 ke widget keys; typed = v1 ke text boxes."""
    session, typed, resources = session if session is not None else {}, typed or {}, resources if resources is not None else {}
    count, log = {}, []
    def tick(name): count[name] = count.get(name, 0) + 1
    st = types.ModuleType('streamlit')
    def cache_resource(f):
        def w(*a, **k):
            if f.__name__ not in resources: resources[f.__name__] = f(*a, **k)
            return resources[f.__name__]
        return w
    st.cache_resource = cache_resource; st.set_page_config = lambda **k: None
    st.markdown = lambda t, **k: log.append(('html', t)); st.caption = lambda m: log.append(('caption', m))
    st.write = lambda *a: log.append(('write', ' '.join(map(str, a))))
    st.error = lambda m: log.append(('error', m)); st.warning = lambda m: log.append(('warning', m)); st.success = lambda m: log.append(('success', m))
    st.dataframe = lambda d: None; st.download_button = lambda *a, **k: None; st.file_uploader = lambda *a, **k: None
    st.stop = lambda: (_ for _ in ()).throw(SystemExit)
    st.empty = lambda: types.SimpleNamespace(markdown=lambda t, **k: log.append(('html', t)))
    st.session_state = session
    def text_input(label, value=''): tick('text_input'); return typed.get(label.split(' (')[0], value)
    def selectbox(label, options, index=0, key=None, format_func=str, help=None):
        tick('selectbox'); return session.setdefault(key, options[index]) if key else options[index]
    def slider(label, lo, hi, v=None, step=None, key=None, help=None):
        tick('slider'); return session.setdefault(key, v) if key else v
    def radio(label, options, key=None, horizontal=False, help=None): tick('radio'); return session.setdefault(key, options[0])
    def multiselect(label, options, key=None): tick('multiselect'); return session.setdefault(key, [])
    def checkbox(label, value=False, key=None): tick('checkbox'); return session.setdefault(key or label, value)
    def button(label, on_click=None):
        tick('button')
        if label in clicks:
            if on_click: on_click()
            return True
        return False
    def text_area(label, key=None, max_chars=None): tick('text_area'); return session.setdefault(key, '')
    st.text_input, st.selectbox, st.slider, st.radio, st.multiselect, st.checkbox, st.button, st.text_area = \
        text_input, selectbox, slider, radio, multiselect, checkbox, button, text_area
    sys.modules['streamlit'] = st
    for m in ['input', 'cache', 'confidence', 'components', 'shap_values', 'feedback']: sys.modules.pop(m, None)
    t0 = time.perf_counter()
    try: exec(compile(open(app_file).read(), app_file, 'exec'), {'__name__': '__main__'})
    except SystemExit: pass
    ms = (time.perf_counter() - t0) * 1000
    shown = any(k == 'write' and 'Model Confidence' in v for k, v in log)
    return {'widgets': count, 'result_shown': shown, 'errors': [v for k, v in log if k == 'error'], 'ms': ms}

In [36]:
import pandas as pd
rows = []
res1, res2 = {}, {}
for name, f in [('v1', 'app_v1.py'), ('v2', 'app_v2_single.py')]:
    res = res1 if name == 'v1' else res2
    r0 = ui_probe(f, resources=res)                                                 
    if name == 'v1':
        r1 = ui_probe(f, typed={'Age': '63'}, resources=res)                          
        r2 = ui_probe(f, typed={'Age': 'abc'}, clicks=('Predict Heart Disease Risk',), resources=res)   
        bad = len(r2['errors']) > 0
    else:
        r1 = ui_probe(f, session={'in_age': 63}, resources=res)
        bad = False                                                                  
    w = r0['widgets']
    rows.append({'version': name,
                 'widgets': sum(w.values()),
                 'free-typing boxes': w.get('text_input', 0),
                 'result on open (no click)': r0['result_shown'],
                 'result after changing Age (no click)': r1['result_shown'],
                 'bad typed value can reach model': 'blocked by message' if bad else ('never possible' if name == 'v2' else '-')})
pd.DataFrame(rows).set_index('version').T

version,v1,v2
widgets,15,26
free-typing boxes,5,0
result on open (no click),False,True
result after changing Age (no click),False,True
bad typed value can reach model,blocked by message,never possible


In [37]:
import numpy as np
res = {}
ui_probe('app_v2_single.py', resources=res)                                  
times_new, times_same = [], []
for age in [40, 45, 50, 55, 60, 65, 70]:
    times_new.append(ui_probe('app_v2_single.py', session={'in_age': age}, resources=res)['ms'])      
for _ in range(7):
    times_same.append(ui_probe('app_v2_single.py', session={'in_age': 70}, resources=res)['ms'])      
print(f'naya input par poora script (Streamlit ke bina): median {np.median(times_new):.0f} ms | wahi input: median {np.median(times_same):.0f} ms')

naya input par poora script (Streamlit ke bina): median 787 ms | wahi input: median 27 ms


In [ ]:
!streamlit run app_v2_single.py

## Task 7: Implement error handling for invalid inputs by designing a robust validation framework that detects and corrects potential errors, guiding users with contextual error messages and intelligent input suggestions.

In [ ]:
%%writefile validation.py
"""
HeartCare - robust input validation framework (detect -> correct -> explain -> suggest).

Layers:
  1. parse_quick(text)         : "age 63, chol 5.2 mmol, bp=130" jaisi free text ko record me todta hai (naam ke typo pakadta hai)
  2. diagnose(record, rules)   : har field ke liye problem dhundta hai. Teen tarah ke result:
        fixed   - safe auto-correction (jaise "1,5" -> 1.5, "63 yrs" -> 63, "5.2 mmol/L" -> 201 mg/dl)
        error   - prediction roki jaati hai; message + (agar ho sake to) suggested value
        warning - prediction chalti hai par user ko batate hain
  3. check_record(record, rules): diagnose + input.validate_input (purana validator) = final {ok, x, errors, warnings, filled, issues}
Rule: jo cheez galat ho use chupchap kabhi "theek" nahi karte; jo 100% safe hai wahi auto-fix hota hai, baaki par user ki haan chahiye.
"""
import difflib
import math
import re

import numpy as np

from input import FEATURES, MAX_MISSING, validate_input

ALIASES = {"bp": "trestbps", "blood_pressure": "trestbps", "resting_bp": "trestbps", "cholesterol": "chol",
           "heart_rate": "thalach", "max_hr": "thalach", "maxhr": "thalach", "st_depression": "oldpeak",
           "vessels": "ca", "chest_pain": "cp", "angina": "exang", "sugar": "fbs", "ecg": "restecg", "gender": "sex"}
MMOL_TO_MGDL = 38.67                     
UNIT_WORDS = {"yrs", "yr", "years", "year", "y", "mmhg", "mg/dl", "mgdl", "bpm", "mm", "hg", "%"}
EXAMPLE = {"age": "55", "trestbps": "130", "chol": "240", "thalach": "150", "oldpeak": "1.0"}


def issue(field, level, message, fix=None, fix_label=None, value=None):
    """fix = suggested value (ya naam-typo me suggested field ka naam); value = user ne jo likha tha (naam-typo ke liye)."""
    return {"field": field, "level": level, "message": message, "fix": fix, "fix_label": fix_label, "value": value}


def _norm(name):
    return re.sub(r"[\s\-]+", "_", str(name).strip().lower())


def match_field(name):
    """Field ka naam pehchano. Return (field ya None, suggestions list)."""
    n = _norm(name)
    if n in FEATURES:
        return n, []
    if n in ALIASES:
        return ALIASES[n], []
    close = difflib.get_close_matches(n, FEATURES + list(ALIASES), n=3, cutoff=0.6)
    close = [ALIASES.get(c, c) for c in close]
    close = list(dict.fromkeys(close))
    return None, close


def parse_quick(text):
    """Free text -> (record, issues). Separator: ; ya naya line ya ', ' ke baad naya naam (1,5 jaisa comma-decimal bacha rehta hai)."""
    record, issues = {}, []
    parts = re.split(r"[;\n]|,\s*(?=[A-Za-z])", str(text or ""))
    for part in parts:
        part = part.strip()
        if not part:
            continue
        m = re.match(r"^([A-Za-z_][A-Za-z_ \-]*?)\s*(?:[=:]\s*|\s+(?=[-+\d.,])|$)(.*)$", part)
        if m and match_field(m.group(1))[0] is None and " " in m.group(1).strip():       
            first, rest = part.split(None, 1)
            m = re.match(r"^(.*)$", first), None
            m = type("M", (), {"group": lambda self, i, a=first, b=rest.lstrip("=: "): a if i == 1 else b})()
        if not m:
            issues.append(issue(None, "error", f"'{part}' samajh nahi aaya. Aise likho: age 63, chol 240"))
            continue
        field, close = match_field(m.group(1))
        if field is None:
            hint = f" Kya aap '{close[0]}' kehna chahte the?" if close else f" Fields: {', '.join(FEATURES)}"
            issues.append(issue(None, "error", f"'{m.group(1).strip()}' naam pehchana nahi gaya.{hint}",
                                fix=close[0] if close else None, fix_label=f"'{close[0]}' maano" if close else None,
                                value=m.group(2).strip()))
            continue
        if _norm(m.group(1)) != field:
            issues.append(issue(field, "fixed", f"'{m.group(1).strip()}' ko '{field}' samjha."))
        if field in record:
            issues.append(issue(field, "warning", f"{field} do baar likha hai; aakhri value li gayi."))
        record[field] = m.group(2).strip()
    return record, issues


_NUM = re.compile(r"^([-+]?\d*[.,]?\d+)\s*([A-Za-z/%]*[A-Za-z%])?$")


def _to_number(field, raw, rules):
    """String -> (number ya None, issues). Comma decimal, unit suffix, mmol/L conversion yahan hota hai."""
    out = []
    s = str(raw).strip()
    m = _NUM.match(s.replace(" ", "")) if re.match(r"^[-+]?\.?\d", s) else None
    if m is None:
        r = rules[field]
        ex = EXAMPLE.get(field, "")
        rng = f"{r['min']:g} se {r['max']:g}" if r["kind"] == "number" else "allowed: " + "/".join(str(int(a)) for a in r["allowed"])
        out.append(issue(field, "error", f"{field}: '{s}' number nahi hai. Sirf number likho ({rng})" + (f", jaise {ex}." if ex else "."),
                         fix=float(r["fill"]), fix_label=f"typical value {r['fill']:g} lo"))
        return None, out
    num_txt, unit = m.group(1), (m.group(2) or "").lower()
    if "," in num_txt:
        out.append(issue(field, "fixed", f"{field}: '{num_txt}' ko {num_txt.replace(',', '.')} samjha (comma = decimal)."))
        num_txt = num_txt.replace(",", ".")
    value = float(num_txt)
    if unit:
        if unit in ("mmol/l", "mmol", "mmoll") and field == "chol":
            out.append(issue(field, "fixed", f"chol: {value:g} mmol/L ko {value * MMOL_TO_MGDL:.0f} mg/dl me badla (x{MMOL_TO_MGDL})."))
            value = value * MMOL_TO_MGDL
        elif unit in UNIT_WORDS:
            out.append(issue(field, "fixed", f"{field}: unit '{unit}' hata diya, value {value:g} li."))
        else:
            out.append(issue(field, "error", f"{field}: unit '{unit}' pehchana nahi gaya. Sirf number likho."))
            return None, out
    return value, out


def _suggest_number(field, v, r):
    """Hard limit ke bahar number ke liye sabse samajhdar suggestion."""
    if field == "chol" and 2 <= v < 40:                                 
        fixed = v * MMOL_TO_MGDL
        if r["hard_lo"] <= fixed <= r["hard_hi"]:
            return issue(field, "error", f"chol: {v:g} bahut kam hai (mg/dl me). Kya ye mmol/L me hai? To {fixed:.0f} mg/dl banta hai.",
                         fix=round(fixed), fix_label=f"{fixed:.0f} mg/dl lo")
    if field == "age" and v > r["hard_hi"] and v / 12 >= r["hard_lo"] and v / 12 <= r["hard_hi"]:
        return issue(field, "error", f"age: {v:g} saal impossible hai. Kya ye mahine hain? To {v / 12:.0f} saal banta hai.",
                     fix=round(v / 12), fix_label=f"{v / 12:.0f} saal lo")
    if v < 0 and r["hard_lo"] <= abs(v) <= r["hard_hi"]:
        return issue(field, "error", f"{field}: {v:g} negative nahi ho sakta. Kya {abs(v):g} matlab tha?", fix=abs(v), fix_label=f"{abs(v):g} lo")
    near = min(max(v, r["min"]), r["max"])                 
    return issue(field, "error", f"{field}: {v:g} impossible hai (limit {r['hard_lo']:g} se {r['hard_hi']:g}). Aam range ({r['min']:g} se {r['max']:g}) me sabse paas ki value {near:g} hai.",
                 fix=round(near, 2), fix_label=f"{near:g} lo")


def diagnose(record, rules, model=None, partial=False):
    """Return {'clean': record (auto-fix ke baad), 'issues': [...], 'blocked': bool}.
    partial=True: sirf diye gaye fields jaanchte hain (quick entry), missing-count ka rule nahi lagta."""
    issues, clean, missing = [], {}, []
    rec = dict(record) if isinstance(record, dict) else {}
    for f in FEATURES:
        raw = rec.get(f)
        r = rules[f]
        if raw is None or (isinstance(raw, str) and raw.strip() == "") or (isinstance(raw, float) and math.isnan(raw)):
            clean[f] = None; missing.append(f); continue
        if isinstance(raw, bool):
            raw = int(raw)
        if isinstance(raw, str):
            low = raw.strip().lower()
            if low in ("true", "false"):
                raw = 1 if low == "true" else 0
            else:
                raw, extra = _to_number(f, raw, rules); issues += extra
                if raw is None:
                    clean[f] = None; continue
        try:
            v = float(raw)
        except (TypeError, ValueError):
            issues.append(issue(f, "error", f"{f}: '{raw}' number nahi hai.", fix=float(r["fill"]), fix_label=f"typical value {r['fill']:g} lo"))
            clean[f] = None; continue
        if not math.isfinite(v):
            issues.append(issue(f, "error", f"{f}: infinity/NaN allowed nahi.", fix=float(r["fill"]), fix_label=f"typical value {r['fill']:g} lo"))
            clean[f] = None; continue
        if r["kind"] == "category":
            allowed = [float(a) for a in r["allowed"]]
            if v not in allowed:
                order = sorted(allowed, key=lambda a: (abs(a - v), a))
                alt = " ya ".join(f"{int(a)}" for a in order[:2]) if abs(order[0] - v) == abs(order[1] - v) else f"{int(order[0])}"
                issues.append(issue(f, "error", f"{f}: {v:g} allowed nahi (allowed: {'/'.join(str(int(a)) for a in allowed)}). Paas ki value: {alt}.",
                                    fix=int(order[0]), fix_label=f"{int(order[0])} lo"))
                clean[f] = None; continue
        else:
            if not (r["hard_lo"] <= v <= r["hard_hi"]):
                issues.append(_suggest_number(f, v, r)); clean[f] = None; continue
            if not (r["min"] <= v <= r["max"]):
                issues.append(issue(f, "warning", f"{f}: {v:g} training range ({r['min']:g} se {r['max']:g}) ke bahar hai; prediction kam bharosemand ho sakti hai."))
        clean[f] = v

    # fields ke beech ki jaanch (sirf chetavni)
    a, hr = clean.get("age"), clean.get("thalach")
    if a is not None and hr is not None and hr > (220 - a) + 15:
        issues.append(issue("thalach", "warning", f"thalach {hr:g} is umar (age {a:g}) ke aam max heart rate (~{220 - a:g}) se kaafi upar hai. Value dobara check kar lo."))

    # bahut zyada missing: sabse kaam ke fields batao
    if not partial and len(missing) > MAX_MISSING:
        pri = _priority(model)
        ask = [f for f in pri if f in missing][: len(missing) - MAX_MISSING]
        issues.append(issue(None, "error", f"{len(missing)} fields khali hain (max {MAX_MISSING}). Kam se kam {len(missing) - MAX_MISSING} aur bharo - "
                            f"sabse zyada madad karne wale: {', '.join(ask)}."))
    return {"clean": clean, "issues": issues, "blocked": any(i["level"] == "error" for i in issues)}


def _priority(model):
    if model is not None and hasattr(model, "feature_importances_"):
        return [FEATURES[i] for i in np.argsort(-model.feature_importances_)]
    return list(FEATURES)


def check_record(record, rules, model=None):
    """Poora pipeline. Return input.validate_input jaisa dict + 'issues'."""
    d = diagnose(record, rules, model)
    if d["blocked"]:
        errs = [i["message"] for i in d["issues"] if i["level"] == "error"]
        warns = [i["message"] for i in d["issues"] if i["level"] != "error"]
        return {"ok": False, "x": None, "errors": errs, "warnings": warns, "filled": [], "issues": d["issues"]}
    res = validate_input(d["clean"], rules)
    notes = [i["message"] for i in d["issues"] if i["level"] in ("warning", "fixed")]
    res["warnings"] = notes + [w for w in res["warnings"] if w not in notes]
    res["issues"] = d["issues"]
    return res


def apply_fix(record, iss):
    """Suggestion maanne par record badalta hai. Naam-typo me value naye field par jaati hai, warna field ki value badalti hai."""
    out = dict(record)
    if iss["field"] is None and iss.get("value") is not None:      
        out[iss["fix"]] = iss["value"]
    elif iss["field"] in FEATURES:
        out[iss["field"]] = iss["fix"]
    return out

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import importlib, joblib, pandas as pd, numpy as np
import validation as V; importlib.reload(V)
from input import load_rules
model = joblib.load('heart_model_v20261013_160903.joblib'); rules = load_rules('heart.csv')
df_all = pd.read_csv('heart.csv').drop_duplicates()

line = "age 63, choll 5.2 mmol/L, bp 130; oldpeak 1,5, thalach 250 bpm, cpp 2, sex male, ca 9"
record, issues = V.parse_quick(line)         
d = V.diagnose(record, rules, model, partial=True)   
print('Likha gaya :', line)
print('Samjha     :', record, '\n')
for i in issues + d['issues']:
    sug = f"   -> suggestion: {i['fix_label']}" if i['fix'] is not None else ''
    print(f"[{i['level']:7s}] {i['message']}{sug}")

In [ ]:
good = df_all[V.FEATURES].iloc[0].to_dict()
cases = {'comma decimal (oldpeak "2,3")': {'oldpeak': '2,3'}, 'unit likha ("63 yrs")': {'age': '63 yrs'},
         'cholesterol mmol/L me (5.2)': {'chol': 5.2}, 'age mahino me (756)': {'age': 756},
         'negative BP (-120)': {'trestbps': -120}, 'category 1.5 (cp)': {'cp': 1.5}, 'category 9 (ca)': {'ca': 9},
         'text ("sixty")': {'age': 'sixty'}, 'bahut bada (chol 5000)': {'chol': 5000}}
rows = []
for name, change in cases.items():
    rec = {**good, **change}
    d = V.diagnose(rec, rules, model)
    errs = [i for i in d['issues'] if i['level'] == 'error']
    if errs and errs[0]['fix'] is not None:
        after = V.check_record(V.apply_fix(rec, errs[0]), rules, model)['ok']; outcome = 'error + suggestion'
    elif errs:
        after, outcome = False, 'error (suggestion nahi)'
    else:
        after = V.check_record(rec, rules, model)['ok']; outcome = 'auto-fix'
    msg = (errs[0]['message'] if errs else d['issues'][0]['message'])[:95]
    rows.append({'galti': name, 'outcome': outcome, 'valid ban gaya': after, 'message': msg})
pd.DataFrame(rows).set_index('galti')

In [ ]:
import random, time
rng = random.Random(0); recs = df_all[V.FEATURES].to_dict('records')
def corrupt(rec):
    f = rng.choice(V.FEATURES); kind = rng.choice(['comma', 'unit', 'text', 'negative', 'huge', 'category', 'mmol', 'months'])
    r, c = dict(rec), rec[f]; num = rules[f]['kind'] == 'number'
    if kind == 'comma' and num and float(c) != int(c): r[f] = str(c).replace('.', ',')
    elif kind == 'unit' and num: r[f] = f'{c:g} bpm' if f == 'thalach' else f'{c:g} yrs'
    elif kind == 'text': r[f] = 'sixty'
    elif kind == 'negative' and num and c > 0: r[f] = -c
    elif kind == 'huge' and num: r[f] = c * 20 + 1000
    elif kind == 'category' and not num: r[f] = max(rules[f]['allowed']) + rng.choice([1.5, 2, 7])
    elif kind == 'mmol' and f == 'chol': r[f] = round(c / 38.67, 1)
    elif kind == 'months' and f == 'age': r[f] = c * 12
    else: return None
    return r, f, kind
stats, times = {}, []
for _ in range(3000):
    out = corrupt(rng.choice(recs))
    if out is None: continue
    r, f, kind = out; t = time.perf_counter(); d = V.diagnose(r, rules, model); times.append((time.perf_counter() - t) * 1000)
    mine = [i for i in d['issues'] if i['field'] == f]; errs = [i for i in mine if i['level'] == 'error']
    s = stats.setdefault(kind, dict(n=0, detected=0, suggested=0, valid_after=0, crashed=0)); s['n'] += 1; s['detected'] += bool(mine)
    if errs and errs[0]['fix'] is not None:
        s['suggested'] += 1; s['valid_after'] += V.check_record(V.apply_fix(r, errs[0]), rules, model)['ok']
    elif not errs and mine:
        s['suggested'] += 1; s['valid_after'] += V.check_record(r, rules, model)['ok']      # auto-fix
t = pd.DataFrame(stats).T; t['detected %'] = (t.detected / t.n * 100).round(0); t['fixed/suggested %'] = (t.suggested / t.n * 100).round(0)
t['valid after %'] = (t.valid_after / t.suggested.clip(lower=1) * 100).round(0)
print(f'{int(t.n.sum())} bigade hue inputs | diagnose time: {np.mean(times):.3f} ms per input')
t[['n', 'detected %', 'fixed/suggested %', 'valid after %']]

In [ ]:
junk = [None, [], {}, 5, 'abc', {'age': [1, 2]}, {'age': {'a': 1}}, {f: '' for f in V.FEATURES}, {f: float('nan') for f in V.FEATURES}, {'age': True}]
crashed = 0
for j in junk:
    try: V.check_record(j, rules, model)
    except Exception as e: crashed += 1; print('CRASH:', repr(j)[:40], type(e).__name__, e)
print(f'{len(junk)} bekaar inputs, crashes = {crashed}')
r = V.check_record({f: '' for f in V.FEATURES}, rules, model); print('Sab khali:', r['errors'][0])

## Task 8: Add a feature that explains model predictions to users by integrating natural language generation techniques to provide human-readable justifications, dynamically adjusting explanations based on prediction confidence.

In [ ]:
%%writefile explain_text.py
"""
HeartCare - prediction ko aam bhasha me samjhana (natural language generation).

Method: template-based NLG (jaan-boojhkar, taaki har shabd data se verify ho sake, koi "banawati" claim na ho).
  1. Content selection : SHAP se top drivers (prediction ki taraf + khilaf), user ne jo values nahi di unka zikr
  2. Confidence tier   : margin (probability threshold se kitni door), estimated values, mixed signals -> 4 tier
  3. Sentence planning : tier ke hisaab se lambai aur hedging badalti hai (high = chhota aur seedha; borderline = zyada savdhaan)
  4. Counterfactual    : ek value badalne se prediction palatti hai ya nahi (model ke hisaab se)
Har sentence ke saath 'facts' bhi lautta hai, taaki test kar sakein ki likha hua sach hai.
"""
import numpy as np

from confidence import CLASS_NAMES, FEATURES, sweep, sweep_values

HUMAN = {"age": "age", "sex": "sex", "cp": "chest pain type", "trestbps": "resting blood pressure", "chol": "cholesterol",
         "fbs": "fasting blood sugar", "restecg": "resting ECG", "thalach": "maximum heart rate", "exang": "exercise-induced angina",
         "oldpeak": "ST depression (oldpeak)", "slope": "ST slope", "ca": "number of major vessels (ca)", "thal": "thal test result"}
UNIT = {"age": " saal", "trestbps": " mmHg", "chol": " mg/dl", "thalach": " bpm"}

TIERS = { 
    "high": (0.30, "&#10003;", "High confidence", "model ko kaafi bharosa hai ki"),
    "moderate": (0.15, "&#9679;", "Moderate confidence", "model ka jhukaav is taraf hai ki"),
    "low": (0.05, "&#9888;", "Low confidence (close call)", "model ko thoda hi jhukaav hai ki"),
    "toss-up": (0.0, "&#9888;", "Toss-up", "model lagbhag barabar soch raha hai, halka sa jhukaav ye hai ki"),
}
ORDER = ["high", "moderate", "low", "toss-up"]


def pct(p):
    return "<1%" if p < 0.005 else (">99%" if p > 0.995 else f"{p * 100:.0f}%")


def value_text(f, v):
    return f"{v:g}{UNIT.get(f, '')}"


def confidence_tier(prob, threshold=0.5, n_filled=0, phi=None):
    """Return (tier, reasons, base). base = sirf margin se tier; tier = estimated values / mixed signals se ek level neeche."""
    margin = abs(prob - threshold)
    base = next(t for t in ORDER if margin >= TIERS[t][0])
    reasons = []
    i = ORDER.index(base)
    if n_filled >= 2 and i < 3:
        i += 1; reasons.append(f"{n_filled} values aapne nahi di (estimated), isliye confidence ek level kam maani gayi")
    if phi is not None:
        pos, neg = phi[phi > 0].sum(), -phi[phi < 0].sum()
        lead, opp = max(pos, neg), min(pos, neg)
        if lead > 0 and opp / lead >= 0.6 and i < 3:
            i += 1; reasons.append("kuch features ek taraf aur kuch doosri taraf khinch rahe hain (mixed signals)")
    return ORDER[i], reasons, base


def counterfactual(model, x, rules, threshold=0.5, features=("chol", "trestbps", "thalach", "oldpeak")):
    """Kisi ek number feature ko sabse kam badalkar decision palatti hai? Return dict ya None. (Model ka pattern, medical salah nahi.)"""
    base_p = float(sweep(model, x, features[0], [float(x[FEATURES.index(features[0])])])[0])
    want_high = base_p <= threshold
    best = None
    for f in features:
        r = rules[f]; cur = float(x[FEATURES.index(f)])
        vals = sweep_values(r, 41)
        ps = sweep(model, x, f, vals)
        for v, p in zip(vals, ps):
            crossed = (p > threshold) if want_high else (p <= threshold)
            if crossed:
                rel = abs(v - cur) / (r["max"] - r["min"])
                if best is None or rel < best["rel"]:
                    best = {"feature": f, "from": cur, "to": float(v), "p_after": float(p), "rel": rel}
    return best


def explain(info_shap, prob, threshold=0.5, filled=(), model=None, rules=None, x=None, population=None):
    """Return {'sentences': [...], 'tier', 'reasons', 'facts': [...], 'counterfactual'}."""
    phi = info_shap["phi"]; xs = info_shap["x"]
    tier, reasons, base = confidence_tier(prob, threshold, len(filled), phi)
    decision = 1 if prob > threshold else 0
    lead_cls, other_cls = CLASS_NAMES[decision], CLASS_NAMES[1 - decision]
    sign = 1 if decision == 1 else -1                    
    order = np.argsort(-np.abs(phi))
    for_ = [i for i in order if np.sign(phi[i]) == sign][:2 if tier == "high" else 3]
    against = [i for i in order if np.sign(phi[i]) == -sign][:1]
    facts, sents = [], []

    p_lead = prob if decision == 1 else 1 - prob
    sents.append(f"{TIERS[base][3]} is patient ka profile <b>{lead_cls}</b> hai (model ke hisaab se {CLASS_NAMES[1]} probability {pct(prob)}, "
                 f"threshold {threshold * 100:.0f}%).")
    facts.append({"type": "headline", "decision": decision, "prob": prob})

    def phrase(i):
        f = FEATURES[i]; est = " (estimated value)" if f in filled else ""
        pct = ""
        if population is not None and f in population and f in UNIT:
            share = float((np.asarray(population[f]) < xs[i]).mean() * 100)
            pct = f", jo dataset ke {share:.0f}% patients se zyada hai" if share >= 50 else f", jo dataset ke {100 - share:.0f}% patients se kam hai"
        return f"{HUMAN[f]} = {value_text(f, xs[i])}{est}{pct} (<b>{abs(phi[i]) * 100:.1f} pts</b>)"

    if for_:
        sents.append(f"Sabse zyada asar: {'; '.join(phrase(i) for i in for_)} - ye {lead_cls} ki taraf dhakel rahe hain.")
        facts += [{"type": "for", "feature": FEATURES[i], "pts": float(abs(phi[i]) * 100), "dir": sign} for i in for_]
    if against and tier != "high":
        i = against[0]
        sents.append(f"Ulta asar: {phrase(i)} - ye {other_cls} ki taraf khinch raha hai.")
        facts.append({"type": "against", "feature": FEATURES[i], "pts": float(abs(phi[i]) * 100), "dir": -sign})

    if tier == "high":
        sents.append("Kai features ek hi taraf ishara kar rahe hain, isliye ye result stable lagta hai.")
    else:
        why = " ".join(r[0].upper() + r[1:] + "." for r in reasons)
        close = base in ("low", "toss-up")                 
        line = {"moderate": "Result ek taraf jhuka hai, par pakka nahi.",
                "low": "Ye close call hai: thoda sa input badalne par result palat sakta hai." if close else
                       "Is result par bharosa kam hai; ise pakka mat maano.",
                "toss-up": "Ye lagbhag 50-50 hai; is result par zyada bharosa mat karo." if close else
                           "Is result par bharosa kaafi kam hai; ise pakka mat maano."}[tier]
        sents.append((line + " " + why).strip())

    cf = None
    if model is not None and rules is not None and x is not None and tier != "high":
        cf = counterfactual(model, x, rules, threshold)
        if cf:
            sents.append(f"Model ke hisaab se sirf {HUMAN[cf['feature']]} {value_text(cf['feature'], cf['from'])} se "
                         f"{value_text(cf['feature'], cf['to'])} hota to prediction {other_cls} ho jaati ({CLASS_NAMES[1]} probability {pct(cf['p_after'])}).")
        else:
            sents.append("In 4 numbers (cholesterol, BP, heart rate, oldpeak) me se akele ek badalne se prediction nahi palatti.")
    sents.append("Ye model ke data-patterns hain, doctor ki salah ya karan (cause) nahi.")
    return {"sentences": sents, "tier": tier, "reasons": reasons, "facts": facts, "counterfactual": cf}


CSS = """
<style>
.ex-card { background: var(--card, #fff); border: 1px solid var(--line, #cbd5e1); border-left-width: 6px; border-radius: 12px;
  padding: 16px 18px; margin: 12px 0; animation: ex-in .4s ease both; }
.ex-card.high { border-left-color: #1d4ed8; } .ex-card.moderate { border-left-color: #0f766e; }
.ex-card.low, .ex-card.toss-up { border-left-color: #b45309; }
.ex-badge { display: inline-block; font-weight: 700; font-size: 14px; padding: 3px 10px; border-radius: 999px; background: #e8eefb; color: #142b53; margin-bottom: 8px; }
.ex-card.low .ex-badge, .ex-card.toss-up .ex-badge { background: #fdf0d9; color: #6b3a05; }
.ex-card p { margin: 6px 0; line-height: 1.55; color: #1e293b; font-size: 16px; }
@keyframes ex-in { from { opacity: 0; transform: translateY(6px); } to { opacity: 1; transform: none; } }
@media (prefers-reduced-motion: reduce) { .ex-card { animation: none; } }
</style>
"""


def render(ex):
    icon, label = TIERS[ex["tier"]][1], TIERS[ex["tier"]][2]
    body = "".join(f"<p>{s}</p>" for s in ex["sentences"])
    return f'<div class="ex-card {ex["tier"]}" role="region" aria-label="Prediction explanation"><span class="ex-badge">{icon} {label}</span>{body}</div>'

In [ ]:
import re, importlib, numpy as np
import shap_values as sv, explain_text as ex; importlib.reload(ex)
from confidence import analyse, FEATURES
bg = sv.build_background(df_all, k=20)
pop = {f: df_all[f].values for f in ('age', 'trestbps', 'chol', 'thalach')}
plain = lambda e: [re.sub('<[^>]+>', '', s) for s in e['sentences']]

X60 = df_all[FEATURES].astype(float).sample(60, random_state=3).values
found = {}
for xx in X60:                             
    p = analyse(model, xx)['probs'][1]; si = sv.explain(model, xx, bg)
    e = ex.explain(si, p, 0.5, [], model, rules, xx, pop)
    found.setdefault(e['tier'], (xx, p, si, e))
for tier in ex.ORDER:
    if tier in found:
        print(f"===== {tier.upper()}  (P = {found[tier][1]:.0%})"); [print(' -', s) for s in plain(found[tier][3])]
print('\nMile hue tiers:', list(found))

In [ ]:
xx = df_all[FEATURES].astype(float).values[0]; p = analyse(model, xx)['probs'][1]; si = sv.explain(model, xx, bg)
print(f'P(Higher Risk) = {p:.0%}')
for thr, filled in [(0.5, []), (0.5, ['chol', 'trestbps']), (0.7, []), (0.75, [])]:
    e = ex.explain(si, p, thr, filled, model, rules, xx, pop)
    print(f"threshold {thr:.0%}, estimated values {filled or 'koi nahi'} -> tier = {e['tier']:8s} | sentences = {len(e['sentences'])} | words = {sum(len(s.split()) for s in plain(e))}")
print('\n--- threshold 50%, 2 values estimated:')
e = ex.explain(si, p, 0.5, ['chol', 'trestbps'], model, rules, xx, pop)
for s in plain(e): print(' -', s)

In [ ]:
from collections import Counter
bad_dir = bad_pts = bad_head = cf_n = cf_bad = 0; tiers = []; words = {}
for xx in X60:
    p = analyse(model, xx)['probs'][1]; si = sv.explain(model, xx, bg)
    for thr in (0.5, 0.7):
        e = ex.explain(si, p, thr, [], model, rules, xx, pop); text = ' '.join(plain(e)); tiers.append(e['tier'])
        words.setdefault(e['tier'], []).append(len(text.split()))
        for f in e['facts']:
            if f['type'] == 'headline': bad_head += (f['decision'] != int(p > thr))
            else:
                i = FEATURES.index(f['feature'])
                bad_dir += (np.sign(si['phi'][i]) != f['dir'])                      
                bad_pts += (abs(abs(si['phi'][i]) * 100 - f['pts']) > 1e-6 or f"{f['pts']:.1f} pts" not in text)
        cf = e['counterfactual']
        if cf:                                                                     
            x2 = xx.copy(); x2[FEATURES.index(cf['feature'])] = cf['to']; p2 = analyse(model, x2)['probs'][1]
            cf_n += 1; cf_bad += ((p2 > thr) == (p > thr)) or abs(p2 - cf['p_after']) > 1e-9
print('Explanations:', len(tiers), '| headline galat:', bad_head, '| direction galat:', bad_dir, '| pts mismatch:', bad_pts)
print('Counterfactual claims:', cf_n, '| model me verify karne par galat:', cf_bad)
print('Tier counts:', dict(Counter(tiers)))
print('Average words per explanation:', {k: round(np.mean(v)) for k, v in words.items()})

## Task 9: Improve UI aesthetics using custom CSS in Streamlit by manually styling components, optimizing color schemes for accessibility, and enhancing interactivity with animations and transitions for a seamless user experience.

In [ ]:
%%writefile theme.py
"""
HeartCare - theme: custom CSS (haath se), accessible colors (contrast check ke saath), animations/transitions.

Accessibility rules jo yahan follow hote hain:
  * Text contrast >= 4.5:1 (WCAG AA), UI parts (border, focus ring, bar) >= 3:1  -> audit() code se check karta hai
  * Rang akela matlab nahi batata: risk label me icon + text hota hai (red/green na dikhne par bhi samajh aaye)
  * Keyboard focus ring dikhta hai (:focus-visible), buttons phone par >= 44px
  * Animations chhoti (<0.5s) aur prefers-reduced-motion par band
"""

TOKENS = {
    "bg": "#f3f6fb", "card": "#ffffff", "navy": "#142b53", "blue": "#244577", "ink": "#1e293b", "muted": "#475569",
    "line": "#64748b", "soft_line": "#e2e8f0", "focus": "#1d4ed8", "risk": "#b42323", "safe": "#177245",
    "warn_ink": "#6b3a05", "warn_bg": "#fdf0d9", "on_navy": "#ffffff", "sub_on_navy": "#dbe4f5",
}

PAIRS = [
    ("body text on card", "ink", "card", 4.5), ("body text on page", "ink", "bg", 4.5),
    ("muted text on card", "muted", "card", 4.5), ("muted text on page", "muted", "bg", 4.5),
    ("title on card", "navy", "card", 4.5), ("header title on navy", "on_navy", "navy", 4.5),
    ("header subtitle on navy", "sub_on_navy", "navy", 4.5), ("button text on blue", "on_navy", "blue", 4.5),
    ("Higher Risk text on card", "risk", "card", 4.5), ("Lower Risk text on card", "safe", "card", 4.5),
    ("warning text", "warn_ink", "warn_bg", 4.5),
    ("input border on card (UI)", "line", "card", 3.0), ("focus ring on card (UI)", "focus", "card", 3.0),
    ("focus ring on page (UI)", "focus", "bg", 3.0), ("risk bar on card (UI)", "risk", "card", 3.0), ("safe bar on card (UI)", "safe", "card", 3.0),
]


def _lum(hex_color):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
    f = lambda c: c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * f(r) + 0.7152 * f(g) + 0.0722 * f(b)


def contrast(fg, bg):
    """WCAG contrast ratio (1 se 21)."""
    a, b = sorted([_lum(fg), _lum(bg)], reverse=True)
    return (a + 0.05) / (b + 0.05)


def audit(tokens=None):
    """Har pair ka ratio aur pass/fail. Return list of dict."""
    t = tokens or TOKENS
    return [{"pair": n, "fg": t[f], "bg": t[b], "ratio": round(contrast(t[f], t[b]), 2), "needed": need,
             "pass": contrast(t[f], t[b]) >= need} for n, f, b, need in PAIRS]


def risk_badge(decision, names=("Lower Risk", "Higher Risk")):
    """Icon + text (sirf rang pe nirbhar nahi)."""
    icon = "&#9650;" if decision == 1 else "&#9660;"
    cls = "badge-risk" if decision == 1 else "badge-safe"
    return f'<span class="badge {cls}">{icon} {names[decision]}</span>'


def root_vars():
    return ":root {" + "".join(f" --{k.replace('_', '-')}: {v};" for k, v in TOKENS.items()) + " }"


CSS = """
<style>
__ROOT__

/* ---------- base ---------- */
.stApp { background: var(--bg); color: var(--ink); }
header, footer { visibility: hidden; }
.main .block-container { max-width: 850px; padding: 25px 20px; animation: page-in .35s ease both; }
h1, h2, h3 { color: var(--navy); }
p, label, .stMarkdown, .stCaption, [data-testid="stCaptionContainer"] { color: var(--ink); }
[data-testid="stCaptionContainer"] { color: var(--muted); font-size: 14px; }

/* ---------- header / section cards ---------- */
.header { background-color: var(--navy); background-image: linear-gradient(135deg, var(--navy), var(--blue)); color: var(--on-navy); padding: 25px; border-radius: 14px;
  text-align: center; margin-bottom: 25px; animation: fade-up .5s ease both; }
.header h1 { color: var(--on-navy); margin: 0 0 4px; }
.header p { color: var(--sub-on-navy); margin: 0; }
.section { background: var(--card); padding: 20px; border-radius: 12px; margin-bottom: 18px; border: 1px solid var(--soft-line);
  animation: fade-up .45s ease both; transition: box-shadow .2s ease, transform .2s ease; }
.section:hover { box-shadow: 0 6px 18px rgba(20, 43, 83, .10); transform: translateY(-1px); }
.section h3 { color: var(--navy); margin-top: 0; }
.page-card { display: block; background: var(--card); border: 1px solid var(--soft-line); border-radius: 12px; padding: 16px 18px; margin: 10px 0; }

/* ---------- buttons (min 44px touch target) ---------- */
.stButton button, .stDownloadButton button { background: var(--blue); color: var(--on-navy); width: 100%; min-height: 44px; border-radius: 8px;
  border: 2px solid var(--blue); font-weight: 600;
  transition: background .15s ease, transform .1s ease, box-shadow .15s ease; }
.stButton button:hover, .stDownloadButton button:hover { background: var(--navy); border-color: var(--navy); color: var(--on-navy);
  box-shadow: 0 4px 12px rgba(20, 43, 83, .25); transform: translateY(-1px); }
.stButton button:active { transform: translateY(0) scale(.98); }
.stButton button p, .stDownloadButton button p { color: var(--on-navy); }

/* ---------- keyboard focus (dikhna zaroori) ---------- */
:focus-visible { outline: 3px solid var(--focus) !important; outline-offset: 2px !important; }
input:focus, textarea:focus, [data-baseweb="select"]:focus-within { box-shadow: 0 0 0 3px rgba(29, 78, 216, .35) !important; }

/* ---------- small badges (icon + text) ---------- */
.badge { display: inline-block; font-weight: 700; padding: 3px 10px; border-radius: 999px; font-size: 14px; }
.badge-risk { background: #fbe9e9; color: var(--risk); } .badge-safe { background: #e4f4ec; color: var(--safe); }
.notice { background: var(--warn-bg); color: var(--warn-ink); border-radius: 10px; padding: 10px 14px; margin: 8px 0; line-height: 1.5; }

/* ---------- sidebar navigation ---------- */
[data-testid="stSidebarNav"] a { border-radius: 8px; transition: background .15s ease, padding-left .15s ease; }
[data-testid="stSidebarNav"] a:hover { background: #e3eafa; padding-left: 14px; }

/* ---------- animations ---------- */
@keyframes fade-up { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: none; } }
@keyframes page-in { from { opacity: 0; } to { opacity: 1; } }
@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation: none !important; transition: none !important; }
}

/* ---------- small screens ---------- */
@media (max-width: 600px) {
  .main .block-container { padding: 12px 10px; }
  .header { padding: 18px; } .header h1 { font-size: 24px; }
  .section { padding: 14px; }
}
</style>
""".replace("__ROOT__", root_vars())


def css(extra=""):
    return CSS + extra


STREAMLIT_CONFIG = """[theme]
base = "light"
primaryColor = "#244577"
backgroundColor = "#f3f6fb"
secondaryBackgroundColor = "#ffffff"
textColor = "#1e293b"
"""

In [ ]:
import os
os.makedirs('.streamlit', exist_ok=True)

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#244577"
backgroundColor = "#f3f6fb"
secondaryBackgroundColor = "#ffffff"
textColor = "#1e293b"

In [ ]:
import importlib, pandas as pd
import theme; importlib.reload(theme)
old = {'Higher Risk text (old #d64545)': '#d64545', 'Lower Risk text (old #2e9e6b)': '#2e9e6b', 'SHAP centre line (old #94a3b8)': '#94a3b8'}
before = pd.DataFrame([{'item': k, 'ratio': round(theme.contrast(c, '#ffffff'), 2)} for k, c in old.items()])
before['needed'] = [4.5, 4.5, 3.0]; before['pass'] = before.ratio >= before.needed
print('--- Pehle (purani values):'); print(before.to_string(index=False))
after = pd.DataFrame(theme.audit())
print('\n--- Ab (theme.TOKENS):', after['pass'].sum(), 'of', len(after), 'pass'); after[['pair', 'fg', 'bg', 'ratio', 'needed', 'pass']]

In [ ]:
import confidence as C, shap_values as S
print('Charts ke colors ab:', C.COLORS)
print('Risk label / safe label white card par contrast:', round(theme.contrast(C.COLORS[1], '#ffffff'), 2), round(theme.contrast(C.COLORS[0], '#ffffff'), 2))
print('SHAP centre line colour:', '#64748b' in S.SHAP_CSS)

In [ ]:
import os, numpy as np, pandas as pd
import components as ui, explain_text as ex, shap_values as sv, confidence as C
xx = df_all[C.FEATURES].astype(float).values[40]; info = C.analyse(model, xx); p = info['probs'][1]; si = sv.explain(model, xx, bg)
e1 = ex.explain(si, p, .5, ['chol'], model, rules, xx, pop); e2 = ex.explain(si, .52, .5, ['chol', 'age'], model, rules, xx, {})
vals = C.sweep_values(rules['age'], 25); ps = C.sweep(model, xx, 'age', vals)
body = ('<div class="header"><h1>Predict</h1><p>Patient ki details do</p></div><div class="section"><h3>Basic</h3><p>text</p></div>'
        '<div class="stButton"><button id="b">Predict Heart Disease Risk</button></div><input id="i" type="text" value="age 63" style="border:1px solid #64748b">'
        '<div style="padding:8px">' + theme.risk_badge(1) + ' ' + theme.risk_badge(0) + '</div><div class="notice">Pehle Predict page par jao</div>'
        + ui.risk_card(p, .5, .6, ['chol']) + ex.render(e1) + ex.render(e2) + sv.shap_chart(si, 8, 'ca', filled=['chol'])
        + ui.curve_chart('Risk curve: age', 'age', vals, ps, float(xx[0]), p, df_all['age'].values, False)
        + C.bar_chart('Class probabilities', ['Lower Risk', 'Higher Risk'], info['probs'], colors=[C.COLORS[0], C.COLORS[1]], threshold=.5)
        + '<div style="text-align:center;color:#475569;padding:20px;">HeartCare | Educational Purpose Only</div>')
page_html = ('<html><head><meta name="viewport" content="width=device-width,initial-scale=1"></head><body style="margin:0;background:#f3f6fb;font-family:sans-serif;color:#1e293b">'
             + theme.CSS + C.CSS + ui.CSS + sv.SHAP_CSS + ex.CSS + '<div class="main"><div class="block-container">' + body + '</div></div></body></html>')
open('theme_preview.html', 'w').write(page_html)
JS = """() => {
  const lum = c => { const f = v => { v /= 255; return v <= 0.03928 ? v / 12.92 : Math.pow((v + 0.055) / 1.055, 2.4) }; return 0.2126 * f(c[0]) + 0.7152 * f(c[1]) + 0.0722 * f(c[2]) };
  const parse = s => { const m = s.match(/[\\d.]+/g).map(Number); return { rgb: m.slice(0, 3), a: m.length > 3 ? m[3] : 1 } };
  const bgOf = el => { while (el) { const c = parse(getComputedStyle(el).backgroundColor); if (c.a > 0.9) return c.rgb; el = el.parentElement } return [255, 255, 255] };
  const ratio = (a, b) => { const x = lum(a), y = lum(b); return (Math.max(x, y) + 0.05) / (Math.min(x, y) + 0.05) };
  const out = [], seen = new Set(), walker = document.createTreeWalker(document.body, NodeFilter.SHOW_TEXT);
  while (walker.nextNode()) { const n = walker.currentNode; if (!n.textContent.trim()) continue; const el = n.parentElement;
    if (el.tagName.toLowerCase() === 'title' || seen.has(el)) continue; seen.add(el); const cs = getComputedStyle(el);
    const fg = el.namespaceURI.includes('svg') ? parse(cs.fill) : parse(cs.color);
    out.push({ text: n.textContent.trim().slice(0, 28), ratio: Math.round(ratio(fg.rgb, bgOf(el)) * 100) / 100, size: parseFloat(cs.fontSize) }) }
  return out }"""
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as pw:
        b = pw.chromium.launch()
        for name, w in [('phone', 360), ('laptop', 1280)]:
            pg = b.new_context(viewport={'width': w, 'height': 900}).new_page(); pg.goto('file://' + os.path.abspath('theme_preview.html')); pg.wait_for_timeout(1500)
            r = pg.evaluate(JS)
            print(f"{name:6s} {w}px | text items {len(r)} | min contrast {min(t['ratio'] for t in r)} | below 4.5: {sum(t['ratio'] < 4.5 for t in r)} | "
                  f"smallest font {min(t['size'] for t in r):g}px | sideways scroll: {pg.evaluate('document.documentElement.scrollWidth>document.documentElement.clientWidth')}")
            if name == 'laptop':
                bg0 = pg.evaluate("getComputedStyle(document.getElementById('b')).backgroundColor"); pg.hover('#b'); pg.wait_for_timeout(400)
                bg1 = pg.evaluate("getComputedStyle(document.getElementById('b')).backgroundColor")
                pg.focus('#i'); pg.keyboard.press('Tab'); pg.keyboard.press('Shift+Tab')
                print('       button hover:', bg0, '->', bg1, '| button height:', pg.evaluate("document.getElementById('b').getBoundingClientRect().height"), 'px',
                      '| focus ring:', pg.evaluate("getComputedStyle(document.getElementById('i')).outlineWidth"))
            pg.close()
        pg = b.new_context(viewport={'width': 800, 'height': 600}, reduced_motion='reduce').new_page(); pg.goto('file://' + os.path.abspath('theme_preview.html'))
        print('reduced-motion ON: section animation =', pg.evaluate("getComputedStyle(document.querySelector('.section')).animationName"), '| button transition =',
              pg.evaluate("getComputedStyle(document.getElementById('b')).transitionDuration"))
        b.close()
except Exception as err:
    print('Playwright is machine par nahi hai (optional):', type(err).__name__, '- neeche preview se visually dekho.')

In [ ]:
from IPython.display import HTML
HTML(page_html)

In [ ]:
import os
BASE = {
    'input.py': '"""\nHeartCare - user input handling for prediction.\n\n1. Dynamic validation : rules (allowed categories, number ranges) heart.csv se khud banti hain.\n2. Edge cases         : missing, text, out-of-range, unknown category, inf/NaN, extra keys, wrong length.\n3. Real-time speed    : ek hi predict_proba pass, repeated input ka cache, batch me vectorized validation.\n"""\nimport numpy as np\nimport pandas as pd\nfrom functools import lru_cache\n\nFEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",\n            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]\nMISSING_WORDS = {"", "na", "n/a", "nan", "null", "none", "?", "-", "unknown", "not known"}\nMAX_MISSING = 3            # isse zyada features missing ho to prediction reject\n\n\n# ---------------------------------------------------------------- 1. dynamic rules\ndef build_rules(df):\n    """Training data se rules: <= 5 alag values wala column = category, baaki = number."""\n    rules = {}\n    for f in FEATURES:\n        col = df[f].astype(float).dropna()\n        vals = np.sort(col.unique())\n        r = {"min": float(col.min()), "max": float(col.max())}\n        if len(vals) <= 5:\n            r.update(kind="category", allowed=vals.tolist(), fill=float(col.mode()[0]))\n        else:\n            span = col.max() - col.min()\n            lo, hi = col.min() - 0.25 * span, col.max() + 0.25 * span     # impossible-value limit\n            if col.min() >= 0:\n                lo = max(lo, 0.0)                                          # negative nahi ho sakta\n            r.update(kind="number", hard_lo=float(lo), hard_hi=float(hi), fill=float(col.median()))\n        rules[f] = r\n    return rules\n\n\ndef load_rules(csv_path="heart.csv"):\n    return build_rules(pd.read_csv(csv_path).drop_duplicates().dropna())\n\n\n# ---------------------------------------------------------------- 2. single input\ndef is_missing(v):\n    if v is None or v is pd.NA:\n        return True\n    if isinstance(v, str):\n        return v.strip().lower() in MISSING_WORDS\n    return isinstance(v, (float, np.floating)) and np.isnan(v)\n\n\ndef to_number(v):\n    if isinstance(v, (bool, int, float, np.integer, np.floating)):\n        return float(v)\n    if isinstance(v, str):\n        t = v.strip().lower().replace(",", ".")            # "2,3" bhi chalega\n        return {"true": 1.0, "false": 0.0}.get(t) if t in ("true", "false") else float(t)\n    raise TypeError("number nahi hai")\n\n\ndef validate_input(record, rules):\n    """record = dict ya 13 values ki list. Return: {ok, x, errors, warnings, filled}"""\n    out = {"ok": False, "x": None, "errors": [], "warnings": [], "filled": []}\n    if isinstance(record, (list, tuple, np.ndarray)):\n        if len(record) != len(FEATURES):\n            out["errors"].append(f"{len(FEATURES)} values chahiye, {len(record)} mili")\n            return out\n        record = dict(zip(FEATURES, record))\n    if not isinstance(record, dict):\n        out["errors"].append("input dict ya list hona chahiye")\n        return out\n\n    rec = {str(k).strip().lower().replace(" ", "_"): v for k, v in record.items()}   # " Age " -> "age"\n    extra = [k for k in rec if k not in rules]\n    if extra:\n        out["warnings"].append(f"extra fields ignore kiye: {extra}")\n\n    x = np.zeros(len(FEATURES))\n    for i, f in enumerate(FEATURES):\n        v, r = rec.get(f), rules[f]\n        if is_missing(v):\n            x[i] = r["fill"]; out["filled"].append(f); continue\n        try:\n            val = to_number(v)\n        except (ValueError, TypeError):\n            out["errors"].append(f"{f}: \'{v}\' number nahi hai"); continue\n        if not np.isfinite(val):\n            out["errors"].append(f"{f}: inf/NaN allowed nahi"); continue\n        if r["kind"] == "category":\n            if val not in r["allowed"]:\n                out["errors"].append(f"{f}: {val:g} allowed nahi (allowed: {[int(a) for a in r[\'allowed\']]})")\n        elif not r["hard_lo"] <= val <= r["hard_hi"]:\n            out["errors"].append(f"{f}: {val:g} impossible value (limit {r[\'hard_lo\']:.1f} se {r[\'hard_hi\']:.1f})")\n        elif not r["min"] <= val <= r["max"]:\n            out["warnings"].append(f"{f}: {val:g} training range [{r[\'min\']:g}, {r[\'max\']:g}] ke bahar hai, prediction kam bharosemand")\n        x[i] = val\n\n    if len(out["filled"]) > MAX_MISSING:\n        out["errors"].append(f"{len(out[\'filled\'])} fields missing hain (max {MAX_MISSING} allowed)")\n    elif out["filled"]:\n        out["warnings"].append(f"missing fields median/mode se bhare gaye: {out[\'filled\']}")\n    if not out["errors"]:\n        out["ok"], out["x"] = True, x\n    return out\n\n\n# ---------------------------------------------------------------- prediction (fast)\ndef predict_result(model, x):\n    """x = 13 numbers. Pehle 2 pass (predict + predict_proba) the, ab sirf 1 pass."""\n    probabilities = model.predict_proba(pd.DataFrame([x], columns=FEATURES))[0]\n    prediction = int(model.classes_[np.argmax(probabilities)])\n    return prediction, probabilities\n\n\ndef make_predictor(model, size=1024):\n    """Same input dobara aaye to model dobara nahi chalta (cache)."""\n    @lru_cache(maxsize=size)\n    def cached(key):\n        return predict_result(model, list(key))\n    return lambda x: cached(tuple(float(v) for v in x))\n\n\n# ---------------------------------------------------------------- 3. batch (vectorized)\ndef _to_numbers(s):\n    """Series -> (numbers, missing_mask). Text wali value NaN banti hai aur missing nahi maani jati."""\n    if pd.api.types.is_numeric_dtype(s):\n        num = s.astype(float).to_numpy()\n        return num, np.isnan(num)\n    txt = s.astype(str).str.strip().str.lower()\n    miss = (s.isna() | txt.isin(MISSING_WORDS)).to_numpy()\n    num = pd.to_numeric(txt.str.replace(",", ".", regex=False).replace({"true": "1", "false": "0"}), errors="coerce").to_numpy(dtype=float)\n    return num, miss\n\n\ndef validate_batch(raw, rules):\n    """DataFrame ki saari rows ek saath validate. Return: X, ok(bool array), reason(Series)"""\n    raw = raw.rename(columns=lambda c: str(c).strip().lower().replace(" ", "_"))\n    n = len(raw)\n    X, bad, n_miss = np.zeros((n, len(FEATURES))), np.zeros(n, bool), np.zeros(n, int)\n    reason = pd.Series([""] * n, index=raw.index, dtype=object)\n    for i, f in enumerate(FEATURES):\n        r = rules[f]\n        if f in raw:\n            num, miss = _to_numbers(raw[f])\n        else:\n            num, miss = np.full(n, np.nan), np.ones(n, bool)\n        bad_f = ~miss & ~np.isfinite(num)                                 # text ya inf\n        ok_num = ~miss & np.isfinite(num)\n        if r["kind"] == "category":\n            bad_f |= ok_num & ~np.isin(num, r["allowed"])\n        else:\n            bad_f |= ok_num & ((num < r["hard_lo"]) | (num > r["hard_hi"]))\n        reason = reason + np.where(bad_f, f + " galat; ", "")\n        bad |= bad_f; n_miss += miss\n        X[:, i] = np.where(miss | bad_f, r["fill"], num)\n    too_many = n_miss > MAX_MISSING\n    reason = reason + np.where(too_many, "bahut fields missing; ", "")\n    return X, ~(bad | too_many), reason\n\n\ndef predict_batch(model, rules, raw):\n    """CSV/DataFrame ke liye: ek hi predict_proba call poori valid rows par."""\n    X, ok, reason = validate_batch(raw, rules)\n    out = pd.DataFrame({"status": np.where(ok, "ok", "rejected"), "reason": reason.str.strip("; ").to_numpy(),\n                        "prediction": np.nan, "probability_%": np.nan}, index=raw.index)\n    if ok.any():\n        p = model.predict_proba(pd.DataFrame(X[ok], columns=FEATURES))\n        out.loc[ok, "prediction"] = model.classes_[p.argmax(axis=1)]\n        out.loc[ok, "probability_%"] = (p.max(axis=1) * 100).round(2)\n    return out\n',
    'cache.py': '"""\nHeartCare - custom caching for predictions (Streamlit ka built-in cache nahi).\n\n- LRU     : size limit; sabse purani entry hatti hai\n- TTL     : entry kuch der baad expire (model/data purana na ho)\n- Thread-safe : Lock se\n- Single-flight : same input par 10 users ek saath aaye to model sirf 1 baar chalta hai,\n                  baaki 9 wahi result ka wait karte hain (concurrent requests efficient)\n"""\nimport time\nimport threading\nfrom collections import OrderedDict\n\n\nclass _Call:\n    """Jo computation chal rahi hai, uska result yahan rakhte hain."""\n    def __init__(self):\n        self.done = threading.Event()\n        self.value = None\n        self.error = None\n\n\nclass PredictionCache:\n\n    def __init__(self, max_size=512, ttl=600):\n        self.max_size = max_size\n        self.ttl = ttl                      # seconds; None = kabhi expire nahi\n        self._data = OrderedDict()          # key -> (value, created_time)\n        self._inflight = {}                 # key -> _Call\n        self._lock = threading.Lock()\n        self.hits = self.misses = self.waits = self.evictions = self.expired = 0\n\n    def get_or_compute(self, key, compute):\n        """Cache me ho to wahi do, warna compute() chalao (same key par ek hi baar)."""\n        with self._lock:\n            item = self._data.get(key)\n            if item is not None:\n                value, created = item\n                if self.ttl is None or time.monotonic() - created < self.ttl:\n                    self._data.move_to_end(key)          # recently used\n                    self.hits += 1\n                    return value\n                del self._data[key]                      # expire ho chuka\n                self.expired += 1\n\n            call = self._inflight.get(key)\n            owner = call is None\n            if owner:\n                call = self._inflight[key] = _Call()\n                self.misses += 1\n            else:\n                self.waits += 1\n\n        if owner:\n            try:\n                call.value = compute()                   # lock ke bahar: dusre keys block nahi hote\n            except BaseException as error:\n                call.error = error                       # error cache nahi hota, sirf waiters ko milta hai\n            with self._lock:\n                if call.error is None:\n                    self._data[key] = (call.value, time.monotonic())\n                    self._data.move_to_end(key)\n                    while len(self._data) > self.max_size:\n                        self._data.popitem(last=False)\n                        self.evictions += 1\n                del self._inflight[key]\n            call.done.set()\n        else:\n            call.done.wait()\n\n        if call.error is not None:\n            raise call.error\n        return call.value\n\n    def clear(self):\n        with self._lock:\n            self._data.clear()\n\n    def stats(self):\n        with self._lock:\n            total = self.hits + self.misses + self.waits\n            return {"size": len(self._data), "hits": self.hits, "misses": self.misses,\n                    "waits": self.waits, "evictions": self.evictions, "expired": self.expired,\n                    "hit_rate_%": round(100 * (self.hits + self.waits) / total, 1) if total else 0.0}\n',
    'confidence.py': '"""\nHeartCare - prediction confidence ko bar charts me dikhana.\n\n1. Probability distribution HAATH se: Random Forest ki har tree ka probability nikal ke khud average karte hain\n   (model.predict_proba ka use sirf check karne ke liye).\n2. Charts HTML + CSS se bante hain. Har bar purani value se nayi value tak animate hota hai (CSS keyframes).\n3. Interactive: threshold slider, what-if (ek feature badalke confidence kaise badalti hai).\n"""\nimport numpy as np\nimport pandas as pd\n\nFEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",\n            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]\nCLASS_NAMES = {0: "Lower Risk", 1: "Higher Risk"}        # label badalna ho to sirf yahan\nCOLORS = {0: "#177245", 1: "#b42323"}          # accessible: white par contrast 5.95 aur 6.55 (WCAG AA 4.5+)\nN_BINS = 10\n\n\n# ---------------------------------------------------------------- 1. probability distribution (manual)\ndef analyse(model, x):\n    """Ek patient ke liye poori confidence picture. Cache me yahi dict jati hai."""\n    x32 = np.ascontiguousarray(x, dtype=np.float32).reshape(1, -1)               # trees float32 lete hain\n    per_tree = np.array([t.predict_proba(x32, check_input=False)[0] for t in model.estimators_])\n    probs = per_tree.mean(axis=0)                                                  # har class ki probability (manual average)\n    votes = np.bincount(per_tree.argmax(axis=1), minlength=len(probs))             # kitni trees ne kaunsi class chuni\n    hist = np.histogram(per_tree[:, 1], bins=N_BINS, range=(0, 1))[0]              # trees ke P(class 1) ka distribution\n    return {"probs": probs, "votes": votes, "hist": hist, "n_trees": len(per_tree),\n            "std": float(per_tree[:, 1].std()), "x": np.asarray(x, dtype=float)}\n\n\ndef decide(probs, threshold=0.5):\n    """Threshold ke hisaab se class (default 0.5 = model.predict jaisa)."""\n    return int(probs[1] > threshold)\n\n\ndef sweep(model, x, feature, values):\n    """What-if: ek feature ki value badalke P(class 1) (ek hi predict_proba call me)."""\n    rows = np.tile(np.asarray(x, dtype=float), (len(values), 1))\n    rows[:, FEATURES.index(feature)] = values\n    return model.predict_proba(pd.DataFrame(rows, columns=FEATURES))[:, 1]\n\n\ndef sweep_values(rule, n=10):\n    """Rules (input.py) se feature ki values: category = allowed, number = n barabar points."""\n    if rule["kind"] == "category":\n        return [float(v) for v in rule["allowed"]]\n    return [float(v) for v in np.round(np.linspace(rule["min"], rule["max"], n), 1)]\n\n\n# ---------------------------------------------------------------- 2. HTML/CSS charts (animated)\nCSS = """\n<style>\n.hc-chart { background: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 16px; margin-bottom: 14px; }\n.hc-chart h4 { margin: 0 0 12px 0; color: #142b53; font-size: 16px; }\n.hc-row { display: flex; align-items: center; margin: 8px 0; }\n.hc-label { width: 120px; font-size: 14px; color: #334155; }\n.hc-track { position: relative; flex: 1; height: 22px; background: #eef2f7; border-radius: 6px; overflow: hidden; }\n.hc-fill { height: 100%; width: var(--to); animation: hc-grow 0.9s ease-out; }\n.hc-value { width: 64px; text-align: right; font-size: 14px; font-weight: 600; color: #142b53; }\n.hc-thr { position: absolute; top: 0; bottom: 0; width: 2px; background: #142b53; }\n.hc-cols { display: flex; align-items: flex-end; height: 130px; gap: 6px; border-bottom: 1px solid #cbd5e1; }\n.hc-colwrap { flex: 1; height: 100%; display: flex; flex-direction: column; justify-content: flex-end; align-items: center; }\n.hc-col { width: 100%; height: var(--to); border-radius: 4px 4px 0 0; animation: hc-rise 0.9s ease-out; }\n.hc-colnum { font-size: 11px; color: #475569; height: 14px; }\n.hc-xlabels { display: flex; gap: 6px; margin-top: 4px; }\n.hc-xlabels span { flex: 1; text-align: center; font-size: 11px; color: #475569; }\n.hc-note { font-size: 13px; color: #64748b; margin-top: 8px; }\n@keyframes hc-grow { from { width: var(--from); } to { width: var(--to); } }\n@keyframes hc-rise { from { height: var(--from); } to { height: var(--to); } }\n@media (max-width: 600px) { .hc-label { width: 84px; font-size: 12px; } .hc-value { width: 52px; } }\n@media (prefers-reduced-motion: reduce) { .hc-fill, .hc-col { animation: none; } }\n</style>\n"""\n\n\ndef _prev(prev, i, scale):\n    """Purani value (animation ka start). Pehli baar 0 se grow hota hai."""\n    return 0.0 if prev is None or i >= len(prev) else float(prev[i]) * scale\n\n\ndef bar_chart(title, labels, values, prev=None, colors=None, threshold=None, note=""):\n    """Horizontal bars. values 0-1 (probability). threshold = vertical line."""\n    rows = ""\n    for i, (lab, v) in enumerate(zip(labels, values)):\n        color = colors[i] if colors else "#244577"\n        thr = f\'<div class="hc-thr" style="left:{threshold * 100:.1f}%"></div>\' if threshold is not None else ""\n        rows += (f\'<div class="hc-row"><div class="hc-label">{lab}</div>\'\n                 f\'<div class="hc-track"><div class="hc-fill" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div>{thr}</div>\'\n                 f\'<div class="hc-value">{v * 100:.1f}%</div></div>\')\n    note_html = f\'<div class="hc-note">{note}</div>\' if note else ""\n    return f\'<div class="hc-chart"><h4>{title}</h4>{rows}{note_html}</div>\'\n\n\ndef column_chart(title, x_labels, values, prev=None, color="#244577", counts=None, note=""):\n    """Vertical bars. values 0-1 (height ka fraction); counts = bar ke upar likha number."""\n    cols, labs = "", ""\n    for i, (lab, v) in enumerate(zip(x_labels, values)):\n        shown = counts[i] if counts is not None else f"{v * 100:.0f}%"\n        cols += (f\'<div class="hc-colwrap"><div class="hc-colnum">{shown}</div>\'\n                 f\'<div class="hc-col" style="--from:{_prev(prev, i, 100):.1f}%;--to:{v * 100:.1f}%;background:{color}"></div></div>\')\n        labs += f"<span>{lab}</span>"\n    note_html = f\'<div class="hc-note">{note}</div>\' if note else ""\n    return f\'<div class="hc-chart"><h4>{title}</h4><div class="hc-cols">{cols}</div><div class="hc-xlabels">{labs}</div>{note_html}</div>\'\n',
    'components.py': '"""\nHeartCare - custom visualization components (HTML + inline SVG + CSS, koi chart library nahi).\n\n- risk_card   : live result card (sliders hilte hi update; purani value se nayi tak animate)\n- curve_chart : interactive risk curve (SVG): population histogram + model ka risk curve + "aap yahan hain" marker\n- percentile_text : slider ke neeche "aap dataset me kahan hain"\n\nMultiple devices: SVG viewBox se chart apne aap scale hota hai, cards flex-wrap karte hain,\nfonts mobile par chhote nahi hote, animations \'reduced motion\' setting ko maante hain.\n"""\nimport numpy as np\n\nfrom confidence import CLASS_NAMES, COLORS\n\nCSS = """\n<style>\n.ui-card { background: white; border: 1px solid #e2e8f0; border-radius: 14px; padding: 16px 18px; margin-bottom: 14px; }\n.ui-top { display: flex; flex-wrap: wrap; align-items: center; gap: 12px 20px; }\n.ui-big { font-size: 40px; font-weight: 700; line-height: 1.1; color: #142b53; }\n.ui-label { font-size: 16px; font-weight: 600; }\n.ui-sub { font-size: 13px; color: #64748b; }\n.ui-meter { position: relative; height: 14px; border-radius: 7px; margin-top: 14px; background: linear-gradient(90deg, #2e9e6b, #f2c94c, #d64545); }\n.ui-pin { position: absolute; top: -5px; width: 6px; height: 24px; border-radius: 3px; background: #142b53; left: var(--to); margin-left: -3px; animation: ui-move 0.8s ease-out; }\n.ui-thr { position: absolute; top: -2px; bottom: -2px; width: 2px; background: #475569; opacity: .7; }\n.ui-scale { display: flex; justify-content: space-between; font-size: 12px; color: #64748b; margin-top: 4px; }\n.ui-svg { width: 100%; height: auto; max-width: 640px; display: block; margin: 0 auto; }\n.ui-svg text { font-family: inherit; fill: #475569; }\n.ui-line { fill: none; stroke: #d64545; stroke-width: 2.5; stroke-linejoin: round; stroke-dasharray: 1; stroke-dashoffset: 0; animation: ui-draw 1s ease-out; }\n.ui-pt:hover { r: 6; }\n@keyframes ui-move { from { left: var(--from); } to { left: var(--to); } }\n@keyframes ui-draw { from { stroke-dashoffset: 1; } to { stroke-dashoffset: 0; } }\n@media (max-width: 600px) { .ui-big { font-size: 32px; } .ui-card { padding: 12px; } }\n@media (prefers-reduced-motion: reduce) { .ui-pin, .ui-line { animation: none; } }\n</style>\n"""\n\n\ndef risk_card(prob, threshold=0.5, prev_prob=None, estimated=()):\n    """Top par live card: P(Higher Risk), decision, pichle update se farak, meter."""\n    decision = 1 if prob > threshold else 0\n    start = prob if prev_prob is None else prev_prob\n    delta = "" if prev_prob is None or abs(prob - prev_prob) < 1e-9 else \\\n        f\'<span class="ui-sub">{"&#9650;" if prob > prev_prob else "&#9660;"} {abs(prob - prev_prob) * 100:.1f} pts pichhle update se</span>\'\n    est = f\'<div class="ui-sub">Estimated (aapne nahi diya): {", ".join(estimated)}</div>\' if estimated else ""\n    return (f\'<div class="ui-card"><div class="ui-top"><div class="ui-big">{prob * 100:.1f}%</div>\'\n            f\'<div><div class="ui-label" style="color:{COLORS[decision]}">{CLASS_NAMES[decision]}</div>\'\n            f\'<div class="ui-sub">{CLASS_NAMES[1]} probability (threshold {threshold * 100:.0f}%)</div>{delta}</div></div>\'\n            f\'<div class="ui-meter"><div class="ui-thr" style="left:{threshold * 100:.1f}%"></div>\'\n            f\'<div class="ui-pin" style="--from:{start * 100:.1f}%;--to:{prob * 100:.1f}%"></div></div>\'\n            f\'<div class="ui-scale"><span>0%</span><span>50%</span><span>100%</span></div>{est}</div>\')\n\n\ndef curve_chart(title, feature, xs, ps, current_x, current_p, population, is_category, threshold=0.5):\n    """SVG risk curve. xs/ps = feature ki values aur model ka P(Higher Risk); population = dataset ki us feature ki values."""\n    W, H, L, R, T, B = 380, 240, 44, 10, 14, 46\n    pw, ph = W - L - R, H - T - B\n    xs = np.asarray(xs, float); ps = np.asarray(ps, float)\n    lo, hi = xs.min(), xs.max()\n    sx = (lambda v: L + (np.where(np.asarray(xs) == v)[0][0] + 0.5) / len(xs) * pw) if is_category else \\\n         (lambda v: L + (float(v) - lo) / (hi - lo + 1e-9) * pw)\n    sy = lambda p: T + (1 - float(p)) * ph\n    out = [f\'<svg class="ui-svg" viewBox="0 0 {W} {H}" role="img" aria-label="{title}">\']\n    for g in (0, .25, .5, .75, 1):                                                   # grid + y labels\n        out.append(f\'<line x1="{L}" x2="{W - R}" y1="{sy(g):.1f}" y2="{sy(g):.1f}" stroke="#e2e8f0"/><text x="{L - 6}" y="{sy(g) + 4:.1f}" font-size="13" text-anchor="end">{g * 100:.0f}%</text>\')\n    pop = np.asarray(population, float)                                              # population histogram (peeche halka)\n    if is_category:\n        counts = np.array([(pop == v).sum() for v in xs]); edges = None\n    else:\n        counts, edges = np.histogram(pop, bins=12, range=(lo, hi))\n    top = counts.max() or 1\n    for i, c in enumerate(counts):\n        if is_category:\n            x0, bw = L + i / len(xs) * pw + 4, pw / len(xs) - 8\n        else:\n            x0, bw = L + (edges[i] - lo) / (hi - lo + 1e-9) * pw + 1, (edges[i + 1] - edges[i]) / (hi - lo + 1e-9) * pw - 2\n        bh = c / top * ph * 0.35\n        out.append(f\'<rect x="{x0:.1f}" y="{T + ph - bh:.1f}" width="{max(bw, 1):.1f}" height="{bh:.1f}" fill="#cbd5e1" opacity=".6"><title>{int(c)} patients</title></rect>\')\n    ty = sy(threshold)                                                               # threshold line\n    out.append(f\'<line x1="{L}" x2="{W - R}" y1="{ty:.1f}" y2="{ty:.1f}" stroke="#475569" stroke-dasharray="4 4"/><text x="{W - R}" y="{ty - 4:.1f}" font-size="12" text-anchor="end">threshold</text>\')\n    pts = " ".join(f"{sx(x):.1f},{sy(p):.1f}" for x, p in zip(xs, ps))\n    out.append(f\'<polyline class="ui-line" pathLength="1" points="{pts}"/>\')\n    for x, p in zip(xs, ps):                                                         # hover/touch tooltip points\n        out.append(f\'<circle class="ui-pt" cx="{sx(x):.1f}" cy="{sy(p):.1f}" r="4" fill="#d64545"><title>{feature} = {x:g}: {CLASS_NAMES[1]} {p * 100:.1f}%</title></circle>\')\n    cx = sx(xs[np.abs(xs - current_x).argmin()]) if is_category else sx(min(max(current_x, lo), hi))   # "aap yahan hain"\n    out.append(f\'<line x1="{cx:.1f}" x2="{cx:.1f}" y1="{T}" y2="{T + ph}" stroke="#142b53" stroke-width="2"/>\'\n               f\'<circle cx="{cx:.1f}" cy="{sy(current_p):.1f}" r="7" fill="#142b53" stroke="white" stroke-width="2"><title>Current: {feature} = {current_x:g}, {current_p * 100:.1f}%</title></circle>\')\n    step = max(1, len(xs) // 6)                                                      # x labels\n    for i in range(0, len(xs), step):\n        out.append(f\'<text x="{sx(xs[i]):.1f}" y="{H - 26}" font-size="13" text-anchor="middle">{xs[i]:g}</text>\')\n    out.append(f\'<text x="{L + pw / 2}" y="{H - 6}" font-size="13" text-anchor="middle">{feature} (grey = dataset ke patients)</text></svg>\')\n    return f\'<div class="hc-chart"><h4>{title}</h4>{"".join(out)}</div>\'\n\n\ndef percentile_text(value, population):\n    """Slider value ka dataset me position."""\n    pop = np.asarray(population, float)\n    pct = (pop < value).mean() * 100\n    return f"Dataset me {pct:.0f}% patients se zyada (median {np.median(pop):g})"\n',
    'shap_values.py': '"""\nHeartCare - SHAP values (haath se) aur unka interactive display.\n\nShapley value: ek feature ka "fair share" jo prediction ko average patient se door le jata hai.\n  phi_i = sum over coalitions S (i ke bina)  |S|! (M-|S|-1)! / M!  *  [ v(S + i) - v(S) ]\n  v(S)  = S ke features patient se, baaki features background (reference) patients se; model ka average output.\nYahan koi `shap` library nahi. 13 features = 2^13 = 8192 coalitions, saare exactly evaluate hote hain.\nContributions "probability points" (percentage points) me hain: P(class 1) = P(Higher Risk).\n"""\nfrom math import factorial\nimport numpy as np\nimport pandas as pd\n\nfrom confidence import FEATURES, CLASS_NAMES, COLORS\n\nM = len(FEATURES)\nMASKS = ((np.arange(2 ** M)[:, None] >> np.arange(M)) & 1).astype(bool)      # har coalition: kaun se features "on"\nSIZES = MASKS.sum(axis=1)\nWEIGHTS = np.array([factorial(s) * factorial(M - s - 1) / factorial(M) for s in range(M)])\n\n\n# ---------------------------------------------------------------- 1. manual computation\ndef build_background(df, k=20, seed=0):\n    """Reference patients: training data se k random rows."""\n    return df[FEATURES].sample(k, random_state=seed).to_numpy(dtype=float)\n\n\ndef coalition_values(model, x, background):\n    """v(S) har coalition ke liye (8192 values): model ka average P(class 1)."""\n    k = len(background)\n    X = np.where(MASKS[None, :, :], np.asarray(x, float)[None, None, :], background[:, None, :])   # (k, 8192, 13)\n    p = model.predict_proba(pd.DataFrame(X.reshape(-1, M), columns=FEATURES))[:, 1]\n    return p.reshape(k, -1).mean(axis=0)\n\n\ndef shapley_exact(model, x, background):\n    """Exact Shapley values. Return: phi (13,), base value, prediction f(x)."""\n    v = coalition_values(model, x, background)\n    phi = np.zeros(M)\n    for i in range(M):\n        without = np.where(~MASKS[:, i])[0]                      # i ke bina coalitions\n        phi[i] = (WEIGHTS[SIZES[without]] * (v[without | (1 << i)] - v[without])).sum()\n    return phi, float(v[0]), float(v[-1])                        # v[0] = sab background, v[-1] = sab patient\n\n\ndef shapley_sampling(model, x, background, n_perm=200, seed=0):\n    """Doosra independent tareeka (verification ke liye): random orderings me feature ek-ek karke patient wali value par laate hain."""\n    rng = np.random.default_rng(seed)\n    x = np.asarray(x, float)\n    phi = np.zeros(M)\n    for _ in range(n_perm):\n        order = rng.permutation(M)\n        b = background[rng.integers(len(background))].copy()\n        rows = [b.copy()]\n        for j in order:\n            b[j] = x[j]; rows.append(b.copy())\n        p = model.predict_proba(pd.DataFrame(np.array(rows), columns=FEATURES))[:, 1]\n        phi[order] += np.diff(p)\n    return phi / n_perm\n\n\ndef explain(model, x, background):\n    phi, base, fx = shapley_exact(model, x, background)\n    return {"phi": phi, "base": base, "pred": fx, "x": np.asarray(x, float), "k": len(background)}\n\n\n# ---------------------------------------------------------------- 2. plain-language text\ndef fmt(v):\n    return f"{v:g}"\n\n\ndef plain_summary(info, filled=()):\n    """Top drivers ek line me, aam bhasha me."""\n    order = np.argsort(-np.abs(info["phi"]))[:3]\n    parts = []\n    for i in order:\n        f = FEATURES[i]; pts = info["phi"][i] * 100\n        direction = CLASS_NAMES[1] if pts > 0 else CLASS_NAMES[0]\n        est = " (estimated)" if f in filled else ""\n        parts.append(f"{f} = {fmt(info[\'x\'][i])}{est} {direction} ki taraf {abs(pts):.1f} pts")\n    return "Sabse bade kaaran: " + "; ".join(parts) + "."\n\n\ndef feature_detail(info, feature, bg_mean, filled=()):\n    """Ek feature ka detail: value, average patient se compare, contribution."""\n    i = FEATURES.index(feature); pts = info["phi"][i] * 100\n    side = f"{CLASS_NAMES[1]} ki taraf" if pts > 0 else f"{CLASS_NAMES[0]} ki taraf"\n    est = " Ye value aapne nahi di, isliye typical value se bhari gayi hai." if feature in filled else ""\n    return (f"<b>{feature} = {fmt(info[\'x\'][i])}</b> (reference patients ka average {bg_mean[i]:.1f}). "\n            f"Is feature ne prediction ko <b>{abs(pts):.1f} percentage points {side}</b> dhakela.{est}")\n\n\n# ---------------------------------------------------------------- 3. interactive chart (HTML/CSS, animated)\nSHAP_CSS = """\n<style>\n.sv-row { display: flex; align-items: center; margin: 6px 0; padding: 2px 4px; border-radius: 6px; }\n.sv-row.sel { background: #eef4ff; outline: 2px solid #244577; }\n.sv-label { width: 128px; font-size: 14px; color: #334155; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }\n.sv-track { position: relative; flex: 1; height: 22px; background: #f1f5f9; border-radius: 6px; }\n.sv-mid { position: absolute; left: 50%; top: 0; bottom: 0; width: 2px; background: #64748b; }\n.sv-bar { position: absolute; top: 3px; bottom: 3px; width: var(--to); animation: sv-grow 0.9s ease-out; }\n.sv-bar.pos { left: 50%; border-radius: 0 5px 5px 0; }\n.sv-bar.neg { right: 50%; border-radius: 5px 0 0 5px; }\n.sv-val { width: 78px; text-align: right; font-size: 13px; font-weight: 600; color: #142b53; }\n.sv-total { display: flex; justify-content: space-between; font-size: 14px; padding: 8px 4px 0 4px; color: #142b53; border-top: 1px solid #e2e8f0; margin-top: 8px; flex-wrap: wrap; gap: 6px; }\n.sv-legend { font-size: 12px; color: #475569; margin-top: 6px; }\n@keyframes sv-grow { from { width: var(--from); } to { width: var(--to); } }\n@media (max-width: 600px) { .sv-label { width: 92px; font-size: 12px; } .sv-val { width: 62px; font-size: 12px; } }\n@media (prefers-reduced-motion: reduce) { .sv-bar { animation: none; } }\n</style>\n"""\n\n\ndef shap_chart(info, top_n=8, selected=None, prev=None, filled=()):\n    """Diverging bars: kaun sa feature prediction ko kis taraf aur kitna dhakel raha hai.\n    prev = {feature: pichli phi} (animation ka start)."""\n    phi = info["phi"]; order = np.argsort(-np.abs(phi)); top = order[:top_n]; rest = order[top_n:]\n    scale = max(np.abs(phi[order[0]]), 1e-9)                                    # sabse bada bar = aadha track\n    rows = ""\n    for i in top:\n        f = FEATURES[i]; v = phi[i]; cls = "pos" if v > 0 else "neg"\n        old = (prev or {}).get(f)\n        start = abs(old) / scale * 50 if old is not None and np.sign(old) == np.sign(v) else 0.0\n        color = COLORS[1] if v > 0 else COLORS[0]\n        est = "*" if f in filled else ""\n        sel = " sel" if f == selected else ""\n        rows += (f\'<div class="sv-row{sel}" title="{f}: {v * 100:+.2f} percentage points"><div class="sv-label">{f} = {fmt(info["x"][i])}{est}</div>\'\n                 f\'<div class="sv-track"><div class="sv-mid"></div><div class="sv-bar {cls}" style="--from:{start:.1f}%;--to:{abs(v) / scale * 50:.1f}%;background:{color}"></div></div>\'\n                 f\'<div class="sv-val">{v * 100:+.1f} pts</div></div>\')\n    if len(rest):\n        other = phi[rest].sum() * 100\n        rows += (f\'<div class="sv-row"><div class="sv-label">baaki {len(rest)} features</div><div class="sv-track"><div class="sv-mid"></div></div>\'\n                 f\'<div class="sv-val">{other:+.1f} pts</div></div>\')\n    total = (f\'<div class="sv-total"><span>Average patient: <b>{info["base"] * 100:.1f}%</b></span>\'\n             f\'<span>+ contributions: <b>{phi.sum() * 100:+.1f} pts</b></span><span>= Is patient: <b>{info["pred"] * 100:.1f}%</b></span></div>\')\n    legend = (f\'<div class="sv-legend"><span style="color:{COLORS[1]}">&#9632;</span> {CLASS_NAMES[1]} ki taraf &nbsp; \'\n              f\'<span style="color:{COLORS[0]}">&#9632;</span> {CLASS_NAMES[0]} ki taraf &nbsp; (* = estimated value)</div>\')\n    return f\'<div class="hc-chart"><h4>Kis feature ne prediction ko kitna dhakela (SHAP values)</h4>{rows}{total}{legend}</div>\'\n',
    'feedback.py': '"""\nHeartCare - usability feedback ikatha karna aur analyze karna.\n\n- add_feedback : app ke andar se feedback CSV me (thread-safe, health inputs store nahi hote)\n- summarize    : success rate, ease rating, device-wise, aur comments ke themes (manual keyword coding)\n- sus_score    : System Usability Scale (10 sawaal) ka standard scoring\n"""\nimport csv\nimport os\nimport threading\nimport time\nfrom collections import Counter\n\nimport pandas as pd\n\nFIELDS = ["time", "version", "device", "task_success", "ease_1to5", "comment"]\n_LOCK = threading.Lock()\n\n# Comments ko themes me baantne ke liye keywords (English + Hinglish)\nTHEMES = {\n    "labels/jargon":  ["confus", "unclear", "jargon", "term", "medical", "understand", "samajh", "meaning", "label"],\n    "length/scroll":  ["long", "scroll", "many", "too much", "lamba", "lambi", "zyada"],\n    "mobile/touch":   ["phone", "mobile", "small", "touch", "tap"],\n    "speed":          ["slow", "lag", "wait", "loading", "dhima"],\n    "charts/results": ["chart", "graph", "bar", "shap", "result", "explanation"],\n    "inputs":         ["slider", "type", "typing", "input", "field", "default"],\n}\n\n\ndef _safe(text, limit=500):\n    """CSV injection se bachao: =,+,-,@ se shuru text ke aage \' laga do. Length limit."""\n    text = str(text).replace("\\r", " ").replace("\\n", " ").strip()[:limit]\n    return "\'" + text if text[:1] in ("=", "+", "-", "@") else text\n\n\ndef add_feedback(path, device, task_success, ease, comment="", version="v2"):\n    """Ek feedback row save karo. Galat values par ValueError."""\n    if task_success not in ("Yes", "Partly", "No"):\n        raise ValueError("task_success: Yes / Partly / No")\n    if not 1 <= int(ease) <= 5:\n        raise ValueError("ease 1 se 5 ke beech hona chahiye")\n    row = [time.strftime("%Y-%m-%d %H:%M:%S"), version, _safe(device, 30), task_success, int(ease), _safe(comment)]\n    with _LOCK:                                                      # ek saath kai users likhein to rows na toote\n        new = not os.path.exists(path)\n        with open(path, "a", newline="", encoding="utf-8") as f:\n            w = csv.writer(f)\n            if new:\n                w.writerow(FIELDS)\n            w.writerow(row)\n\n\ndef load_feedback(path):\n    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=FIELDS)\n\n\ndef summarize(df):\n    """Feedback ka summary. Kam responses (n<5) par conclusions pakke nahi hote, isliye n bhi return hota hai."""\n    if len(df) == 0:\n        return {"n": 0}\n    themes = Counter()\n    for c in df["comment"].fillna("").str.lower():\n        for name, words in THEMES.items():\n            if any(w in c for w in words):\n                themes[name] += 1\n    return {"n": len(df),\n            "success_rate_%": round((df["task_success"] == "Yes").mean() * 100, 1),\n            "mean_ease": round(df["ease_1to5"].mean(), 2),\n            "by_device": df.groupby("device")["ease_1to5"].agg(["count", "mean"]).round(2).to_dict("index"),\n            "themes": dict(themes.most_common())}\n\n\ndef sus_score(answers):\n    """SUS: 10 jawab (1-5). Odd sawaal: jawab-1, even sawaal: 5-jawab, total x 2.5 -> 0-100."""\n    if len(answers) != 10 or any(not 1 <= a <= 5 for a in answers):\n        raise ValueError("SUS ke liye 10 jawab chahiye, har ek 1 se 5")\n    return sum((a - 1) if i % 2 == 0 else (5 - a) for i, a in enumerate(answers)) * 2.5\n',
    'validation.py': '"""\nHeartCare - robust input validation framework (detect -> correct -> explain -> suggest).\n\nLayers:\n  1. parse_quick(text)         : "age 63, chol 5.2 mmol, bp=130" jaisi free text ko record me todta hai (naam ke typo pakadta hai)\n  2. diagnose(record, rules)   : har field ke liye problem dhundta hai. Teen tarah ke result:\n        fixed   - safe auto-correction (jaise "1,5" -> 1.5, "63 yrs" -> 63, "5.2 mmol/L" -> 201 mg/dl)\n        error   - prediction roki jaati hai; message + (agar ho sake to) suggested value\n        warning - prediction chalti hai par user ko batate hain\n  3. check_record(record, rules): diagnose + input.validate_input (purana validator) = final {ok, x, errors, warnings, filled, issues}\nRule: jo cheez galat ho use chupchap kabhi "theek" nahi karte; jo 100% safe hai wahi auto-fix hota hai, baaki par user ki haan chahiye.\n"""\nimport difflib\nimport math\nimport re\n\nimport numpy as np\n\nfrom input import FEATURES, MAX_MISSING, validate_input\n\nALIASES = {"bp": "trestbps", "blood_pressure": "trestbps", "resting_bp": "trestbps", "cholesterol": "chol",\n           "heart_rate": "thalach", "max_hr": "thalach", "maxhr": "thalach", "st_depression": "oldpeak",\n           "vessels": "ca", "chest_pain": "cp", "angina": "exang", "sugar": "fbs", "ecg": "restecg", "gender": "sex"}\nMMOL_TO_MGDL = 38.67                      # cholesterol: 1 mmol/L = 38.67 mg/dl\nUNIT_WORDS = {"yrs", "yr", "years", "year", "y", "mmhg", "mg/dl", "mgdl", "bpm", "mm", "hg", "%"}\nEXAMPLE = {"age": "55", "trestbps": "130", "chol": "240", "thalach": "150", "oldpeak": "1.0"}\n\n\ndef issue(field, level, message, fix=None, fix_label=None, value=None):\n    """fix = suggested value (ya naam-typo me suggested field ka naam); value = user ne jo likha tha (naam-typo ke liye)."""\n    return {"field": field, "level": level, "message": message, "fix": fix, "fix_label": fix_label, "value": value}\n\n\ndef _norm(name):\n    return re.sub(r"[\\s\\-]+", "_", str(name).strip().lower())\n\n\ndef match_field(name):\n    """Field ka naam pehchano. Return (field ya None, suggestions list)."""\n    n = _norm(name)\n    if n in FEATURES:\n        return n, []\n    if n in ALIASES:\n        return ALIASES[n], []\n    close = difflib.get_close_matches(n, FEATURES + list(ALIASES), n=3, cutoff=0.6)\n    close = [ALIASES.get(c, c) for c in close]\n    close = list(dict.fromkeys(close))\n    return None, close\n\n\ndef parse_quick(text):\n    """Free text -> (record, issues). Separator: ; ya naya line ya \', \' ke baad naya naam (1,5 jaisa comma-decimal bacha rehta hai)."""\n    record, issues = {}, []\n    parts = re.split(r"[;\\n]|,\\s*(?=[A-Za-z])", str(text or ""))\n    for part in parts:\n        part = part.strip()\n        if not part:\n            continue\n        m = re.match(r"^([A-Za-z_][A-Za-z_ \\-]*?)\\s*(?:[=:]\\s*|\\s+(?=[-+\\d.,])|$)(.*)$", part)\n        if m and match_field(m.group(1))[0] is None and " " in m.group(1).strip():       # "sex male": pehla shabd naam, baaki value\n            first, rest = part.split(None, 1)\n            m = re.match(r"^(.*)$", first), None\n            m = type("M", (), {"group": lambda self, i, a=first, b=rest.lstrip("=: "): a if i == 1 else b})()\n        if not m:\n            issues.append(issue(None, "error", f"\'{part}\' samajh nahi aaya. Aise likho: age 63, chol 240"))\n            continue\n        field, close = match_field(m.group(1))\n        if field is None:\n            hint = f" Kya aap \'{close[0]}\' kehna chahte the?" if close else f" Fields: {\', \'.join(FEATURES)}"\n            issues.append(issue(None, "error", f"\'{m.group(1).strip()}\' naam pehchana nahi gaya.{hint}",\n                                fix=close[0] if close else None, fix_label=f"\'{close[0]}\' maano" if close else None,\n                                value=m.group(2).strip()))\n            continue\n        if _norm(m.group(1)) != field:\n            issues.append(issue(field, "fixed", f"\'{m.group(1).strip()}\' ko \'{field}\' samjha."))\n        if field in record:\n            issues.append(issue(field, "warning", f"{field} do baar likha hai; aakhri value li gayi."))\n        record[field] = m.group(2).strip()\n    return record, issues\n\n\n_NUM = re.compile(r"^([-+]?\\d*[.,]?\\d+)\\s*([A-Za-z/%]*[A-Za-z%])?$")\n\n\ndef _to_number(field, raw, rules):\n    """String -> (number ya None, issues). Comma decimal, unit suffix, mmol/L conversion yahan hota hai."""\n    out = []\n    s = str(raw).strip()\n    m = _NUM.match(s.replace(" ", "")) if re.match(r"^[-+]?\\.?\\d", s) else None\n    if m is None:\n        r = rules[field]\n        ex = EXAMPLE.get(field, "")\n        rng = f"{r[\'min\']:g} se {r[\'max\']:g}" if r["kind"] == "number" else "allowed: " + "/".join(str(int(a)) for a in r["allowed"])\n        out.append(issue(field, "error", f"{field}: \'{s}\' number nahi hai. Sirf number likho ({rng})" + (f", jaise {ex}." if ex else "."),\n                         fix=float(r["fill"]), fix_label=f"typical value {r[\'fill\']:g} lo"))\n        return None, out\n    num_txt, unit = m.group(1), (m.group(2) or "").lower()\n    if "," in num_txt:\n        out.append(issue(field, "fixed", f"{field}: \'{num_txt}\' ko {num_txt.replace(\',\', \'.\')} samjha (comma = decimal)."))\n        num_txt = num_txt.replace(",", ".")\n    value = float(num_txt)\n    if unit:\n        if unit in ("mmol/l", "mmol", "mmoll") and field == "chol":\n            out.append(issue(field, "fixed", f"chol: {value:g} mmol/L ko {value * MMOL_TO_MGDL:.0f} mg/dl me badla (x{MMOL_TO_MGDL})."))\n            value = value * MMOL_TO_MGDL\n        elif unit in UNIT_WORDS:\n            out.append(issue(field, "fixed", f"{field}: unit \'{unit}\' hata diya, value {value:g} li."))\n        else:\n            out.append(issue(field, "error", f"{field}: unit \'{unit}\' pehchana nahi gaya. Sirf number likho."))\n            return None, out\n    return value, out\n\n\ndef _suggest_number(field, v, r):\n    """Hard limit ke bahar number ke liye sabse samajhdar suggestion."""\n    if field == "chol" and 2 <= v < 40:                                    # mmol/L jaisa lagta hai\n        fixed = v * MMOL_TO_MGDL\n        if r["hard_lo"] <= fixed <= r["hard_hi"]:\n            return issue(field, "error", f"chol: {v:g} bahut kam hai (mg/dl me). Kya ye mmol/L me hai? To {fixed:.0f} mg/dl banta hai.",\n                         fix=round(fixed), fix_label=f"{fixed:.0f} mg/dl lo")\n    if field == "age" and v > r["hard_hi"] and v / 12 >= r["hard_lo"] and v / 12 <= r["hard_hi"]:\n        return issue(field, "error", f"age: {v:g} saal impossible hai. Kya ye mahine hain? To {v / 12:.0f} saal banta hai.",\n                     fix=round(v / 12), fix_label=f"{v / 12:.0f} saal lo")\n    if v < 0 and r["hard_lo"] <= abs(v) <= r["hard_hi"]:\n        return issue(field, "error", f"{field}: {v:g} negative nahi ho sakta. Kya {abs(v):g} matlab tha?", fix=abs(v), fix_label=f"{abs(v):g} lo")\n    near = min(max(v, r["min"]), r["max"])                  # data ki range ke andar sabse paas ki value\n    return issue(field, "error", f"{field}: {v:g} impossible hai (limit {r[\'hard_lo\']:g} se {r[\'hard_hi\']:g}). Aam range ({r[\'min\']:g} se {r[\'max\']:g}) me sabse paas ki value {near:g} hai.",\n                 fix=round(near, 2), fix_label=f"{near:g} lo")\n\n\ndef diagnose(record, rules, model=None, partial=False):\n    """Return {\'clean\': record (auto-fix ke baad), \'issues\': [...], \'blocked\': bool}.\n    partial=True: sirf diye gaye fields jaanchte hain (quick entry), missing-count ka rule nahi lagta."""\n    issues, clean, missing = [], {}, []\n    rec = dict(record) if isinstance(record, dict) else {}\n    for f in FEATURES:\n        raw = rec.get(f)\n        r = rules[f]\n        if raw is None or (isinstance(raw, str) and raw.strip() == "") or (isinstance(raw, float) and math.isnan(raw)):\n            clean[f] = None; missing.append(f); continue\n        if isinstance(raw, bool):\n            raw = int(raw)\n        if isinstance(raw, str):\n            low = raw.strip().lower()\n            if low in ("true", "false"):\n                raw = 1 if low == "true" else 0\n            else:\n                raw, extra = _to_number(f, raw, rules); issues += extra\n                if raw is None:\n                    clean[f] = None; continue\n        try:\n            v = float(raw)\n        except (TypeError, ValueError):\n            issues.append(issue(f, "error", f"{f}: \'{raw}\' number nahi hai.", fix=float(r["fill"]), fix_label=f"typical value {r[\'fill\']:g} lo"))\n            clean[f] = None; continue\n        if not math.isfinite(v):\n            issues.append(issue(f, "error", f"{f}: infinity/NaN allowed nahi.", fix=float(r["fill"]), fix_label=f"typical value {r[\'fill\']:g} lo"))\n            clean[f] = None; continue\n        if r["kind"] == "category":\n            allowed = [float(a) for a in r["allowed"]]\n            if v not in allowed:\n                order = sorted(allowed, key=lambda a: (abs(a - v), a))\n                alt = " ya ".join(f"{int(a)}" for a in order[:2]) if abs(order[0] - v) == abs(order[1] - v) else f"{int(order[0])}"\n                issues.append(issue(f, "error", f"{f}: {v:g} allowed nahi (allowed: {\'/\'.join(str(int(a)) for a in allowed)}). Paas ki value: {alt}.",\n                                    fix=int(order[0]), fix_label=f"{int(order[0])} lo"))\n                clean[f] = None; continue\n        else:\n            if not (r["hard_lo"] <= v <= r["hard_hi"]):\n                issues.append(_suggest_number(f, v, r)); clean[f] = None; continue\n            if not (r["min"] <= v <= r["max"]):\n                issues.append(issue(f, "warning", f"{f}: {v:g} training range ({r[\'min\']:g} se {r[\'max\']:g}) ke bahar hai; prediction kam bharosemand ho sakti hai."))\n        clean[f] = v\n\n    # fields ke beech ki jaanch (sirf chetavni)\n    a, hr = clean.get("age"), clean.get("thalach")\n    if a is not None and hr is not None and hr > (220 - a) + 15:\n        issues.append(issue("thalach", "warning", f"thalach {hr:g} is umar (age {a:g}) ke aam max heart rate (~{220 - a:g}) se kaafi upar hai. Value dobara check kar lo."))\n\n    # bahut zyada missing: sabse kaam ke fields batao\n    if not partial and len(missing) > MAX_MISSING:\n        pri = _priority(model)\n        ask = [f for f in pri if f in missing][: len(missing) - MAX_MISSING]\n        issues.append(issue(None, "error", f"{len(missing)} fields khali hain (max {MAX_MISSING}). Kam se kam {len(missing) - MAX_MISSING} aur bharo - "\n                            f"sabse zyada madad karne wale: {\', \'.join(ask)}."))\n    return {"clean": clean, "issues": issues, "blocked": any(i["level"] == "error" for i in issues)}\n\n\ndef _priority(model):\n    if model is not None and hasattr(model, "feature_importances_"):\n        return [FEATURES[i] for i in np.argsort(-model.feature_importances_)]\n    return list(FEATURES)\n\n\ndef check_record(record, rules, model=None):\n    """Poora pipeline. Return input.validate_input jaisa dict + \'issues\'."""\n    d = diagnose(record, rules, model)\n    if d["blocked"]:\n        errs = [i["message"] for i in d["issues"] if i["level"] == "error"]\n        warns = [i["message"] for i in d["issues"] if i["level"] != "error"]\n        return {"ok": False, "x": None, "errors": errs, "warnings": warns, "filled": [], "issues": d["issues"]}\n    res = validate_input(d["clean"], rules)\n    notes = [i["message"] for i in d["issues"] if i["level"] in ("warning", "fixed")]\n    res["warnings"] = notes + [w for w in res["warnings"] if w not in notes]\n    res["issues"] = d["issues"]\n    return res\n\n\ndef apply_fix(record, iss):\n    """Suggestion maanne par record badalta hai. Naam-typo me value naye field par jaati hai, warna field ki value badalti hai."""\n    out = dict(record)\n    if iss["field"] is None and iss.get("value") is not None:      # naam-typo: \'choll\' -> chol\n        out[iss["fix"]] = iss["value"]\n    elif iss["field"] in FEATURES:\n        out[iss["field"]] = iss["fix"]\n    return out\n',
    'explain_text.py': '"""\nHeartCare - prediction ko aam bhasha me samjhana (natural language generation).\n\nMethod: template-based NLG (jaan-boojhkar, taaki har shabd data se verify ho sake, koi "banawati" claim na ho).\n  1. Content selection : SHAP se top drivers (prediction ki taraf + khilaf), user ne jo values nahi di unka zikr\n  2. Confidence tier   : margin (probability threshold se kitni door), estimated values, mixed signals -> 4 tier\n  3. Sentence planning : tier ke hisaab se lambai aur hedging badalti hai (high = chhota aur seedha; borderline = zyada savdhaan)\n  4. Counterfactual    : ek value badalne se prediction palatti hai ya nahi (model ke hisaab se)\nHar sentence ke saath \'facts\' bhi lautta hai, taaki test kar sakein ki likha hua sach hai.\n"""\nimport numpy as np\n\nfrom confidence import CLASS_NAMES, FEATURES, sweep, sweep_values\n\nHUMAN = {"age": "age", "sex": "sex", "cp": "chest pain type", "trestbps": "resting blood pressure", "chol": "cholesterol",\n         "fbs": "fasting blood sugar", "restecg": "resting ECG", "thalach": "maximum heart rate", "exang": "exercise-induced angina",\n         "oldpeak": "ST depression (oldpeak)", "slope": "ST slope", "ca": "number of major vessels (ca)", "thal": "thal test result"}\nUNIT = {"age": " saal", "trestbps": " mmHg", "chol": " mg/dl", "thalach": " bpm"}\n\nTIERS = {  # name: (min margin, icon, label, headline verb)\n    "high": (0.30, "&#10003;", "High confidence", "model ko kaafi bharosa hai ki"),\n    "moderate": (0.15, "&#9679;", "Moderate confidence", "model ka jhukaav is taraf hai ki"),\n    "low": (0.05, "&#9888;", "Low confidence (close call)", "model ko thoda hi jhukaav hai ki"),\n    "toss-up": (0.0, "&#9888;", "Toss-up", "model lagbhag barabar soch raha hai, halka sa jhukaav ye hai ki"),\n}\nORDER = ["high", "moderate", "low", "toss-up"]\n\n\ndef pct(p):\n    return "<1%" if p < 0.005 else (">99%" if p > 0.995 else f"{p * 100:.0f}%")\n\n\ndef value_text(f, v):\n    return f"{v:g}{UNIT.get(f, \'\')}"\n\n\ndef confidence_tier(prob, threshold=0.5, n_filled=0, phi=None):\n    """Return (tier, reasons, base). base = sirf margin se tier; tier = estimated values / mixed signals se ek level neeche."""\n    margin = abs(prob - threshold)\n    base = next(t for t in ORDER if margin >= TIERS[t][0])\n    reasons = []\n    i = ORDER.index(base)\n    if n_filled >= 2 and i < 3:\n        i += 1; reasons.append(f"{n_filled} values aapne nahi di (estimated), isliye confidence ek level kam maani gayi")\n    if phi is not None:\n        pos, neg = phi[phi > 0].sum(), -phi[phi < 0].sum()\n        lead, opp = max(pos, neg), min(pos, neg)\n        if lead > 0 and opp / lead >= 0.6 and i < 3:\n            i += 1; reasons.append("kuch features ek taraf aur kuch doosri taraf khinch rahe hain (mixed signals)")\n    return ORDER[i], reasons, base\n\n\ndef counterfactual(model, x, rules, threshold=0.5, features=("chol", "trestbps", "thalach", "oldpeak")):\n    """Kisi ek number feature ko sabse kam badalkar decision palatti hai? Return dict ya None. (Model ka pattern, medical salah nahi.)"""\n    base_p = float(sweep(model, x, features[0], [float(x[FEATURES.index(features[0])])])[0])\n    want_high = base_p <= threshold\n    best = None\n    for f in features:\n        r = rules[f]; cur = float(x[FEATURES.index(f)])\n        vals = sweep_values(r, 41)\n        ps = sweep(model, x, f, vals)\n        for v, p in zip(vals, ps):\n            crossed = (p > threshold) if want_high else (p <= threshold)\n            if crossed:\n                rel = abs(v - cur) / (r["max"] - r["min"])\n                if best is None or rel < best["rel"]:\n                    best = {"feature": f, "from": cur, "to": float(v), "p_after": float(p), "rel": rel}\n    return best\n\n\ndef explain(info_shap, prob, threshold=0.5, filled=(), model=None, rules=None, x=None, population=None):\n    """Return {\'sentences\': [...], \'tier\', \'reasons\', \'facts\': [...], \'counterfactual\'}."""\n    phi = info_shap["phi"]; xs = info_shap["x"]\n    tier, reasons, base = confidence_tier(prob, threshold, len(filled), phi)\n    decision = 1 if prob > threshold else 0\n    lead_cls, other_cls = CLASS_NAMES[decision], CLASS_NAMES[1 - decision]\n    sign = 1 if decision == 1 else -1                    # decision ki taraf wale drivers ka SHAP sign\n    order = np.argsort(-np.abs(phi))\n    for_ = [i for i in order if np.sign(phi[i]) == sign][:2 if tier == "high" else 3]\n    against = [i for i in order if np.sign(phi[i]) == -sign][:1]\n    facts, sents = [], []\n\n    p_lead = prob if decision == 1 else 1 - prob\n    sents.append(f"{TIERS[base][3]} is patient ka profile <b>{lead_cls}</b> hai (model ke hisaab se {CLASS_NAMES[1]} probability {pct(prob)}, "\n                 f"threshold {threshold * 100:.0f}%).")\n    facts.append({"type": "headline", "decision": decision, "prob": prob})\n\n    def phrase(i):\n        f = FEATURES[i]; est = " (estimated value)" if f in filled else ""\n        pct = ""\n        if population is not None and f in population and f in UNIT:\n            share = float((np.asarray(population[f]) < xs[i]).mean() * 100)\n            pct = f", jo dataset ke {share:.0f}% patients se zyada hai" if share >= 50 else f", jo dataset ke {100 - share:.0f}% patients se kam hai"\n        return f"{HUMAN[f]} = {value_text(f, xs[i])}{est}{pct} (<b>{abs(phi[i]) * 100:.1f} pts</b>)"\n\n    if for_:\n        sents.append(f"Sabse zyada asar: {\'; \'.join(phrase(i) for i in for_)} - ye {lead_cls} ki taraf dhakel rahe hain.")\n        facts += [{"type": "for", "feature": FEATURES[i], "pts": float(abs(phi[i]) * 100), "dir": sign} for i in for_]\n    if against and tier != "high":\n        i = against[0]\n        sents.append(f"Ulta asar: {phrase(i)} - ye {other_cls} ki taraf khinch raha hai.")\n        facts.append({"type": "against", "feature": FEATURES[i], "pts": float(abs(phi[i]) * 100), "dir": -sign})\n\n    if tier == "high":\n        sents.append("Kai features ek hi taraf ishara kar rahe hain, isliye ye result stable lagta hai.")\n    else:\n        why = " ".join(r[0].upper() + r[1:] + "." for r in reasons)\n        close = base in ("low", "toss-up")                 # margin sach me kam hai?\n        line = {"moderate": "Result ek taraf jhuka hai, par pakka nahi.",\n                "low": "Ye close call hai: thoda sa input badalne par result palat sakta hai." if close else\n                       "Is result par bharosa kam hai; ise pakka mat maano.",\n                "toss-up": "Ye lagbhag 50-50 hai; is result par zyada bharosa mat karo." if close else\n                           "Is result par bharosa kaafi kam hai; ise pakka mat maano."}[tier]\n        sents.append((line + " " + why).strip())\n\n    cf = None\n    if model is not None and rules is not None and x is not None and tier != "high":\n        cf = counterfactual(model, x, rules, threshold)\n        if cf:\n            sents.append(f"Model ke hisaab se sirf {HUMAN[cf[\'feature\']]} {value_text(cf[\'feature\'], cf[\'from\'])} se "\n                         f"{value_text(cf[\'feature\'], cf[\'to\'])} hota to prediction {other_cls} ho jaati ({CLASS_NAMES[1]} probability {pct(cf[\'p_after\'])}).")\n        else:\n            sents.append("In 4 numbers (cholesterol, BP, heart rate, oldpeak) me se akele ek badalne se prediction nahi palatti.")\n    sents.append("Ye model ke data-patterns hain, doctor ki salah ya karan (cause) nahi.")\n    return {"sentences": sents, "tier": tier, "reasons": reasons, "facts": facts, "counterfactual": cf}\n\n\nCSS = """\n<style>\n.ex-card { background: var(--card, #fff); border: 1px solid var(--line, #cbd5e1); border-left-width: 6px; border-radius: 12px;\n  padding: 16px 18px; margin: 12px 0; animation: ex-in .4s ease both; }\n.ex-card.high { border-left-color: #1d4ed8; } .ex-card.moderate { border-left-color: #0f766e; }\n.ex-card.low, .ex-card.toss-up { border-left-color: #b45309; }\n.ex-badge { display: inline-block; font-weight: 700; font-size: 14px; padding: 3px 10px; border-radius: 999px; background: #e8eefb; color: #142b53; margin-bottom: 8px; }\n.ex-card.low .ex-badge, .ex-card.toss-up .ex-badge { background: #fdf0d9; color: #6b3a05; }\n.ex-card p { margin: 6px 0; line-height: 1.55; color: #1e293b; font-size: 16px; }\n@keyframes ex-in { from { opacity: 0; transform: translateY(6px); } to { opacity: 1; transform: none; } }\n@media (prefers-reduced-motion: reduce) { .ex-card { animation: none; } }\n</style>\n"""\n\n\ndef render(ex):\n    icon, label = TIERS[ex["tier"]][1], TIERS[ex["tier"]][2]\n    body = "".join(f"<p>{s}</p>" for s in ex["sentences"])\n    return f\'<div class="ex-card {ex["tier"]}" role="region" aria-label="Prediction explanation"><span class="ex-badge">{icon} {label}</span>{body}</div>\'\n',
    'theme.py': '"""\nHeartCare - theme: custom CSS (haath se), accessible colors (contrast check ke saath), animations/transitions.\n\nAccessibility rules jo yahan follow hote hain:\n  * Text contrast >= 4.5:1 (WCAG AA), UI parts (border, focus ring, bar) >= 3:1  -> audit() code se check karta hai\n  * Rang akela matlab nahi batata: risk label me icon + text hota hai (red/green na dikhne par bhi samajh aaye)\n  * Keyboard focus ring dikhta hai (:focus-visible), buttons phone par >= 44px\n  * Animations chhoti (<0.5s) aur prefers-reduced-motion par band\n"""\n\nTOKENS = {\n    "bg": "#f3f6fb", "card": "#ffffff", "navy": "#142b53", "blue": "#244577", "ink": "#1e293b", "muted": "#475569",\n    "line": "#64748b", "soft_line": "#e2e8f0", "focus": "#1d4ed8", "risk": "#b42323", "safe": "#177245",\n    "warn_ink": "#6b3a05", "warn_bg": "#fdf0d9", "on_navy": "#ffffff", "sub_on_navy": "#dbe4f5",\n}\n\n# (naam, text/foreground, background, kam se kam ratio)  - ratio 4.5 = normal text, 3 = UI part / bada text\nPAIRS = [\n    ("body text on card", "ink", "card", 4.5), ("body text on page", "ink", "bg", 4.5),\n    ("muted text on card", "muted", "card", 4.5), ("muted text on page", "muted", "bg", 4.5),\n    ("title on card", "navy", "card", 4.5), ("header title on navy", "on_navy", "navy", 4.5),\n    ("header subtitle on navy", "sub_on_navy", "navy", 4.5), ("button text on blue", "on_navy", "blue", 4.5),\n    ("Higher Risk text on card", "risk", "card", 4.5), ("Lower Risk text on card", "safe", "card", 4.5),\n    ("warning text", "warn_ink", "warn_bg", 4.5),\n    ("input border on card (UI)", "line", "card", 3.0), ("focus ring on card (UI)", "focus", "card", 3.0),\n    ("focus ring on page (UI)", "focus", "bg", 3.0), ("risk bar on card (UI)", "risk", "card", 3.0), ("safe bar on card (UI)", "safe", "card", 3.0),\n]\n\n\ndef _lum(hex_color):\n    h = hex_color.lstrip("#")\n    r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))\n    f = lambda c: c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4\n    return 0.2126 * f(r) + 0.7152 * f(g) + 0.0722 * f(b)\n\n\ndef contrast(fg, bg):\n    """WCAG contrast ratio (1 se 21)."""\n    a, b = sorted([_lum(fg), _lum(bg)], reverse=True)\n    return (a + 0.05) / (b + 0.05)\n\n\ndef audit(tokens=None):\n    """Har pair ka ratio aur pass/fail. Return list of dict."""\n    t = tokens or TOKENS\n    return [{"pair": n, "fg": t[f], "bg": t[b], "ratio": round(contrast(t[f], t[b]), 2), "needed": need,\n             "pass": contrast(t[f], t[b]) >= need} for n, f, b, need in PAIRS]\n\n\ndef risk_badge(decision, names=("Lower Risk", "Higher Risk")):\n    """Icon + text (sirf rang pe nirbhar nahi)."""\n    icon = "&#9650;" if decision == 1 else "&#9660;"\n    cls = "badge-risk" if decision == 1 else "badge-safe"\n    return f\'<span class="badge {cls}">{icon} {names[decision]}</span>\'\n\n\ndef root_vars():\n    return ":root {" + "".join(f" --{k.replace(\'_\', \'-\')}: {v};" for k, v in TOKENS.items()) + " }"\n\n\nCSS = """\n<style>\n__ROOT__\n\n/* ---------- base ---------- */\n.stApp { background: var(--bg); color: var(--ink); }\nheader, footer { visibility: hidden; }\n.main .block-container { max-width: 850px; padding: 25px 20px; animation: page-in .35s ease both; }\nh1, h2, h3 { color: var(--navy); }\np, label, .stMarkdown, .stCaption, [data-testid="stCaptionContainer"] { color: var(--ink); }\n[data-testid="stCaptionContainer"] { color: var(--muted); font-size: 14px; }\n\n/* ---------- header / section cards ---------- */\n.header { background-color: var(--navy); background-image: linear-gradient(135deg, var(--navy), var(--blue)); color: var(--on-navy); padding: 25px; border-radius: 14px;\n  text-align: center; margin-bottom: 25px; animation: fade-up .5s ease both; }\n.header h1 { color: var(--on-navy); margin: 0 0 4px; }\n.header p { color: var(--sub-on-navy); margin: 0; }\n.section { background: var(--card); padding: 20px; border-radius: 12px; margin-bottom: 18px; border: 1px solid var(--soft-line);\n  animation: fade-up .45s ease both; transition: box-shadow .2s ease, transform .2s ease; }\n.section:hover { box-shadow: 0 6px 18px rgba(20, 43, 83, .10); transform: translateY(-1px); }\n.section h3 { color: var(--navy); margin-top: 0; }\n.page-card { display: block; background: var(--card); border: 1px solid var(--soft-line); border-radius: 12px; padding: 16px 18px; margin: 10px 0; }\n\n/* ---------- buttons (min 44px touch target) ---------- */\n.stButton button, .stDownloadButton button { background: var(--blue); color: var(--on-navy); width: 100%; min-height: 44px; border-radius: 8px;\n  border: 2px solid var(--blue); font-weight: 600;\n  transition: background .15s ease, transform .1s ease, box-shadow .15s ease; }\n.stButton button:hover, .stDownloadButton button:hover { background: var(--navy); border-color: var(--navy); color: var(--on-navy);\n  box-shadow: 0 4px 12px rgba(20, 43, 83, .25); transform: translateY(-1px); }\n.stButton button:active { transform: translateY(0) scale(.98); }\n.stButton button p, .stDownloadButton button p { color: var(--on-navy); }\n\n/* ---------- keyboard focus (dikhna zaroori) ---------- */\n:focus-visible { outline: 3px solid var(--focus) !important; outline-offset: 2px !important; }\ninput:focus, textarea:focus, [data-baseweb="select"]:focus-within { box-shadow: 0 0 0 3px rgba(29, 78, 216, .35) !important; }\n\n/* ---------- small badges (icon + text) ---------- */\n.badge { display: inline-block; font-weight: 700; padding: 3px 10px; border-radius: 999px; font-size: 14px; }\n.badge-risk { background: #fbe9e9; color: var(--risk); } .badge-safe { background: #e4f4ec; color: var(--safe); }\n.notice { background: var(--warn-bg); color: var(--warn-ink); border-radius: 10px; padding: 10px 14px; margin: 8px 0; line-height: 1.5; }\n\n/* ---------- sidebar navigation ---------- */\n[data-testid="stSidebarNav"] a { border-radius: 8px; transition: background .15s ease, padding-left .15s ease; }\n[data-testid="stSidebarNav"] a:hover { background: #e3eafa; padding-left: 14px; }\n\n/* ---------- animations ---------- */\n@keyframes fade-up { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: none; } }\n@keyframes page-in { from { opacity: 0; } to { opacity: 1; } }\n@media (prefers-reduced-motion: reduce) {\n  *, *::before, *::after { animation: none !important; transition: none !important; }\n}\n\n/* ---------- small screens ---------- */\n@media (max-width: 600px) {\n  .main .block-container { padding: 12px 10px; }\n  .header { padding: 18px; } .header h1 { font-size: 24px; }\n  .section { padding: 14px; }\n}\n</style>\n""".replace("__ROOT__", root_vars())\n\n\ndef css(extra=""):\n    return CSS + extra\n\n\nSTREAMLIT_CONFIG = """[theme]\nbase = "light"\nprimaryColor = "#244577"\nbackgroundColor = "#f3f6fb"\nsecondaryBackgroundColor = "#ffffff"\ntextColor = "#1e293b"\n"""\n',
}
for name, text in BASE.items():
    if os.path.exists(name): print('hai   :', name)
    else: open(name, 'w').write(text); print('banayi:', name)
need = ['heart.csv', 'heart_model_v20261013_160903.joblib']
miss = [f for f in need if not os.path.exists(f)]
print('\nHEART.CSV / MODEL file folder me nahi hai:' if miss else '\nheart.csv aur model file hai.', miss or '')
if 'heart_model_v20261013_160903.joblib' in miss: print('Model banane ke liye notebook ka training cell (RandomForest wala, upar) ek baar chalao.')

In [ ]:
import os
os.makedirs('pages', exist_ok=True)

In [ ]:
%%writefile common.py
"""
HeartCare - saare pages ka shared code: model/rules/cache loading, page setup (theme + header), input widgets.
Har page `import common as cm` karta hai. Model, rules, cache aur SHAP background server me ek baar bante hain
(st.cache_resource) aur saare pages / users share karte hain.
"""
import joblib
import pandas as pd
import streamlit as st

import theme
import shap_values as sv
import components as ui
import explain_text as ex
from confidence import CSS as CHART_CSS, FEATURES
from input import load_rules
from cache import PredictionCache

MODEL_FILE = "heart_model_v20261013_160903.joblib"
DATA_FILE = "heart.csv"
FEEDBACK_FILE = "feedback.csv"
MAX_UNKNOWN = 3

SECTIONS = [
    ("Basic", ["age", "sex"]),
    ("Chest & heart rate", ["cp", "exang", "thalach", "oldpeak", "slope"]),
    ("Tests (blood / ECG)", ["trestbps", "chol", "fbs", "restecg"]),
    ("Scan results", ["ca", "thal"]),
]
LABELS = {"age": "Age", "sex": "Sex", "cp": "Chest Pain Type", "trestbps": "Resting Blood Pressure",
          "chol": "Cholesterol", "fbs": "Fasting Blood Sugar", "restecg": "Resting ECG",
          "thalach": "Maximum Heart Rate", "exang": "Exercise Induced Angina", "oldpeak": "Oldpeak",
          "slope": "Slope", "ca": "Major Vessels", "thal": "Thal"}
HELP = {"age": "Years me.", "sex": "Dataset ka code: 0 ya 1.", "cp": "Chest pain type ka code (0 se 3).",
        "trestbps": "Resting blood pressure, mm Hg.", "chol": "Serum cholesterol, mg/dl.",
        "fbs": "Fasting blood sugar ka code (0 ya 1).", "restecg": "Resting ECG result ka code (0 se 2).",
        "thalach": "Exercise me maximum heart rate, beats per minute.", "exang": "Exercise se chest pain hua? 0 ya 1.",
        "oldpeak": "Exercise ke baad ST depression (number).", "slope": "ST segment slope ka code (0 se 2).",
        "ca": "Major vessels ki ginti (dataset code 0 se 4).", "thal": "Thal test ka code (0 se 3)."}
WHOLE = {"age", "trestbps", "chol", "thalach"}    
KEEP = ("unknown", "threshold", "top_n", "inspect", "explore", "quick")     


@st.cache_resource
def load_everything():
    model = joblib.load(MODEL_FILE)
    rules = load_rules(DATA_FILE)
    cache = PredictionCache(max_size=512, ttl=600)
    df = pd.read_csv(DATA_FILE).drop_duplicates()
    background = sv.build_background(df, k=20)
    return model, rules, cache, MODEL_FILE, df, background


def load():
    """Model etc. lao; file na mile to saaf message aur page band."""
    try:
        return load_everything()
    except FileNotFoundError as e:
        st.error(f"File nahi mili: {e.filename}. Sab files (app.py, pages/, common.py, model, heart.csv) ek hi folder me rakho.")
        st.stop()


def page_setup(title, subtitle):
    """Har page ke upar: page config + theme CSS + header. set_page_config sabse pehle hona chahiye."""
    st.set_page_config(page_title=f"HeartCare - {title}", layout="centered")
    keep_inputs()
    st.markdown(theme.CSS + CHART_CSS + ui.CSS + sv.SHAP_CSS + ex.CSS, unsafe_allow_html=True)
    st.markdown(f'<div class="header"><h1>{title}</h1><p>{subtitle}</p></div>', unsafe_allow_html=True)


def keep_inputs():
    """Streamlit page badalne par widget ki value bhool jaata hai; ye use dobara set kar deta hai."""
    for k in list(st.session_state.keys()):
        if k.startswith("in_") or k in KEEP:
            st.session_state[k] = st.session_state[k]


def previous(key, current):
    """Chart ki pichli values (animation start). Session me yaad rakhte hain."""
    old = st.session_state.get("prev", {}).get(key)
    st.session_state.setdefault("prev", {})[key] = list(current)
    return old


def default_for(rules, f):
    r = rules[f]
    if r["kind"] == "number":
        return int(round(r["fill"])) if f in WHOLE else float(r["fill"])
    return int(r["fill"])


def set_input(rules, f, value):
    """Widget ki value programmatically badalna (callback me). Type widget ke hisaab se."""
    r = rules[f]
    if r["kind"] == "number":
        v = float(value)
        lo, hi = r["min"], r["max"]
        v = min(max(v, lo), hi)                   
        st.session_state["in_" + f] = int(round(v)) if f in WHOLE else round(v, 1)
    else:
        st.session_state["in_" + f] = int(value)


def reset_inputs(rules):
    for f in FEATURES:
        st.session_state["in_" + f] = default_for(rules, f)
    st.session_state["unknown"] = []
    st.session_state.pop("check", None)
    st.session_state.pop("qissues", None)


def widget(rules, data_df, f):
    """Number -> slider, 2 se 4 options -> radio, zyada options -> selectbox. Default median/mode."""
    r = rules[f]; key = "in_" + f
    st.session_state.setdefault(key, default_for(rules, f))
    if r["kind"] == "number":
        if f in WHOLE:
            v = st.slider(LABELS[f], int(r["min"]), int(r["max"]), key=key, help=HELP[f])
        else:
            v = st.slider(LABELS[f], float(r["min"]), float(r["max"]), step=0.1, key=key, help=HELP[f])
        st.caption(ui.percentile_text(v, data_df[f].values))
        return v
    options = [int(a) for a in r["allowed"]]
    if len(options) <= 4:
        return st.radio(LABELS[f], options, key=key, horizontal=True, help=HELP[f])
    return st.selectbox(LABELS[f], options, key=key, help=HELP[f])


def need_check():
    """Result wale pages ke liye: Predict page ka result lo, na ho to wapas bhejo."""
    check = st.session_state.get("check")
    if not check:
        st.markdown('<div class="notice">Pehle <b>Predict</b> page par patient ki details do. Wahan se aane par yahan result dikhega.</div>',
                    unsafe_allow_html=True)
        st.page_link("pages/1_Predict.py", label="Predict page par jao")
        st.stop()
    return check


def footer():
    st.markdown('<div style="text-align:center;color:#475569;padding:20px;">HeartCare | Educational Purpose Only<br>'
                'Not a substitute for professional medical diagnosis.</div>', unsafe_allow_html=True)

In [ ]:
%%writefile app.py
"""HeartCare - Home page. Baaki pages `pages/` folder me hain; Streamlit unhe sidebar me apne aap dikhata hai."""
import streamlit as st

import common as cm

cm.page_setup("HeartCare", "Heart Disease Risk Prediction System")
cm.load()

st.markdown("""
<div class="section">
    <h3>Kaise use karein</h3>
    <p>Dashboard ke alag-alag kaam alag pages par hain. Left sidebar se page chuno, ya neeche ke links se.</p>
</div>
""", unsafe_allow_html=True)

PAGES = [
    ("pages/1_Predict.py", "1. Predict", "Patient ki details do (sliders / quick entry). Galat input par saaf message aur suggestion milta hai. Risk turant dikhta hai."),
    ("pages/2_Explain.py", "2. Explain", "Prediction kyun aaya: aam bhasha me explanation (confidence ke hisaab se badalti hai), SHAP chart aur risk curve."),
    ("pages/3_Batch.py", "3. Batch", "CSV se bahut patients ek saath. Galat rows ke saath suggestion."),
    ("pages/4_Feedback.py", "4. Feedback", "App kaisa laga? Usability feedback do aur summary dekho."),
    ("pages/5_Model_Info.py", "5. Model Info", "Model, data, cache ki jaankari aur app ki seemayein."),
]
for path, label, text in PAGES:
    st.markdown(f'<div class="page-card"><b>{label}</b><br>{text}</div>', unsafe_allow_html=True)
    st.page_link(path, label=f"{label} page kholo")

if st.session_state.get("check"):
    st.markdown('<div class="notice">Aapka pichhla prediction session me saved hai. <b>Explain</b> page par uska reason dekh sakte ho.</div>',
                unsafe_allow_html=True)

cm.footer()

In [ ]:
%%writefile pages/1_Predict.py
"""Page 1: Predict - input (sliders + quick entry), validation framework (Task 1), risk card, confidence charts."""
import time

import streamlit as st

import common as cm
import components as ui
import theme
import validation as V
from confidence import CLASS_NAMES, COLORS, FEATURES, analyse, bar_chart, column_chart, decide

cm.page_setup("Predict", "Patient ki details do, risk turant dekho")
model, rules, cache, model_version, data_df, background = cm.load()


def apply_values(clean):
    """Jo values valid hain unhe form me daalo. Slider ki range ke bahar ho to clamp karke batao (chupchap nahi)."""
    notes = []
    for f, v in clean.items():
        if v is None:
            continue
        r = rules[f]
        if r["kind"] == "number" and not (r["min"] <= v <= r["max"]):
            near = min(max(v, r["min"]), r["max"])
            notes.append(V.issue(f, "warning", f"{f}: {v:g} slider ki range ({r['min']:g} se {r['max']:g}) ke bahar hai, isliye slider par {near:g} set kiya. "
                                 f"Bahut alag value ke liye Batch page (CSV) use karo."))
        cm.set_input(rules, f, v)
    return notes


def apply_quick():
    record, issues = V.parse_quick(st.session_state.get("quick", ""))
    d = V.diagnose(record, rules, model, partial=True)
    st.session_state["qissues"] = issues + d["issues"] + apply_values(d["clean"])


def take_fix(idx):
    issues = st.session_state.get("qissues", [])
    if idx >= len(issues):
        return
    chosen = issues[idx]
    record = V.apply_fix({}, chosen)
    d = V.diagnose(record, rules, model, partial=True)
    rest = [i for k, i in enumerate(issues) if k != idx]
    st.session_state["qissues"] = rest + d["issues"] + apply_values(d["clean"])


def show_issues(issues, prefix):
    """Har issue ka contextual message; jis par suggestion hai uske neeche 'use karo' button."""
    for k, i in enumerate(issues):
        if i["level"] == "error":
            st.error(i["message"])
            if i["fix"] is not None and prefix == "q":
                st.button(i["fix_label"] or "Suggestion lo", key=f"fix_{k}", on_click=take_fix, args=(k,))
        elif i["level"] == "warning":
            st.warning(i["message"])
        else:
            st.info("Auto-correct: " + i["message"])


top = st.empty()

st.markdown('<div class="section"><h3>Quick entry (optional)</h3><p>Ek line me likho, jaise <b>age 63, chol 5.2 mmol/L, bp 130, oldpeak 1,5</b>. '
            'Typo, comma-decimal, unit sab sambhal liye jaate hain.</p></div>', unsafe_allow_html=True)
st.text_input("Quick entry", key="quick", placeholder="age 63, chol 240, ca 2", label_visibility="collapsed")
st.button("Apply quick entry", on_click=apply_quick)
show_issues(st.session_state.get("qissues", []), "q")

values = {}
for title, feats in cm.SECTIONS:
    st.markdown(f'<div class="section"><h3>{title}</h3></div>', unsafe_allow_html=True)
    for f in feats:
        values[f] = cm.widget(rules, data_df, f)

unknown = st.multiselect("Values I don't know (max 3)", [cm.LABELS[f] for f in FEATURES], key="unknown")
too_many = len(unknown) > cm.MAX_UNKNOWN
if too_many:
    st.error(f"Maximum {cm.MAX_UNKNOWN} values 'unknown' ho sakti hain. Ek-do hata do.")

live = st.checkbox("Live update (slider hilate hi prediction update ho)", value=True)
clicked = st.button("Predict Heart Disease Risk")
st.button("Reset to typical values", on_click=cm.reset_inputs, args=(rules,))

if (clicked or live) and not too_many:
    record = {f: (None if cm.LABELS[f] in unknown else values[f]) for f in FEATURES}
    check = V.check_record(record, rules, model)
    show_issues([i for i in check["issues"] if i["level"] == "error"], "form")
    if check["ok"]:
        st.session_state["check"] = check
    else:
        st.session_state.pop("check", None)

if too_many:
    st.session_state.pop("check", None)

check = st.session_state.get("check")

if check:
    st.markdown('<div class="section"><h3>Prediction & Confidence</h3></div>', unsafe_allow_html=True)
    st.session_state.setdefault("threshold", 50)          
    st.slider("Decision threshold (%): Higher Risk tab maana jaye jab probability isse zyada ho", 1, 99, key="threshold")
    threshold = st.session_state["threshold"] / 100
    st.session_state["thr"] = threshold                   

    start = time.perf_counter()
    key = (model_version, tuple(float(v) for v in check["x"]))
    info = cache.get_or_compute(key, lambda: analyse(model, check["x"]))
    elapsed = (time.perf_counter() - start) * 1000

    for message in check["warnings"]:
        st.warning(message)

    filled = list(check["filled"])
    probs = info["probs"]
    p1 = float(probs[1])
    prev_card = st.session_state.get("last_p1")
    st.session_state["last_p1"] = p1
    top.markdown(ui.risk_card(p1, threshold, prev_card, filled), unsafe_allow_html=True)

    prediction = decide(probs, threshold)
    st.markdown(theme.risk_badge(prediction, (CLASS_NAMES[0], CLASS_NAMES[1])), unsafe_allow_html=True)
    st.write("Model Confidence:", round(max(probs) * 100, 2), "%")
    if threshold != 0.5:
        st.caption(f"Custom threshold {threshold * 100:.0f}%: {CLASS_NAMES[1]} probability {probs[1] * 100:.1f}% hai, "
                   f"isliye prediction {CLASS_NAMES[prediction]} dikh raha hai (model ka default threshold 50% hai).")
    st.caption(f"{info['votes'][1]} of {info['n_trees']} trees ne {CLASS_NAMES[1]} chuna | trees ke beech spread (std): {info['std']:.2f}")

    labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
    st.markdown(bar_chart("Class probabilities (trees ka average)", labels, probs,
                          prev=cm.previous("probs", probs), colors=[COLORS[0], COLORS[1]], threshold=threshold,
                          note=f"Kali line = threshold ({threshold * 100:.0f}%)"), unsafe_allow_html=True)
    votes = info["votes"] / info["n_trees"]
    st.markdown(bar_chart("Tree votes (kitni trees ne kaunsi class chuni)", labels, votes,
                          prev=cm.previous("votes", votes), colors=[COLORS[0], COLORS[1]]), unsafe_allow_html=True)
    hist = info["hist"]
    bins = [f"{i * 10}-{i * 10 + 10}" for i in range(len(hist))]
    st.markdown(column_chart(f"Har tree ka {CLASS_NAMES[1]} probability (%) - kitni trees kis range me", bins, hist / hist.max(),
                             prev=cm.previous("hist", hist / hist.max()), color="#244577", counts=[int(h) for h in hist],
                             note="Bar ke upar tree ginti."), unsafe_allow_html=True)

    if not live:
        st.caption("Ye result pichhle Predict click ke inputs ka hai. Naye inputs ke liye dobara Predict dabao ya Live update on karo.")
    st.page_link("pages/2_Explain.py", label="Ye prediction kyun aaya? Explain page kholo")
    st.caption(f"Is request me {elapsed:.2f} ms")

cm.footer()

In [ ]:
%%writefile pages/2_Explain.py
"""Page 2: Explain - aam bhasha explanation (Task 2), SHAP values, risk curve."""
import streamlit as st

import common as cm
import components as ui
import explain_text as ex
import shap_values as sv
import theme
from confidence import CLASS_NAMES, FEATURES, analyse, sweep, sweep_values

cm.page_setup("Explain", "Prediction kyun aaya?")
model, rules, cache, model_version, data_df, background = cm.load()
check = cm.need_check()

x = check["x"]; filled = list(check["filled"])
threshold = st.session_state.get("thr", 0.5)
xkey = tuple(float(v) for v in x)

info = cache.get_or_compute((model_version, xkey), lambda: analyse(model, x))
p1 = float(info["probs"][1])
sinfo = cache.get_or_compute((model_version, "shap", xkey, len(background)), lambda: sv.explain(model, x, background))
population = {f: data_df[f].values for f in ("age", "trestbps", "chol", "thalach")}
nlg = cache.get_or_compute((model_version, "nlg", xkey, threshold, tuple(filled)),
                           lambda: ex.explain(sinfo, p1, threshold, filled, model, rules, x, population))

st.markdown('<div class="section"><h3>Samajhne layak explanation</h3><p>Ye likhit explanation SHAP numbers se banti hai. '
            'Model jitna kam sure ho, utni savdhaani se likhi jaati hai.</p></div>', unsafe_allow_html=True)
st.markdown(theme.risk_badge(1 if p1 > threshold else 0, (CLASS_NAMES[0], CLASS_NAMES[1])), unsafe_allow_html=True)
st.markdown(ex.render(nlg), unsafe_allow_html=True)

st.markdown('<div class="section"><h3>SHAP values (numbers me)</h3><p>Har feature ne risk ko average patient se kitna upar/neeche dhakela '
            '(percentage points). Ye haath se nikali gayi exact Shapley values hain.</p></div>', unsafe_allow_html=True)
st.session_state.setdefault("top_n", 8)
st.slider("SHAP: kitne top features dikhayein", 3, 13, key="top_n")
inspect = st.selectbox("Ek feature ko detail me dekho", ["(none)"] + FEATURES, key="inspect")
sel = None if inspect == "(none)" else inspect
old_phi = cm.previous("shap", sinfo["phi"])
shap_prev = dict(zip(FEATURES, old_phi)) if old_phi else None
st.markdown(sv.shap_chart(sinfo, st.session_state["top_n"], sel, prev=shap_prev, filled=filled), unsafe_allow_html=True)
if sel:
    st.markdown(sv.feature_detail(sinfo, sel, background.mean(axis=0), filled), unsafe_allow_html=True)
st.caption("Transparency: ye model ke patterns hain, medical karan (cause) nahi. Reference = dataset ke 20 patients ka average. "
           "Base + saare contributions = final probability (exactly).")

st.markdown('<div class="section"><h3>Risk curve (what-if)</h3><p>Ek feature badlo, baaki wahi rakho.</p></div>', unsafe_allow_html=True)
explore = st.selectbox("Risk curve: ek feature badalke dekho", FEATURES, key="explore")
r = rules[explore]; fi = FEATURES.index(explore)
xs = sweep_values(r, 25) if r["kind"] == "number" else sweep_values(r)
ps = sweep(model, x, explore, xs)
st.markdown(ui.curve_chart(f"Risk curve: {explore} badalne par risk", explore, xs, ps, float(x[fi]), p1,
                           data_df[explore].values, r["kind"] == "category", threshold), unsafe_allow_html=True)
st.page_link("pages/1_Predict.py", label="Inputs badalne ke liye Predict page par jao")
cm.footer()

In [ ]:
%%writefile pages/3_Batch.py
"""Page 3: Batch - CSV se bahut patients; galat rows ke liye suggestion."""
import time

import pandas as pd
import streamlit as st

import common as cm
import validation as V
from confidence import FEATURES
from input import predict_batch

MAX_BATCH_ROWS = 100000
MAX_SUGGEST_ROWS = 200         

cm.page_setup("Batch", "CSV se bahut patients ek saath")
model, rules, cache, model_version, data_df, background = cm.load()

st.markdown("""
<div class="section">
    <h3>CSV upload</h3>
    <p>Columns: age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal.
    Galat rows reject hongi, aur 'suggestion' column me bataya jayega ki kya theek karein.</p>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Upload CSV", type=["csv"])
data = None
if uploaded is not None:
    try:
        data = pd.read_csv(uploaded)
    except Exception:
        st.error("CSV padh nahi paye. File sahi CSV format me honi chahiye.")

if data is not None:
    clean_names = [str(c).strip().lower().replace(" ", "_") for c in data.columns]
    missing_cols = [f for f in FEATURES if f not in clean_names]
    if len(data) == 0:
        st.error("CSV khali hai.")
    elif len(data) > MAX_BATCH_ROWS:
        st.error(f"Max {MAX_BATCH_ROWS} rows allowed hain.")
    else:
        if missing_cols:
            close = {c: V.match_field(c)[1][:1] for c in clean_names if c not in FEATURES and V.match_field(c)[0] is None}
            hints = "; ".join(f"'{c}' -> '{s[0]}'?" for c, s in close.items() if s)
            st.warning(f"Ye columns CSV me nahi mile (missing maane gaye): {missing_cols}." + (f" Naam ke typo: {hints}" if hints else ""))

        start = time.perf_counter()
        results = predict_batch(model, rules, data)
        elapsed = (time.perf_counter() - start) * 1000
        rejected_idx = list(results.index[results["status"] == "rejected"])

        suggestion = pd.Series("", index=results.index, name="suggestion")
        renamed = data.copy(); renamed.columns = clean_names
        for i in rejected_idx[:MAX_SUGGEST_ROWS]:
            d = V.diagnose({f: renamed.iloc[i].get(f) for f in FEATURES}, rules, model)
            suggestion.iloc[i] = " | ".join(x["message"] for x in d["issues"] if x["level"] == "error")
        results = pd.concat([results, suggestion], axis=1)

        st.write(f"Rows: {len(data)} | Predicted: {len(data) - len(rejected_idx)} | Rejected: {len(rejected_idx)}")
        st.caption(f"{elapsed:.0f} ms ({elapsed / len(data):.3f} ms per row)")
        if len(rejected_idx) > MAX_SUGGEST_ROWS:
            st.caption(f"Suggestion sirf pehli {MAX_SUGGEST_ROWS} rejected rows ke liye banaye gaye.")
        output = pd.concat([data, results], axis=1)
        st.dataframe(output.head(100))
        st.download_button("Download results", output.to_csv(index=False), "heart_predictions.csv", "text/csv")

cm.footer()

In [ ]:
%%writefile pages/4_Feedback.py
"""Page 4: Feedback - usability feedback form aur summary."""
import streamlit as st

import common as cm
from feedback import add_feedback, load_feedback, summarize

cm.page_setup("Feedback", "App kaisa laga? Isse hum interface sudharte hain")
cm.load()

st.markdown('<div class="section"><h3>Aapka feedback</h3></div>', unsafe_allow_html=True)
device = st.radio("Aap kis device par ho?", ["Phone", "Tablet", "Laptop"], horizontal=True, key="fb_device")
task = st.radio("Kya aap apna kaam (risk dekhna) bina madad ke kar paye?", ["Yes", "Partly", "No"], horizontal=True, key="fb_task")
ease = st.slider("Use karna kitna aasaan laga? (1 = mushkil, 5 = bahut aasaan)", 1, 5, 4, key="fb_ease")
text = st.text_area("Kuch atka ya confusing laga? (optional)", key="fb_text", max_chars=500)
if st.button("Send feedback"):
    try:
        add_feedback(cm.FEEDBACK_FILE, device, task, ease, text)
        st.success("Shukriya! Feedback save ho gaya.")
    except Exception as e:
        st.error(f"Feedback save nahi hua: {e}")

st.markdown('<div class="section"><h3>Ab tak ka summary</h3></div>', unsafe_allow_html=True)
s = summarize(load_feedback(cm.FEEDBACK_FILE))
if s["n"] == 0:
    st.caption("Abhi koi feedback nahi aaya.")
else:
    st.write(f"Responses: {s['n']} | Task success: {s['success_rate_%']}% | Average ease: {s['mean_ease']}/5")
    if s["n"] < 5:
        st.caption("5 se kam responses hain, isliye ye numbers pakke nahi hain.")
    if s["themes"]:
        st.caption("Comments ke themes: " + ", ".join(f"{k} ({v})" for k, v in s["themes"].items()))
cm.footer()

In [ ]:
%%writefile pages/5_Model_Info.py
"""Page 5: Model Info - model, data, cache aur seemayein."""
import pandas as pd
import streamlit as st

import common as cm
from confidence import FEATURES

cm.page_setup("Model Info", "Model, data aur app ki jaankari")
model, rules, cache, model_version, data_df, background = cm.load()

st.markdown('<div class="section"><h3>Model</h3></div>', unsafe_allow_html=True)
st.write(f"File: {model_version}")
st.write(f"Type: {type(model).__name__} | Trees: {len(model.estimators_)} | Features: {len(FEATURES)}")
st.write(f"Training data: {len(data_df)} unique patients (duplicates hataye gaye)")
counts = data_df["target"].value_counts().sort_index()
st.caption("Target distribution: " + ", ".join(f"{k} = {v}" for k, v in counts.items()))

st.markdown('<div class="section"><h3>Input ranges (data se)</h3></div>', unsafe_allow_html=True)
table = pd.DataFrame({f: {"kind": rules[f]["kind"], "min": rules[f]["min"], "max": rules[f]["max"], "typical (fill)": rules[f]["fill"]}
                      for f in FEATURES}).T
st.dataframe(table)

st.markdown('<div class="section"><h3>Cache (kitna kaam bacha)</h3></div>', unsafe_allow_html=True)
stats = cache.stats()
st.write(f"Hits: {stats['hits']} | Misses: {stats['misses']} | Coalesced: {stats['waits']} | Hit rate: {stats['hit_rate_%']}% | Size: {stats['size']}")

st.markdown('<div class="section"><h3>Seemayein (limitations)</h3></div>', unsafe_allow_html=True)
st.markdown("""
- Ye ek chhote dataset (~300 patients) par bana model hai; asli medical decision ke liye nahi.
- SHAP aur explanation model ke patterns dikhate hain, karan (cause) nahi.
- Categories ke codes (sex, cp, thal...) ka asli meaning dataset documentation se confirm karna chahiye.
- `target = 1` ka matlab dataset me kya hai ye confirm karo; `confidence.py` ki `CLASS_NAMES` se naam badal sakte ho.
""")
cm.footer()

In [ ]:
import sys, types, io, os, warnings; warnings.filterwarnings('ignore')
class State(dict):
    touched = set()
    def __setitem__(s, k, v): s.touched.add(k); super().__setitem__(k, v)
    def setdefault(s, k, v=None):
        if k not in s: s.touched.add(k)
        return super().setdefault(k, v)
STATE, RES, WKEYS = State(), {}, set()
PROJECT = ['input', 'cache', 'confidence', 'components', 'shap_values', 'feedback', 'validation', 'explain_text', 'theme', 'common']

def run(page, user=None, clicks=(), upload=None):
    for k, v in (user or {}).items(): STATE[k] = v; WKEYS.add(k)
    def make(clicked, collect):
        log, html, seen, rendered, started = [], [], [], set(), [False]
        st = types.ModuleType('streamlit')
        def guard(f):
            def w(*a, **k):
                if not started[0]: raise RuntimeError('set_page_config pehla streamlit call hona chahiye, mila: ' + f.__name__)
                return f(*a, **k)
            return w
        def set_page_config(**k):
            if started[0]: raise RuntimeError('set_page_config do baar')
            started[0] = True
        def cache_resource(f):
            def w(*a, **k):
                if f.__name__ not in RES: RES[f.__name__] = f(*a, **k)
                return RES[f.__name__]
            return w
        reg = lambda key: (rendered.add(key), WKEYS.add(key)) if key else None
        class Empty:
            def markdown(s, t, **k): html.append(t)
        def slider(label, lo, hi, v=None, step=None, key=None, help=None):
            reg(key); cur = STATE.setdefault(key, v); assert lo <= cur <= hi, (label, cur); return cur
        def radio(label, options, key=None, horizontal=False, help=None):
            reg(key); cur = STATE.setdefault(key, options[0]); assert cur in options, (label, cur); return cur
        def selectbox(label, options, key=None, help=None, index=0, format_func=str):
            reg(key); cur = STATE.setdefault(key, options[index]); assert cur in options, (label, cur); return cur
        def multiselect(label, options, key=None): reg(key); return STATE.setdefault(key, [])
        def checkbox(label, value=False, key=None): reg(key or label); return STATE.setdefault(key or label, value)
        def text_input(label, key=None, placeholder='', label_visibility='visible', value=''): reg(key); return STATE.setdefault(key, value)
        def text_area(label, key=None, max_chars=None): reg(key); return STATE.setdefault(key, '')
        def button(label, on_click=None, args=(), key=None, **k):
            if collect: seen.append((label, key, on_click, args)); return False
            return label in clicked or bool(key and key in clicked)
        def page_link(path, label=None, **k):
            log.append(('link', path)); assert os.path.exists(path), 'page file nahi mili: ' + path
        funcs = dict(markdown=lambda t, **k: html.append(t), caption=lambda m: log.append(('caption', m)), write=lambda *a: log.append(('write', ' '.join(map(str, a)))),
                     error=lambda m: log.append(('error', m)), success=lambda m: log.append(('success', m)), warning=lambda m: log.append(('warning', m)),
                     info=lambda m: log.append(('info', m)), dataframe=lambda d: log.append(('dataframe', d.shape)), download_button=lambda *a, **k: None,
                     page_link=page_link, empty=lambda: Empty(), slider=slider, radio=radio, selectbox=selectbox, multiselect=multiselect, checkbox=checkbox,
                     text_input=text_input, text_area=text_area, button=button, file_uploader=lambda *a, **k: upload)
        for n, f in funcs.items(): setattr(st, n, guard(f))
        st.set_page_config, st.cache_resource, st.session_state = set_page_config, cache_resource, STATE
        st.stop = lambda: (_ for _ in ()).throw(SystemExit)
        st.log = log
        return st, log, html, seen, rendered
    def execute(st):
        sys.modules['streamlit'] = st
        for m in PROJECT: sys.modules.pop(m, None)
        if '.' not in sys.path: sys.path.insert(0, '.')
        State.touched = set()
        try: exec(compile(open(page).read(), page, 'exec'), {'__name__': '__main__'})
        except SystemExit: st.log.append(('stop', ''))
    if clicks:
        st0, _, _, seen, _ = make(set(), True); execute(st0)
        for label, key, cb, args in seen:
            if (label in clicks or (key and key in clicks)) and cb: cb(*args)
    st, log, html, _, rendered = make(set(clicks), False); execute(st)
    for k in list(STATE):
        if k in WKEYS and k not in rendered and k not in State.touched: del STATE[k]
    return log, html

def msgs(r, *kinds): return [x[1] for x in r[0] if x[0] in kinds]
def count(r, s): return sum(s in h for h in r[1])
print('stub tayyar')

In [ ]:
import pandas as pd, shutil
had_fb = os.path.exists('feedback.csv')
if had_fb: shutil.copy('feedback.csv', 'feedback_backup.csv')    
ok = {}
r = run('app.py'); ok['Home: 5 page links'] = len([x for x in r[0] if x[0] == 'link']) == 5 and not msgs(r, 'error')
r = run('pages/2_Explain.py'); ok['Explain bina Predict ke: wapas bhejta hai'] = ('stop', '') in r[0] and ('link', 'pages/1_Predict.py') in r[0]
r = run('pages/1_Predict.py'); ok['Predict: risk card + check save'] = count(r, 'class="ui-card"') == 1 and 'check' in STATE
r = run('pages/2_Explain.py'); ok['Explain: NLG card + SHAP rows + curve'] = count(r, 'class="ex-card') == 1 and count(r, 'class="sv-row') >= 1 and count(r, 'viewBox') >= 1
run('pages/1_Predict.py', user={'quick': 'age 63, choll 5.2 mmol/L, bp 130, oldpeak 1,5, ca 9'}, clicks=('Apply quick entry',))
ok['Quick entry: age/bp/oldpeak form me aaye'] = (STATE['in_age'], STATE['in_trestbps'], STATE['in_oldpeak']) == (63, 130, 1.5)
run('pages/1_Predict.py', clicks=("'chol' maano",)); ok['Typo suggestion maanne par chol = 201'] = STATE['in_chol'] == 201
run('pages/1_Predict.py', clicks=('4 lo',)); ok['ca suggestion maanne par ca = 4'] = STATE['in_ca'] == 4
r = run('pages/1_Predict.py', user={'unknown': ['Cholesterol', 'Age', 'Sex', 'Oldpeak']}); ok['4 unknown: error aur result nahi'] = 'check' not in STATE and any('Maximum' in m for m in msgs(r, 'error'))
r = run('pages/1_Predict.py', user={'unknown': [], 'in_age': 41, 'in_ca': 3})
for p in ['pages/2_Explain.py', 'pages/3_Batch.py', 'pages/4_Feedback.py', 'pages/5_Model_Info.py', 'app.py']: run(p)
run('pages/1_Predict.py'); ok['Pages ghumne ke baad inputs yaad (age 41, ca 3)'] = (STATE['in_age'], STATE['in_ca']) == (41, 3)
raw = pd.read_csv('heart.csv').drop(columns='target').head(6).astype(object)
raw.loc[1, 'age'] = 'abc'; raw.loc[2, 'chol'] = 5.2; raw.loc[3, 'cp'] = 7; raw.loc[4, 'trestbps'] = 900
r = run('pages/3_Batch.py', upload=io.BytesIO(raw.to_csv(index=False).encode())); ok['Batch: 4 rows reject, 2 predict'] = any('Predicted: 2 | Rejected: 4' in m for m in msgs(r, 'write'))
r = run('pages/4_Feedback.py', user={'fb_device': 'Phone', 'fb_task': 'Partly', 'fb_ease': 2, 'fb_text': 'TEST'}, clicks=('Send feedback',)); ok['Feedback: save + summary'] = bool(msgs(r, 'success')) and not msgs(r, 'error')
r = run('pages/5_Model_Info.py'); ok['Model Info: 100 trees'] = any('Trees: 100' in m for m in msgs(r, 'write'))
if had_fb: shutil.move('feedback_backup.csv', 'feedback.csv')
elif os.path.exists('feedback.csv'): os.remove('feedback.csv')     
print(f'{sum(ok.values())} of {len(ok)} pass')

In [ ]:
!streamlit run app.py